<a href="https://colab.research.google.com/github/Eiboya/Dynamic-Programming---Project/blob/main/Information%20textuelle%20et%20pr%C3%A9vision%20de%20la%20volatilit%C3%A9%20financi%C3%A8re.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
#  HAR-RV-X vs GARCH-X × routes textuelles × 3 horizons de prévision × 12 actifs  [FINAL]
#  + Asset fixed effects panel regression with Driscoll-Kraay standard errors
#  + [OPTIMISATION MEMOIRE] Nettoyage explicite des gros objets texte/prédictions
#    devenus inutiles, afin d'éviter le crash RAM observé à la section [15] sur Colab.
#
#  Principales caractéristiques de la v17 (inchangées) :
#    1. Une collection canonique d'articles est construite une seule fois avec les règles v16 inchangées :
#       filtre de pertinence, nettoyage, longueur minimale et déduplication Jaccard intra-journalière.
#       LM, FinBERT, TF-IDF, Word2Vec et FWD_SENT reposent désormais sur exactement ces mêmes articles conservés.
#    2. FinBERT traite l'intégralité de chaque article canonique par segments de 512 tokens maximum,
#       puis agrège les probabilités des segments au niveau de l'article par moyenne pondérée par le nombre de tokens.
#       Son cache v17 est séparé des caches calculés sur les articles bruts.
#    3. TF-IDF et Word2Vec conservent leurs noms de checkpoint afin de réutiliser les résultats antérieurs lorsque
#       daily_text_stemmed est inchangé ; un fingerprint SHA-256 est désormais enregistré et contrôlé pour les futurs caches.
#    4. HAR-X et GARCH-X utilisent le même ensemble complet de variables pour chaque route textuelle.
#    5. HAR : pas OOS égal à h. GARCH : conception A, une origine commune tous les 22 jours pour h=1,5,22.
#       Chaque origine ne produit qu'une seule prévision ; les tests CW/DM OOS utilisent une bande HAC principale égale à zéro.
#    6. Alignement temporel cohérent de toutes les routes textuelles. Le mode principal est TEXT_TIMING_MODE='same_day' ;
#       le mode de robustesse 'lag1_all' décale toutes les routes, y compris FWD_SENT et les interactions.
#    7. HAR et GARCH disposent de checkpoints séparés et signés dans un nouveau répertoire Google Drive.
#
#  [NOUVEAU v17B-mem] Points de nettoyage mémoire ajoutés (chercher "[optimisation mémoire]") :
#    - Section 6.5 : suppression des colonnes de texte brut par article et des objets canoniques
#      dans df_raw une fois toutes les routes textuelles (LM/TFIDF/W2V/FinBERT) calculées.
#      Ce sont les objets les plus volumineux de tout le pipeline et ils ne servent plus après la section 6.
#    - Section 6.6 : suppression des tableaux intermédiaires FinBERT (art_df, art_probs_all, etc.).
#    - Section 14.5 : suppression des quatre grandes tables de prévisions OOS journalières
#      (df_oos_preds*) juste après leur export en CSV — elles ne sont plus utilisées ensuite.
#    - Sections 15 / 15.5 : gc.collect() et plt.close('all') après chaque actif, pour éviter
#      l'accumulation d'objets matplotlib/pandas au fil de la boucle.
#  Ces changements sont purement liés à la mémoire vive (RAM) du runtime Colab : ils ne touchent
#  ni les fichiers déjà enregistrés dans Google Drive, ni la logique statistique/économétrique.
#
#  Mode d'emploi (Google Colab) :
#    1. Exécuter d'abord build_multi_asset_gk_data.py afin de générer /content/dataset_merged.csv.
#    2. Téléverser articles_v3_raw.csv et le dictionnaire Loughran-McDonald dans /content/.
#    3. Exécuter ce script dans l'ordre, du début à la fin.
#    4. Astuce : si la RAM reste tendue malgré les nettoyages, choisir un runtime "High-RAM"
#       dans Colab (Exécution > Modifier le type d'exécution).
# =============================================================================

# %% ── 0. pip install ────────────────────────────────────
import subprocess, sys, os, pickle, shutil, gc
from statsmodels.stats.multitest import multipletests
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('statsmodels', 'linearmodels', 'gensim', 'arch', 'scikit-learn')
pip_install('transformers', 'sentencepiece')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive monté')

# %% ── 1. Configuration ──────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import re, time, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LassoCV
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
import scipy.cluster.hierarchy as sch
from scipy.linalg import orthogonal_procrustes
from scipy.optimize import linear_sum_assignment
from linearmodels.panel import PanelOLS

import nltk
from nltk.stem.snowball import SnowballStemmer
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
STEMMER_EN = SnowballStemmer('english')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='tab10')


ENABLE_FINBERT = True
ENABLE_GARCH   = True
ENABLE_CLUSTER_DIAG      = False
ENABLE_COLLINEARITY_DIAG = True  # VIF + test de robustesse par résidualisation du texte
ENABLE_PREDICTIVE_CAUSALITY = True
PREDICTIVE_CAUSALITY_MAX_LAG = 5
ENABLE_WINDOW_DM_TEST = True
ENABLE_TEXT_COEF_TREND_PLOTS = True
COEF_TREND_WINDOW_TYPES = ['expanding', 'rolling']
# Les trois horizons sont traités dans la même exécution, de manière séquentielle.
COEF_TREND_HORIZONS = [22]
ENABLE_GARCH_COEF_TREND_PLOTS = False  # les coefficients GARCH sont déjà exportés en CSV ; évite ~8 Go de RAM de graphiques
HAR_TEXT_COEF_TREND_SPECS = ['Spec2', 'Spec3']  # Spec2 correspond aux performances OOS/CW principales ; Spec3 ajoute les contrôles macrofinanciers

MIN_TRAIN_OOS  = 500
ROLLING_WINDOW = 750

# HAR : réestimation mensuelle approximative (tous les 22 jours), puis prévisions
# quotidiennes sur le bloc suivant de 22 jours. Les cibles h=5 et h=22 se chevauchent.
HAR_REFIT_STEP = 22
HAR_PREDICTION_BLOCK = 22

# GARCH : conception A, une seule origine commune tous les 22 jours pour tous les horizons.
# Une seule prévision est conservée à chaque origine ; les cibles OOS ne se chevauchent pas.
GARCH_OOS_STEP = 22

# Checkpoints de régression, séparés des anciens caches textuels et des versions v17/v17A.
ENABLE_HAR_MODEL_CACHE   = True
ENABLE_GARCH_MODEL_CACHE = True
FORCE_RERUN_HAR_MODELS   = False
FORCE_RERUN_GARCH_MODELS = False
MODEL_CACHE_VERSION = 'final_v1_har_refit22_daily_overlap_garch_step22_full_routes'

HAR_OOS_SAMPLING_MODE = 'refit22_daily_predictions'
GARCH_OOS_SAMPLING_MODE = 'step22_single_origin_nonoverlap'
GARCH_CW_DM_HAC_LAGS = 0

def har_refit_step_for_h(h):
    # Même fréquence de réestimation pour h=1,5,22 ; h est conservé dans la signature.
    if int(h) < 1:
        raise ValueError('h must be a positive integer')
    return int(HAR_REFIT_STEP)

# Sous-périodes COVID à dates fixes ; le découpage mécanique première/seconde moitié est conservé comme test général de stabilité, sans interprétation épidémiologique.
CW_DATE_SEGMENTS = [
    ('Pre-COVID',  None,         '2020-01-31'),
    ('COVID',      '2020-02-01', '2021-12-31'),
    ('Post-COVID', '2022-01-01', None),
]

DATA_PATH         = '/content/dataset_merged.csv'
ARTICLES_V3_PATH  = '/content/articles_v3_raw.csv'          # [ajout v8]
LM_PATH   = '/content/Loughran-McDonald_MasterDictionary_1993-2025.csv'

# Alignement de l'ensemble d'information :
#   same_day  : texte daté t pour prévoir RV_{t+1:t+h}; mode principal cohérent entre toutes les routes.
#   lag1_all  : test de robustesse où toutes les routes textuelles sont décalées d'un jour.
TEXT_TIMING_MODE = 'same_day'
if TEXT_TIMING_MODE not in {'same_day', 'lag1_all'}:
    raise ValueError("TEXT_TIMING_MODE doit valoir 'same_day' ou 'lag1_all'")

# Toutes les sorties et tous les checkpoints de cette exécution sont regroupés
# sous un nouveau dossier. L'ancien pipeline et ses caches ne sont pas réutilisés.
RUN_TAG = f'final_{TEXT_TIMING_MODE}'
FINAL_ROOT = f'/content/drive/MyDrive/har_pipeline_final_{TEXT_TIMING_MODE}'
RESET_FINAL_PROJECT_ON_START = False
if RESET_FINAL_PROJECT_ON_START and os.path.exists(FINAL_ROOT):
    shutil.rmtree(FINAL_ROOT)
    print(f'Ancien dossier final supprimé avant le nouveau calcul : {FINAL_ROOT}')

OUT_DIR = os.path.join(FINAL_ROOT, 'outputs')
CKPT_DIR = os.path.join(FINAL_ROOT, 'checkpoints', 'text')
MODEL_CKPT_ROOT = os.path.join(FINAL_ROOT, 'checkpoints', 'models')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Répertoire de sortie : {OUT_DIR}')

# Ces répertoires sont propres à FINAL_ROOT et ne réutilisent aucun ancien modèle.
HAR_MODEL_CKPT_DIR = os.path.join(MODEL_CKPT_ROOT, 'HAR')
GARCH_MODEL_CKPT_DIR = os.path.join(MODEL_CKPT_ROOT, 'GARCH')
os.makedirs(HAR_MODEL_CKPT_DIR, exist_ok=True)
os.makedirs(GARCH_MODEL_CKPT_DIR, exist_ok=True)

TICKERS = ['SPY', 'XLE', 'XLF', 'XLK', 'XLY', 'XLP',
           'XLI', 'XLV', 'XLB', 'XLU', 'XLRE', 'XLC']
H_LIST      = [1, 5, 22]

# Invalidation granulaire des checkpoints.
FORCE_RERUN_TEXT_MODELS = False
FORCE_RERUN_TFIDF       = False
FORCE_RERUN_W2V         = False
FORCE_RERUN_FINBERT     = False

# Configuration FinBERT plein article.
# Le nouveau nom de cache empêche toute restauration accidentelle des anciennes sorties
# calculées uniquement sur les 2 000 premiers caractères de chaque article.
FINBERT_MODEL                 = 'ProsusAI/finbert'
FINBERT_CACHE_VERSION         = 'v17_canonical_clean_dedup_full_article_chunks'
FINBERT_DAILY_CKPT_NAME       = f'finbert_daily_{FINBERT_CACHE_VERSION}'
FINBERT_PARTIAL_CKPT_NAME     = f'finbert_articles_partial_{FINBERT_CACHE_VERSION}'
FINBERT_MAX_LENGTH            = 512
FINBERT_STRIDE                = 0       # aucun chevauchement : chaque token du texte est compté une seule fois
FINBERT_ARTICLE_GROUP_SIZE    = 32      # nombre d'articles tokenisés simultanément
FINBERT_INFERENCE_BATCH_SIZE  = 16      # nombre de segments envoyés simultanément au GPU
FINBERT_AGGREGATION           = 'token_weighted_mean'

DEDUP_JACCARD_THRESHOLD = 0.8
TFIDF_MIN_NONZERO = 3

# [ajout v8] Configuration de la fusion des articles prospectifs — valeurs vérifiées sur les données réelles :
#   dans articles_v3_raw.csv, original_blog_baseline duplique presque entièrement les colonnes art_1~19 existantes ;
#   seules les deux catégories ci-dessous apportent un contenu réellement nouveau, avec au plus 5 articles par jour après fusion (médiane : 1),
#   couvrant environ 43 % des jours de négociation. Deux autres catégories n'apparaissent que pendant deux jours de test en juillet 2026 et sont exclues.
NEW_CATEGORIES     = ['institutional_research', 'fed_policy_communication']
EXCLUDE_CATEGORIES = ['earnings_preview_sector', 'market_preview_media']
MAX_NEW_SLOTS      = 5

def ckpt_path(name, ext='pkl'):
    return os.path.join(CKPT_DIR, f'{name}.{ext}')

def ckpt_exists(name, ext='pkl', force_rerun=False):
    return (not FORCE_RERUN_TEXT_MODELS
            and not force_rerun
            and os.path.exists(ckpt_path(name, ext)))

def save_ckpt(name, obj, ext='pkl'):
    p = ckpt_path(name, ext)
    if ext == 'pkl':
        with open(p, 'wb') as f:
            pickle.dump(obj, f, protocol=4)
    elif ext == 'csv':
        obj.to_csv(p, index=False)
    print(f'  💾 Checkpoint enregistré : {os.path.basename(p)}')

def load_ckpt(name, ext='pkl'):
    p = ckpt_path(name, ext)
    if ext == 'pkl':
        with open(p, 'rb') as f:
            return pickle.load(f)
    elif ext == 'csv':
        return pd.read_csv(p)



def _stable_json_hash(obj):
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


_MODEL_INPUT_HASH_MEMO = {}


def _model_input_fingerprint(columns):
    cols = tuple(dict.fromkeys(c for c in columns if c in df_raw.columns))
    if cols in _MODEL_INPUT_HASH_MEMO:
        return _MODEL_INPUT_HASH_MEMO[cols]
    if not cols:
        return None
    frame = df_raw.loc[:, list(cols)].copy()
    for col in frame.columns:
        if pd.api.types.is_datetime64_any_dtype(frame[col]):
            frame[col] = pd.to_datetime(frame[col], errors='coerce').astype('int64')
    hashed = pd.util.hash_pandas_object(frame, index=True).values.tobytes()
    fp = hashlib.sha256(hashed).hexdigest()
    _MODEL_INPUT_HASH_MEMO[cols] = fp
    return fp


def _model_cache_signature(model, ticker, h, unit, feature_spec=None):
    """Signature stricte : un changement de données, de features ou de conception invalide le cache."""
    model = str(model).upper()
    step = HAR_REFIT_STEP if model == 'HAR' else GARCH_OOS_STEP
    feature_spec = feature_spec or {}
    input_cols = feature_spec.get('input_cols', [])
    date_s = pd.to_datetime(df_raw['date'], errors='coerce') if 'df_raw' in globals() else pd.Series(dtype='datetime64[ns]')
    payload = {
        'cache_version': MODEL_CACHE_VERSION,
        'model': model,
        'ticker': ticker,
        'h': int(h),
        'unit': unit,
        'feature_spec': feature_spec,
        'text_timing_mode': TEXT_TIMING_MODE,
        'min_train_oos': MIN_TRAIN_OOS,
        'rolling_window': ROLLING_WINDOW,
        'oos_step': int(step),
        'har_prediction_block': int(HAR_PREDICTION_BLOCK) if model == 'HAR' else None,
        'har_oos_sampling_mode': HAR_OOS_SAMPLING_MODE if model == 'HAR' else None,
        'garch_oos_sampling_mode': GARCH_OOS_SAMPLING_MODE if model == 'GARCH' else None,
        'oos_start_date': str(globals().get('OOS_START_DATE', '')),
        'n_rows': int(len(df_raw)) if 'df_raw' in globals() else None,
        'date_min': str(date_s.min()) if len(date_s) else None,
        'date_max': str(date_s.max()) if len(date_s) else None,
        'model_input_fingerprint': _model_input_fingerprint(input_cols),
        'text_input_fingerprint': globals().get('TEXT_INPUT_FINGERPRINT'),
        'article_input_fingerprint': globals().get('ARTICLE_INPUT_FINGERPRINT'),
    }
    return _stable_json_hash(payload)


def _model_cache_path(model, key):
    root = HAR_MODEL_CKPT_DIR if str(model).upper() == 'HAR' else GARCH_MODEL_CKPT_DIR
    safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(key)).strip('_')
    return os.path.join(root, f'{safe}.pkl')


def _load_model_cache(model, key, signature, force=False):
    enabled = ENABLE_HAR_MODEL_CACHE if str(model).upper() == 'HAR' else ENABLE_GARCH_MODEL_CACHE
    if not enabled or force:
        return None
    path = _model_cache_path(model, key)
    if not os.path.exists(path):
        return None
    try:
        with open(path, 'rb') as f:
            obj = pickle.load(f)
        if obj.get('signature') != signature:
            print(f'  ⚠️ Cache {model} incompatible ignoré : {os.path.basename(path)}')
            return None
        return obj.get('payload')
    except Exception as e:
        print(f'  ⚠️ Lecture impossible du cache {model} {os.path.basename(path)} : {e}')
        return None


def _save_model_cache(model, key, signature, payload):
    enabled = ENABLE_HAR_MODEL_CACHE if str(model).upper() == 'HAR' else ENABLE_GARCH_MODEL_CACHE
    if not enabled:
        return
    path = _model_cache_path(model, key)
    tmp = path + '.tmp'
    obj = {'signature': signature, 'payload': payload}
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    os.replace(tmp, path)
    print(f'  💾 Cache {model} enregistré : {os.path.basename(path)}')


def _snapshot_lengths(accumulators):
    return {name: len(values) for name, values in accumulators.items()}


def _slice_since(accumulators, starts):
    return {name: values[starts[name]:] for name, values in accumulators.items()}


def _restore_lists(accumulators, payload):
    for name, values in accumulators.items():
        values.extend(payload.get(name, []))

print('✅ Configuration terminée')
print(f'   TICKERS ({len(TICKERS)} actifs) : {TICKERS}')
print(f'   ENABLE_FINBERT={ENABLE_FINBERT} | ENABLE_GARCH={ENABLE_GARCH} | '
      f'ENABLE_COLLINEARITY_DIAG={ENABLE_COLLINEARITY_DIAG} | '
      f'ENABLE_PREDICTIVE_CAUSALITY={ENABLE_PREDICTIVE_CAUSALITY}')
print(f'   TEXT_TIMING_MODE={TEXT_TIMING_MODE} | RUN_TAG={RUN_TAG}')

import torch as _torch_probe
DEVICE = _torch_probe.device('cuda' if _torch_probe.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    _gpu_name = _torch_probe.cuda.get_device_name(0)
    print(f'✅ GPU détecté : {_gpu_name} — l’inférence FinBERT utilisera l’accélération GPU')
else:
    print('⚠️  Aucun GPU détecté ; l’inférence FinBERT utilisera le CPU et sera nettement plus lente. '
          'Il est recommandé de sélectionner un environnement T4 dans Colab.')
print('   (TF-IDF, Word2Vec, GARCH et les diagnostics utilisent principalement le CPU.)')


# %% ── 1.5 [ajout v8] Fonctions d’identification du contenu prospectif / de type « outlook » ─────────────────────────
# Cette expression régulière a été calibrée empiriquement : la première version recherchait des expressions comme "consensus forecast/will release"
# dans les 3 000 premiers caractères du texte intégral, ce qui produisait 91 % de jours has_fwd sur les 400 premiers jours.
# Le diagnostic a montré que de nombreux articles purement rétrospectifs (par exemple "ADP: Private Employment increased
# 213,000") mentionnaient accessoirement "beat consensus forecast of X",
# et étaient donc classés à tort comme prospectifs. Les signaux faibles (preview/outlook/nowcast/
# looking ahead) sont désormais recherchés uniquement dans le **titre** ; le corps du texte ne conserve que des signaux forts explicites comme "we expect/we forecast/
# staff forecast". Sur le même échantillon de 400 jours, la part has_fwd tombe à 28 %,
# proche de l’ordre de grandeur de 35,2 % obtenu antérieurement avec une liste de mots-clés plus large,
# ce qui est plus crédible. La précision n’a toutefois pas encore été validée article par article ; avant la version finale du mémoire,
# il est recommandé de contrôler manuellement 10 à 20 articles avec has_fwd=1 afin d’estimer le taux de faux positifs.

STUB_BOILERPLATE_PATTERN = re.compile(
    r'(please enable javascript|for media inquiries|is attached|attachment \(pdf\)|'
    r'minutes are attached|projections \(pdf\)|summary of economic projections.*addendum|'
    r'the full (?:statement|minutes) (?:is|are) available)',
    re.IGNORECASE,
)

# Signaux faibles : recherche uniquement dans le titre (une recherche dans tout le texte classerait à tort de nombreux articles rétrospectifs avec une mention accessoire).
TITLE_FORWARD_PATTERN = re.compile(
    r'(\bpreview\b|\boutlook\b|\bnowcast\b|week ahead|looking ahead)',
    re.IGNORECASE,
)

# Signaux forts : formulation prédictive à la première personne ou institutionnelle, suffisamment précise pour être recherchée dans tout le texte.
BODY_FORWARD_PATTERN = re.compile(
    r'(we expect|we forecast|we project|our (?:baseline )?forecast|staff forecast|'
    r'is expected to (?:rise|fall|increase|decrease|climb|decline)|'
    r'expected to (?:rise|fall|increase|decrease|climb|decline) (?:by|to|from)|'
    r'nowcast(?:s|ing)? (?:point|show|suggest|indicate)|'
    r'projections? (?:show|indicate|suggest|imply))',
    re.IGNORECASE,
)


def is_forward_looking(title, text, head_chars=3000):
    """Détermine si un article est prospectif au niveau de son contenu ; seuls le titre et le corps sont utilisés, pas la catégorie de source."""
    if not isinstance(text, str):
        return False
    head = text[:head_chars]
    ttl  = title if isinstance(title, str) else ''
    if STUB_BOILERPLATE_PATTERN.search(f'{ttl} {head}'):
        return False
    return bool(TITLE_FORWARD_PATTERN.search(ttl)) or bool(BODY_FORWARD_PATTERN.search(head))


NEW_SLOT_START_INDEX = 20  # À partir de art_20, les colonnes proviennent déjà du contenu additionnel sélectionné dans institutional_research/
                            # et fed_policy_communication.

def _slot_index(col_name):
    try:
        return int(col_name.split('_')[1])
    except (IndexError, ValueError):
        return -1

def article_is_fwd(col_name, title, text):
    """
    Combine deux niveaux de signal pour déterminer si un article est prospectif :
      (a) signal de catégorie de source — les colonnes à partir de art_20 sont déjà constituées du contenu additionnel
          sélectionné dans institutional_research/fed_policy_communication ;
          la source appartient donc directement à une catégorie prospective, sans nouvelle vérification par expression régulière ;
      (b) signal issu du contenu — parmi les 19 colonnes historiques (blogs principalement rétrospectifs),
          certains articles réellement prospectifs peuvent néanmoins apparaître, par exemple les previews CPI/FOMC de Calculated Risk ;
          seuls ces articles sont filtrés au moyen des signaux faibles du titre et des signaux forts du corps via is_forward_looking().
    Cette procédure garantit une définition identique dans has_fwd, fwd_lm_polarity et finbert_sent_fwd,
    et évite une incohérence où has_fwd=1 alors que l’article concerné n’est pas inclus dans le calcul du sentiment prospectif.
    """
    if _slot_index(col_name) >= NEW_SLOT_START_INDEX:
        return True
    return is_forward_looking(title, text)


def build_new_article_slots(articles_v3_path):
    """
    Lit articles_v3_raw.csv, conserve uniquement les catégories additionnelles (institutional_research +
    fed_policy_communication), puis les pivote en table large art_20~24_source/title/text/summary,
    avec une ligne par date, destinée à être fusionnée avec df_raw.
    """
    # Attention : au moins un champ de ce fichier dépasse 128 Ko ; la limite par défaut du module csv déclenche une erreur.
    # Le moteur Python de pandas n’impose pas cette limite ; engine='python' évite donc la perte de données
    # (le nombre total de lignes, 36 959, a été vérifié séparément avec le module csv).
    df = pd.read_csv(articles_v3_path, encoding='utf-8-sig',
                      engine='python', on_bad_lines='skip')
    df['date_day'] = pd.to_datetime(df['date_day'], errors='coerce')
    df = df[df['category'].isin(NEW_CATEGORIES)].dropna(subset=['date_day']).copy()

    df['_cat_order'] = df['category'].map({'institutional_research': 0,
                                            'fed_policy_communication': 1})
    df = df.sort_values(['date_day', '_cat_order'])

    wide_rows = []
    for date, grp in df.groupby('date_day'):
        row = {'date': date}
        for i, (_, art) in enumerate(grp.head(MAX_NEW_SLOTS).iterrows(), start=20):
            row[f'art_{i}_source']  = art.get('source')
            row[f'art_{i}_title']   = art.get('title')
            txt = art.get('text')
            if not isinstance(txt, str) or len(txt.strip()) == 0:
                txt = art.get('combined_text')
            row[f'art_{i}_text']    = txt
            row[f'art_{i}_summary'] = art.get('summary')
        wide_rows.append(row)

    return pd.DataFrame(wide_rows)


def _canonical_article_is_fwd(article):
    """Applique la définition FWD à un article déjà filtré, nettoyé et dédupliqué."""
    return article_is_fwd(
        article['text_col'], article.get('title'), article['text'])


def compute_daily_fwd_flags(canonical_articles):
    """Calcule has_fwd et les comptages sur la même collection canonique que toutes les routes textuelles."""
    articles = canonical_articles if isinstance(canonical_articles, list) else []
    n_total = len(articles)
    n_fwd = sum(_canonical_article_is_fwd(article) for article in articles)
    return pd.Series({
        'n_arts_total': n_total,
        'n_fwd_arts':   n_fwd,
        'fwd_share':    (n_fwd / n_total) if n_total > 0 else np.nan,
        'has_fwd':      int(n_fwd > 0),
    })


def compute_fwd_lm_polarity(canonical_articles, lm_score_fn):
    """Polarité LM calculée uniquement sur le sous-ensemble FWD des articles canoniques."""
    p_sum, n_sum = 0, 0
    articles = canonical_articles if isinstance(canonical_articles, list) else []
    for article in articles:
        if not _canonical_article_is_fwd(article):
            continue
        res = lm_score_fn(article['text'])
        if res:
            p, n, u, l, w = res
            p_sum += p
            n_sum += n
    denom = p_sum + n_sum
    return (p_sum - n_sum) / denom if denom > 0 else np.nan


print('✅ Fonctions d’identification du contenu prospectif prêtes')

# %% ── 2. Chargement et prétraitement des données (déduplication des articles du même jour + fusion v8 des articles prospectifs) ─────────────
import re as _re

_SPAN_MCE  = _re.compile(r'<span[^>]*data-mce-type[^>]*>.*?</span>',
                          _re.IGNORECASE | _re.DOTALL)
_HTML_TAGS = _re.compile(r'<[a-zA-Z][^>]*>')
_URLS      = _re.compile(r'https?://\S+')
_BOM       = _re.compile(r'\ufeff')
_BIDI_CTRL = _re.compile(r'[\u2066\u2069\u200b\u200c\u200d\u2063]')
_BULLETS   = _re.compile(r'[•◦▪▸→➤★]')
_NBSP      = _re.compile(r'\xa0')
_MULTI_SP  = _re.compile(r' {2,}')
_WEEKDAY   = _re.compile(
    r'^(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\s*:',
    _re.IGNORECASE)

_FILTER_TITLE_KW = [
    'am reads', 'morning reads', 'afternoon reads', 'weekend reads',
    'linkfest', 'reads for the week', 'succinct summation',
    'animal spirits', 'mib:', 'masters in business',
    'clips from', 'closing bell', 'episode',
    'farewell', 'in memoriam', 'birthday', 'wedding', 'vacation',
    'schedule for the week', 'preview for',
]
_FILTER_TEXT_KW = [
    'mortgage delinquency', 'apartment vacancy', 'home builder',
    'single-family starts', 'freddie mac house price',
    'case-shiller home', 'fannie mae serious',
]
_MIN_CHAR_LEN = 100


def clean_article_text(text: str):
    if not isinstance(text, str):
        return None
    text = _SPAN_MCE.sub(' ', text)
    text = _HTML_TAGS.sub(' ', text)
    text = _URLS.sub(' ', text)
    text = _BOM.sub('', text)
    text = _BIDI_CTRL.sub('', text)
    text = _BULLETS.sub(' ', text)
    text = _NBSP.sub(' ', text)
    text = _MULTI_SP.sub(' ', text).strip()
    return text if len(text) >= _MIN_CHAR_LEN else None


def is_article_relevant(title: str, text: str) -> bool:
    title_lc = str(title).lower() if isinstance(title, str) else ''
    text_lc  = str(text).lower()  if isinstance(text,  str) else ''
    if any(kw in title_lc for kw in _FILTER_TITLE_KW):
        return False
    if _WEEKDAY.match(title_lc) and len(text_lc.strip()) < 800:
        return False
    if any(kw in text_lc for kw in _FILTER_TEXT_KW):
        return False
    if len(text_lc.strip()) < _MIN_CHAR_LEN:
        return False
    return True


_DEDUP_TOKEN_RE = re.compile(r'[a-z]+')

def _jaccard_token_set(text):
    return set(_DEDUP_TOKEN_RE.findall(text.lower()))

def dedup_articles(article_texts):
    kept, kept_sets = [], []
    for txt in article_texts:
        s = _jaccard_token_set(txt)
        if len(s) == 0:
            continue
        is_dup = False
        for ks in kept_sets:
            union = s | ks
            if len(union) == 0:
                continue
            jac = len(s & ks) / len(union)
            if jac > DEDUP_JACCARD_THRESHOLD:
                is_dup = True
                break
        if not is_dup:
            kept.append(txt)
            kept_sets.append(s)
    return kept


def dedup_article_records(article_records):
    """Même algorithme et même ordre que dedup_articles, en conservant les métadonnées de l’article."""
    kept, kept_sets = [], []
    for article in article_records:
        txt = article['text']
        s = _jaccard_token_set(txt)
        if len(s) == 0:
            continue
        is_dup = False
        for ks in kept_sets:
            union = s | ks
            if len(union) == 0:
                continue
            jac = len(s & ks) / len(union)
            if jac > DEDUP_JACCARD_THRESHOLD:
                is_dup = True
                break
        if not is_dup:
            kept.append(article)
            kept_sets.append(s)
    return kept


FIN_COLS       = [f'ret_{t}'       for t in TICKERS]
RV_COLS        = [f'rv_{t}'        for t in TICKERS]
MACRO          = ['VIX', 'FEDFUNDS', 'CPIAUCSL', 'T10Y2Y']
GKG_COLS       = [
    'gkg_tone_mean', 'gkg_tone_median', 'gkg_tone_std',
    'gkg_positive_mean', 'gkg_negative_mean', 'gkg_polarity_mean',
    'gkg_sentiment', 'gkg_negative_tone', 'gkg_n_articles', 'gkg_n_sources',
]
TEXT_COLS_RAW  = [f'art_{i}_text'  for i in range(1, 20)]   # 19 colonnes historiques
TITLE_COLS_RAW = [f'art_{i}_title' for i in range(1, 20)]

load_cols = (
    ['date'] + FIN_COLS + RV_COLS + MACRO + GKG_COLS
    + TEXT_COLS_RAW + TITLE_COLS_RAW
)

df_raw = pd.read_csv(DATA_PATH, usecols=lambda c: c in load_cols, parse_dates=['date'])
df_raw = df_raw.sort_values('date').reset_index(drop=True)

df_raw[GKG_COLS] = df_raw[GKG_COLS].ffill()   # [correction v9 n°2] Remplissage uniquement vers l’avant,
                                                # sans interpolation linéaire bidirectionnelle, afin d’éviter d’utiliser une valeur GKG future
                                                # pour compléter une observation présente, ce qui créerait une fuite d’information.
                                                # Si le début de la série ne contient encore aucune valeur valide,
                                                # la valeur reste NaN et sera supprimée en aval par dropna(),
                                                # plutôt que d’être artificiellement imputée.

# [ajout v8] Fusion des articles prospectifs réellement additionnels → art_20~24
print('\n[2.1] Fusion des articles prospectifs additionnels (institutional_research + fed_policy_communication) ...')
_new_slots_wide = build_new_article_slots(ARTICLES_V3_PATH)
df_raw = df_raw.merge(_new_slots_wide, on='date', how='left')
TEXT_COLS_RAW  += [f'art_{i}_text'  for i in range(20, 20 + MAX_NEW_SLOTS)]
TITLE_COLS_RAW += [f'art_{i}_title' for i in range(20, 20 + MAX_NEW_SLOTS)]
print(f'  Nombre de colonnes d’articles porté de 19 à {len(TEXT_COLS_RAW)} '
      f"(ajout de données prospectives pour {(_new_slots_wide['date'].notna()).sum()} jours calendaires)")

_dedup_stats = {'total_articles': 0, 'kept_articles': 0}

def _build_canonical_articles(row):
    """
    Construit la collection canonique du jour avec exactement les règles v16 existantes.
    L’ordre des slots et la règle Jaccard restent inchangés, de sorte que daily_text reste identique.
    """
    candidates = []
    for txt_col, ttl_col in zip(TEXT_COLS_RAW, TITLE_COLS_RAW):
        raw_text  = row[txt_col]
        raw_title = row[ttl_col]
        if not is_article_relevant(raw_title, raw_text):
            continue
        cleaned = clean_article_text(raw_text)
        if cleaned is not None:
            candidates.append({
                'text_col': txt_col,
                'title_col': ttl_col,
                'title': raw_title,
                'text': cleaned,
            })
    _dedup_stats['total_articles'] += len(candidates)
    kept = dedup_article_records(candidates)
    _dedup_stats['kept_articles'] += len(kept)
    return kept


def _canonical_articles_to_daily_text(articles):
    if not isinstance(articles, list) or len(articles) == 0:
        return None
    return ' '.join(article['text'] for article in articles)


df_raw['_canonical_articles'] = df_raw.apply(_build_canonical_articles, axis=1)
df_raw['daily_text'] = df_raw['_canonical_articles'].apply(_canonical_articles_to_daily_text)
valid_text_mask = df_raw['daily_text'].notna()
texts = df_raw.loc[valid_text_mask, 'daily_text'].values

print(f'Données brutes : {df_raw.shape[0]} lignes  '
      f'{df_raw["date"].min().date()} ~ {df_raw["date"].max().date()}')
print(f'Jours de négociation comportant du texte : {valid_text_mask.sum()}')
print(f"[Déduplication] Nombre total d’articles après nettoyage : {_dedup_stats['total_articles']}  "
      f"Articles conservés après déduplication : {_dedup_stats['kept_articles']}  "
      f"Taux de doublons : {(1 - _dedup_stats['kept_articles']/max(_dedup_stats['total_articles'],1))*100:.1f}%")

print('\nValeurs manquantes et nombre d’observations disponibles pour les colonnes rv de chaque actif :')
for ticker in TICKERS:
    col = f'rv_{ticker}'
    if col in df_raw.columns:
        n_valid = df_raw[col].notna().sum()
        first_valid = df_raw.loc[df_raw[col].notna(), 'date'].min()
        print(f'  {ticker:6s}  observations disponibles={n_valid:5d}  première date valide={first_valid.date() if pd.notna(first_valid) else "N/A"}')
    else:
        print(f'  {ticker:6s}  ⚠️ Colonne absente ; vérifier que dataset_merged.csv a été généré correctement')

# Indicateurs journaliers prospectifs calculés sur les articles canoniques conservés
print('\n[2.2] Calcul des indicateurs journaliers de contenu prospectif ...')
t0_fwd = time.time()
_fwd_flags = df_raw['_canonical_articles'].apply(compute_daily_fwd_flags)
df_raw = pd.concat([df_raw, _fwd_flags], axis=1)
df_raw['has_fwd_lag1'] = df_raw['has_fwd'].shift(1)
print(f'  → Terminé en {time.time()-t0_fwd:.1f}s')
print(f'  Part des jours de négociation comportant un article prospectif : {(df_raw["has_fwd"]>0).mean()*100:.1f}%')
if (df_raw['has_fwd'] > 0).any():
    print(f'  Nombre moyen d’articles prospectifs par jour, conditionnellement à leur présence : '
          f'{df_raw.loc[df_raw["has_fwd"]>0, "n_fwd_arts"].mean():.2f}')

_WORD_RE = re.compile(r'[A-Za-z]+')

def stem_daily_text(text):
    if not isinstance(text, str):
        return None
    tokens = _WORD_RE.findall(text.lower())
    stemmed = [STEMMER_EN.stem(w) for w in tokens if len(w) > 2]
    return ' '.join(stemmed) if stemmed else None

t0_stem = time.time()
df_raw['daily_text_stemmed'] = df_raw['daily_text'].apply(stem_daily_text)
print(f'Racinisation terminée en {time.time()-t0_stem:.1f}s')

# Fingerprints reproductibles des entrées textuelles. Les noms TF-IDF/W2V existants restent inchangés
# afin d’autoriser la réutilisation des caches historiques lorsque daily_text_stemmed est identique.
def _sha256_lines(lines):
    h = hashlib.sha256()
    for line in lines:
        h.update(str(line).encode('utf-8'))
        h.update(b'\n')
    return h.hexdigest()

TEXT_INPUT_FINGERPRINT = _sha256_lines(
    f"{d.isoformat()}\t{t}" for d, t in zip(
        pd.to_datetime(df_raw['date']), df_raw['daily_text_stemmed'].fillna('')))
ARTICLE_INPUT_FINGERPRINT = _sha256_lines(
    f"{pd.Timestamp(row['date']).isoformat()}\t{article['text_col']}\t{article.get('title', '')}\t{article['text']}"
    for _, row in df_raw.iterrows()
    for article in row['_canonical_articles'])

# Le fingerprint des articles fait partie du nom FinBERT : aucune restauration possible depuis une autre base textuelle.
FINBERT_DAILY_CKPT_NAME = (
    f'finbert_daily_{FINBERT_CACHE_VERSION}_{ARTICLE_INPUT_FINGERPRINT[:12]}')
FINBERT_PARTIAL_CKPT_NAME = (
    f'finbert_articles_partial_{FINBERT_CACHE_VERSION}_{ARTICLE_INPUT_FINGERPRINT[:12]}')

with open(os.path.join(OUT_DIR, 'text_input_fingerprints.json'), 'w', encoding='utf-8') as _f:
    json.dump({
        'text_input_fingerprint': TEXT_INPUT_FINGERPRINT,
        'article_input_fingerprint': ARTICLE_INPUT_FINGERPRINT,
        'n_rows': len(df_raw),
        'n_valid_text_days': int(valid_text_mask.sum()),
        'dedup_jaccard_threshold': DEDUP_JACCARD_THRESHOLD,
    }, _f, ensure_ascii=False, indent=2)
print(f'Fingerprint daily_text_stemmed : {TEXT_INPUT_FINGERPRINT}')
print(f'Fingerprint articles canoniques : {ARTICLE_INPUT_FINGERPRINT}')


# %% ── 3. Route fondée sur le dictionnaire LM (incluant la dispersion du sentiment lm_polarity_std) ───────────────────
print('\n[3] Variables de sentiment du dictionnaire LM (calcul en cours...)')
t0 = time.time()
lm_dict = pd.read_csv(LM_PATH)
LM_NEG  = set(lm_dict.loc[lm_dict['Negative']    > 0, 'Word'].str.upper())
LM_POS  = set(lm_dict.loc[lm_dict['Positive']    > 0, 'Word'].str.upper())
LM_UNC  = set(lm_dict.loc[lm_dict['Uncertainty'] > 0, 'Word'].str.upper())
LM_LIT  = set(lm_dict.loc[lm_dict['Litigious']   > 0, 'Word'].str.upper())
NEGATORS = {'NO','NOT','NONE','NEITHER','NEVER','NOBODY',"N'T"}
TOKEN_RE = re.compile(r"[A-Za-z']+")

def _lm_score_one(text):
    if not isinstance(text, str) or not text.strip():
        return None
    tokens = TOKEN_RE.findall(text.upper())
    n = len(tokens)
    if n == 0: return None
    pos = neg = unc = lit = 0
    for i, tok in enumerate(tokens):
        window  = tokens[max(0, i-3):i]
        negated = any(w in NEGATORS for w in window)
        if tok in LM_POS:
            pos += 0 if negated else 1
            neg += 1 if negated else 0
        elif tok in LM_NEG:
            neg += 1
        if tok in LM_UNC: unc += 1
        if tok in LM_LIT: lit += 1
    return pos, neg, unc, lit, n

rows = []
for _, row in df_raw.iterrows():
    p_sum = n_sum = u_sum = l_sum = w_sum = na = 0
    article_polarities = []
    for article in row['_canonical_articles']:
        res = _lm_score_one(article['text'])
        if res:
            p, n, u, l, w = res
            p_sum += p; n_sum += n; u_sum += u; l_sum += l; w_sum += w; na += 1
            denom_i = max(p + n, 1)
            article_polarities.append((p - n) / denom_i)
    denom  = max(p_sum + n_sum, 1)
    wdenom = max(w_sum, 1)
    rows.append({
        'lm_polarity': (p_sum - n_sum) / denom,
        'lm_neg_prop':  n_sum / wdenom,
        'lm_pos_prop':  p_sum / wdenom,
        'lm_unc_prop':  u_sum / wdenom,
        'lm_lit_prop':  l_sum / wdenom,
        'lm_n_words':   w_sum,
        'lm_n_arts':    na,
        'lm_polarity_std': float(np.std(article_polarities)) if len(article_polarities) >= 2 else np.nan,
    })
lm_feats = pd.DataFrame(rows, index=df_raw.index)
df_raw   = pd.concat([df_raw, lm_feats], axis=1)
print(f'  → Terminé en {time.time()-t0:.1f}s')
print(df_raw[['lm_polarity','lm_neg_prop','lm_pos_prop','lm_polarity_std']].describe().round(4))

# %% ── 3.5 [ajout v8] Polarité LM propre aux articles prospectifs (fwd_lm_polarity) ────────────────
print('\n[3.5] Polarité LM des articles prospectifs (fwd_lm_polarity) ...')
t0 = time.time()
df_raw['fwd_lm_polarity'] = df_raw['_canonical_articles'].apply(
    lambda articles: compute_fwd_lm_polarity(articles, _lm_score_one))
# [important] Aucun ffill/interpolate ici : NaN signifie « aucun article prospectif ce jour-là », ce qui est une information réelle,
# et non une donnée manquante. La valeur est explicitement fixée à 0 et has_fwd_lag1 est inclus comme contrôle dans la section [9]
# au sein de la route FWD_SENT, afin de distinguer « aucun article prospectif » de « article prospectif au sentiment neutre »,
# et d’éviter le problème de non-alignement des fenêtres TF-IDF de la v4 qui rendait les résultats OOS incomparables.
n_before_fill = df_raw['fwd_lm_polarity'].notna().sum()
df_raw['fwd_lm_polarity'] = df_raw['fwd_lm_polarity'].fillna(0)
print(f'  → Terminé en {time.time()-t0:.1f}s ; '
      f'nombre de jours avec article prospectif : {n_before_fill} (les autres valeurs sont fixées à 0)')

# %% ── 4. TF-IDF + LassoCV (les folds dégénérés restent prédits ; winsorisation utilisée pour maîtriser les ruptures d’échelle) ────
#      [nouveau dans cette version] Sauvegarde de checkpoints analogue à FinBERT : un checkpoint complet après le walk-forward,
#      puis un checkpoint partiel après chaque fold. En cas d’interruption,
#      l’exécution reprend automatiquement au dernier fold terminé.
print('\n[4] TF-IDF + LassoCV (réestimation walk-forward par blocs, strictement sans fuite) (calcul en cours...)')
t0 = time.time()

REFIT_STEP = 252
TEXT_COL_FOR_TFIDF = 'daily_text_stemmed'
ALPHA_GRID = np.logspace(-6, 0, 40)

n_total = len(df_raw)
boundaries = list(range(MIN_TRAIN_OOS, n_total, REFIT_STEP))
if boundaries[-1] != n_total:
    boundaries.append(n_total)
print(f'  Bornes des blocs (positions dans df_raw) : {boundaries}')
print(f'  Dates correspondantes : {[df_raw["date"].iloc[min(b, n_total-1)].date() for b in boundaries]}')


def _fit_tfidf_lasso(train_mask):
    texts_tr = df_raw.loc[train_mask, TEXT_COL_FOR_TFIDF].values
    y_tr     = df_raw.loc[train_mask, 'rv_SPY'].values
    vec_ = TfidfVectorizer(
        max_features=3000, stop_words='english',
        min_df=3, max_df=0.9, ngram_range=(1, 2), sublinear_tf=True,
    )
    X_tr = vec_.fit_transform(texts_tr)
    lasso_ = LassoCV(cv=5, max_iter=5000, n_jobs=-1, random_state=42, alphas=ALPHA_GRID)
    lasso_.fit(X_tr, y_tr)
    return vec_, lasso_, len(texts_tr)


def _top_features(vec_, lasso_, k=15):
    names_ = vec_.get_feature_names_out()
    coefs_ = lasso_.coef_
    nz_ = np.where(coefs_ != 0)[0]
    top_pos_ = sorted(zip(names_[nz_], coefs_[nz_]), key=lambda x: -x[1])[:k]
    top_neg_ = sorted(zip(names_[nz_], coefs_[nz_]), key=lambda x: x[1])[:k]
    return top_pos_, top_neg_, len(nz_)


TFIDF_CKPT_NAME     = 'tfidf_walkforward_v8'
TFIDF_PARTIAL_CKPT  = ckpt_path('tfidf_partial_v8', 'pkl')
SKIP_TFIDF = ckpt_exists(TFIDF_CKPT_NAME, 'pkl', FORCE_RERUN_TFIDF)
_tfidf_ckpt = None
if SKIP_TFIDF:
    _tfidf_ckpt = load_ckpt(TFIDF_CKPT_NAME, 'pkl')
    _cached_fp = _tfidf_ckpt.get('input_fingerprint') if isinstance(_tfidf_ckpt, dict) else None
    _cached_len = len(_tfidf_ckpt.get('tfidf_idx_series', [])) if isinstance(_tfidf_ckpt, dict) else -1
    if _cached_len != len(df_raw):
        print('  ⚠️  Longueur du cache TF-IDF incompatible ; recalcul obligatoire.')
        SKIP_TFIDF = False
        _tfidf_ckpt = None
    elif _cached_fp is not None and _cached_fp != TEXT_INPUT_FINGERPRINT:
        print('  ⚠️  Fingerprint TF-IDF incompatible ; recalcul obligatoire.')
        SKIP_TFIDF = False
        _tfidf_ckpt = None
    elif _cached_fp is None:
        print('  ⚠️  Cache TF-IDF historique sans fingerprint : réutilisation autorisée car la logique daily_text est inchangée.')

if SKIP_TFIDF:
    print('  ← ✅ Restauration depuis le checkpoint v8 ; recalcul walk-forward TF-IDF+LassoCV ignoré')
    tfidf_idx_series   = _tfidf_ckpt['tfidf_idx_series']
    tfidf_fold_records = _tfidf_ckpt['tfidf_fold_records']
    n_degenerate_folds = _tfidf_ckpt['n_degenerate_folds']
else:
    _resume_tfidf = False
    if (os.path.exists(TFIDF_PARTIAL_CKPT)
            and not FORCE_RERUN_TEXT_MODELS
            and not FORCE_RERUN_TFIDF):
        with open(TFIDF_PARTIAL_CKPT, 'rb') as f:
            _tfidf_partial = pickle.load(f)
        _resume_tfidf = (
            _tfidf_partial.get('input_fingerprint') == TEXT_INPUT_FINGERPRINT
            and len(_tfidf_partial.get('tfidf_idx_series', [])) == len(df_raw))
        if not _resume_tfidf:
            print('  ⚠️  Checkpoint TF-IDF partiel sans fingerprint compatible ; reprise ignorée.')
    if _resume_tfidf:
        start_fold_tfidf   = _tfidf_partial['next_fold']
        tfidf_idx_series   = _tfidf_partial['tfidf_idx_series']
        tfidf_fold_records = _tfidf_partial['tfidf_fold_records']
        n_degenerate_folds = _tfidf_partial['n_degenerate_folds']
        print(f'  ↺ Reprise de TF-IDF au fold {start_fold_tfidf}/{len(boundaries)-1}')
    else:
        start_fold_tfidf   = 0
        tfidf_idx_series   = pd.Series(np.nan, index=df_raw.index)
        tfidf_fold_records = []
        n_degenerate_folds = 0

    for i, b in enumerate(boundaries[:-1]):
        if i < start_fold_tfidf:
            continue

        train_mask = valid_text_mask & (df_raw.index < b)
        vec_i, lasso_i, n_train_i = _fit_tfidf_lasso(train_mask)
        top_pos_i, top_neg_i, n_nz_i = _top_features(vec_i, lasso_i)

        is_degenerate = n_nz_i < TFIDF_MIN_NONZERO

        fill_ranges = []
        if i == 0:
            fill_ranges.append((0, b))
        next_b = boundaries[i + 1]
        fill_ranges.append((b, next_b))

        for r_start, r_end in fill_ranges:
            predict_mask = (valid_text_mask
                             & (df_raw.index >= r_start)
                             & (df_raw.index < r_end))
            texts_pred = df_raw.loc[predict_mask, TEXT_COL_FOR_TFIDF].values
            if len(texts_pred) == 0:
                continue
            X_pred = vec_i.transform(texts_pred)
            tfidf_idx_series.loc[predict_mask] = lasso_i.predict(X_pred)
        if is_degenerate:
            n_degenerate_folds += 1

        train_end_date = df_raw['date'].iloc[b-1].date()
        tfidf_fold_records.append({
            'fold': i, 'train_end_date': train_end_date,
            'n_train_docs': n_train_i, 'n_nonzero': n_nz_i,
            'is_degenerate': is_degenerate,
            'top_pos': top_pos_i, 'top_neg': top_neg_i,
        })
        print(f'  Fold {i+1}/{len(boundaries)-1}  entraînement jusqu’au {train_end_date}  '
              f'documents d’entraînement={n_train_i}  variables non nulles={n_nz_i}/3000  α={lasso_i.alpha_:.6f}  '
              f'{"⚠️ dégénéré (prédiction conservée et signalée)" if is_degenerate else "normal"}')

        with open(TFIDF_PARTIAL_CKPT, 'wb') as f:
            pickle.dump({
                'next_fold': i + 1,
                'tfidf_idx_series': tfidf_idx_series,
                'tfidf_fold_records': tfidf_fold_records,
                'n_degenerate_folds': n_degenerate_folds,
                'input_fingerprint': TEXT_INPUT_FINGERPRINT,
            }, f, protocol=4)

    save_ckpt(TFIDF_CKPT_NAME, {
        'tfidf_idx_series': tfidf_idx_series,
        'tfidf_fold_records': tfidf_fold_records,
        'n_degenerate_folds': n_degenerate_folds,
        'input_fingerprint': TEXT_INPUT_FINGERPRINT,
    }, 'pkl')
    if os.path.exists(TFIDF_PARTIAL_CKPT):
        os.remove(TFIDF_PARTIAL_CKPT)

df_raw['tfidf_idx'] = tfidf_idx_series
df_raw['tfidf_idx'] = df_raw['tfidf_idx'].ffill()   # [correction v9 n°2] ffill remplace l’interpolation linéaire bidirectionnelle

def causal_expanding_winsorize(series, lower_q=0.01, upper_q=0.99,
                                 min_history=MIN_TRAIN_OOS):
    """
    Calcule les seuils de winsorisation uniquement à partir des valeurs historiques antérieures à t, afin d’éviter de réinjecter l’information de distribution future.
    Les min_history premières observations historiques valides ne sont pas winsorisées ; ensuite, la valeur courante est bornée par les quantiles historiques à 1 % et 99 %.
    """
    s = pd.to_numeric(series, errors='coerce').copy()
    hist = s.shift(1)
    lo = hist.expanding(min_periods=min_history).quantile(lower_q)
    hi = hist.expanding(min_periods=min_history).quantile(upper_q)
    valid_threshold = lo.notna() & hi.notna() & s.notna()
    out = s.copy()
    out.loc[valid_threshold] = np.minimum(
        np.maximum(s.loc[valid_threshold], lo.loc[valid_threshold]),
        hi.loc[valid_threshold],
    )
    n_clipped = int((valid_threshold & (out != s)).sum())
    return out, lo, hi, n_clipped


df_raw['tfidf_idx'], _tfidf_lo_hist, _tfidf_hi_hist, _n_winsorized = \
    causal_expanding_winsorize(df_raw['tfidf_idx'])
print(f'  → Walk-forward terminé en {time.time()-t0:.1f}s ; '
      f'{len(tfidf_fold_records)} folds au total, dont {n_degenerate_folds} folds dégénérés (signalés mais conservés pour la prédiction) ; '
      f'{_n_winsorized} observations bornées par winsorisation historique causale '
      f'(seuils calculés uniquement à partir des observations antérieures à chaque date)')

_fold_summary = pd.DataFrame([
    {'fold': r['fold'], 'train_end_date': r['train_end_date'],
     'n_train_docs': r['n_train_docs'], 'n_nonzero': r['n_nonzero'],
     'is_degenerate': r['is_degenerate']}
    for r in tfidf_fold_records
])
print('\n  Évolution du nombre de variables non nulles par fold :')
print(_fold_summary.to_string(index=False))

print('\n  Ajustement supplémentaire d’un modèle descriptif sur l’échantillon complet (uniquement pour présenter/interpréter le vocabulaire ; exclu de toute prédiction et évaluation) ...')
vec_full, lasso_full, n_full = _fit_tfidf_lasso(valid_text_mask)
top_pos_full, top_neg_full, n_nz_full = _top_features(vec_full, lasso_full)
tfidf_full_sample_top = {
    'train_end_date': df_raw['date'].iloc[-1].date(),
    'n_train_docs': n_full, 'n_nonzero': n_nz_full,
    'top_pos': top_pos_full, 'top_neg': top_neg_full,
}
print(f'  Modèle descriptif sur échantillon complet : documents d’entraînement={n_full}, variables non nulles={n_nz_full}/3000')


# [ajout v14.2] Export et visualisation des termes TF-IDF importants.
# Deux lectures complémentaires sont produites :
#   (1) classement descriptif sur l'échantillon complet (jamais utilisé pour l'évaluation OOS) ;
#   (2) stabilité walk-forward : fréquence d'apparition parmi les 15 termes positifs/négatifs de chaque fold.
def export_tfidf_term_rankings(tfidf_full_sample_top, tfidf_fold_records, out_dir, top_k_plot=15):
    full_rows = []
    for direction, key in [('positive', 'top_pos'), ('negative', 'top_neg')]:
        for rank, (term, coef) in enumerate(tfidf_full_sample_top.get(key, []), start=1):
            full_rows.append({
                'sample': 'full_sample_descriptive',
                'direction': direction,
                'rank': rank,
                'term': term,
                'coefficient': float(coef),
                'abs_coefficient': abs(float(coef)),
                'n_train_docs': tfidf_full_sample_top.get('n_train_docs'),
                'n_nonzero': tfidf_full_sample_top.get('n_nonzero'),
                'train_end_date': tfidf_full_sample_top.get('train_end_date'),
            })
    df_full_terms = pd.DataFrame(full_rows)
    df_full_terms.to_csv(
        os.path.join(out_dir, 'diag_tfidf_top_terms_full_sample.csv'), index=False)

    if not df_full_terms.empty:
        pos = (df_full_terms[df_full_terms['direction'] == 'positive']
               .nsmallest(top_k_plot, 'rank').sort_values('coefficient'))
        neg = (df_full_terms[df_full_terms['direction'] == 'negative']
               .nsmallest(top_k_plot, 'rank').sort_values('coefficient', ascending=False))
        fig, axes = plt.subplots(1, 2, figsize=(15, max(5, 0.34 * top_k_plot + 2)))
        axes[0].barh(neg['term'], neg['coefficient'])
        axes[0].axvline(0, color='black', lw=0.8)
        axes[0].set_title('TF-IDF–Lasso : termes à coefficient négatif')
        axes[0].set_xlabel('Coefficient Lasso')
        axes[1].barh(pos['term'], pos['coefficient'])
        axes[1].axvline(0, color='black', lw=0.8)
        axes[1].set_title('TF-IDF–Lasso : termes à coefficient positif')
        axes[1].set_xlabel('Coefficient Lasso')
        fig.suptitle('Classement descriptif sur échantillon complet — exclu des prévisions OOS', y=1.01)
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, 'diag_tfidf_top_terms_full_sample.png'),
                    dpi=180, bbox_inches='tight')
        plt.close(fig)

    fold_rows = []
    for rec in tfidf_fold_records:
        for direction, key in [('positive', 'top_pos'), ('negative', 'top_neg')]:
            for rank, (term, coef) in enumerate(rec.get(key, []), start=1):
                fold_rows.append({
                    'fold': rec.get('fold'),
                    'train_end_date': rec.get('train_end_date'),
                    'n_train_docs': rec.get('n_train_docs'),
                    'n_nonzero': rec.get('n_nonzero'),
                    'is_degenerate': rec.get('is_degenerate'),
                    'direction': direction,
                    'rank_within_fold': rank,
                    'term': term,
                    'coefficient': float(coef),
                    'abs_coefficient': abs(float(coef)),
                })
    df_fold_terms = pd.DataFrame(fold_rows)
    df_fold_terms.to_csv(
        os.path.join(out_dir, 'diag_tfidf_top_terms_by_fold.csv'), index=False)

    if not df_fold_terms.empty:
        stability = (df_fold_terms.groupby('term', as_index=False)
                     .agg(n_appearances=('fold', 'nunique'),
                          mean_coefficient=('coefficient', 'mean'),
                          mean_abs_coefficient=('abs_coefficient', 'mean'),
                          max_abs_coefficient=('abs_coefficient', 'max'),
                          first_fold=('fold', 'min'),
                          last_fold=('fold', 'max')))
        stability['mean_direction'] = np.where(
            stability['mean_coefficient'] >= 0, 'positive', 'negative')
        stability = stability.sort_values(
            ['n_appearances', 'mean_abs_coefficient'], ascending=[False, False])
        stability.to_csv(
            os.path.join(out_dir, 'diag_tfidf_term_stability_walkforward.csv'), index=False)

        top_stable = stability.head(20).sort_values(
            ['n_appearances', 'mean_abs_coefficient'], ascending=True)
        fig, ax = plt.subplots(figsize=(10, max(6, 0.35 * len(top_stable) + 2)))
        ax.barh(top_stable['term'], top_stable['n_appearances'])
        for i, (_, row) in enumerate(top_stable.iterrows()):
            ax.text(row['n_appearances'] + 0.05, i,
                    f"moy. β={row['mean_coefficient']:.4g}",
                    va='center', fontsize=8)
        ax.set_xlabel("Nombre de folds où le terme figure dans le top 15")
        ax.set_ylabel('Terme ou bigramme stemmé')
        ax.set_title('Stabilité walk-forward des termes TF-IDF–Lasso')
        ax.set_xlim(0, max(top_stable['n_appearances'].max() + 1, 2))
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, 'diag_tfidf_term_stability_walkforward.png'),
                    dpi=180, bbox_inches='tight')
        plt.close(fig)
    else:
        stability = pd.DataFrame()

    return df_full_terms, df_fold_terms, stability


df_tfidf_terms_full, df_tfidf_terms_folds, df_tfidf_term_stability = \
    export_tfidf_term_rankings(tfidf_full_sample_top, tfidf_fold_records, OUT_DIR)
print('  → Classements TF-IDF exportés : termes positifs/négatifs du modèle descriptif complet '
      'et stabilité des termes dans les folds walk-forward.')


# %% ── 5. Word2Vec + PCA dynamique, alignée entre les folds ──────────────────
print('\n[5] Word2Vec + PCA walk-forward alignée entre les folds (calcul en cours...)')
t0 = time.time()
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

W2V_PC_COLS = [f'w2v_pc{i+1}' for i in range(5)]
W2V_TOP_WORDS_K = 15
W2V_MIN_COMMON_ALIGN = 100


def doc2vec_mean(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)


def _row_normalize(arr, eps=1e-12):
    arr = np.asarray(arr, dtype=float)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    return arr / np.maximum(norms, eps)


def _top_words_for_pc(w2v_model, scaler_model, pca_model, pc_index_1based, k=15):
    """
    Projection descriptive approximative des vecteurs de mots sur la direction d’une PC
    estimée sur les vecteurs moyens de documents. Cette lecture sert à l’interprétation,
    non à la construction des prévisions.
    """
    vocab = w2v_model.wv.index_to_key
    vecs = np.asarray(w2v_model.wv.vectors)
    vecs_std = scaler_model.transform(vecs)
    scores = vecs_std @ pca_model.components_[pc_index_1based - 1]
    order = np.argsort(scores)
    top_neg = [(vocab[i], float(scores[i])) for i in order[:k]]
    top_pos = [(vocab[i], float(scores[i])) for i in order[::-1][:k]]
    return top_pos, top_neg


def _align_w2v_model_to_anchor(w2v_model, anchor_state):
    """
    Aligne l’espace courant sur l’espace du premier fold par Procrustes orthogonal
    à partir du vocabulaire commun. L’opération conserve les produits scalaires
    internes au modèle courant et ne modifie donc pas sa géométrie relative.
    """
    if anchor_state is None:
        state = {
            'anchor_words': list(w2v_model.wv.index_to_key),
            'anchor_vectors': np.asarray(w2v_model.wv.vectors, dtype=np.float32).copy(),
            'anchor_pc_dirs_raw': None,
        }
        return w2v_model, state, {
            'n_common_vocab': len(state['anchor_words']),
            'mean_common_cosine': 1.0,
        }

    anchor_words = anchor_state['anchor_words']
    anchor_vectors = np.asarray(anchor_state['anchor_vectors'])
    anchor_index = {w: j for j, w in enumerate(anchor_words)}
    current_index = w2v_model.wv.key_to_index
    common = [w for w in anchor_words if w in current_index]

    if len(common) < W2V_MIN_COMMON_ALIGN:
        raise RuntimeError(
            f'Vocabulaire commun insuffisant pour aligner Word2Vec : {len(common)} '
            f'< {W2V_MIN_COMMON_ALIGN}'
        )

    x = np.vstack([w2v_model.wv[w] for w in common])
    y = np.vstack([anchor_vectors[anchor_index[w]] for w in common])
    rotation, _ = orthogonal_procrustes(_row_normalize(x), _row_normalize(y))
    w2v_model.wv.vectors[:] = np.asarray(w2v_model.wv.vectors) @ rotation

    x_aligned = np.vstack([w2v_model.wv[w] for w in common])
    cos = np.sum(_row_normalize(x_aligned) * _row_normalize(y), axis=1)
    return w2v_model, anchor_state, {
        'n_common_vocab': len(common),
        'mean_common_cosine': float(np.mean(cos)),
    }


def _pca_raw_directions(pca_model, scaler_model):
    scale = np.asarray(scaler_model.scale_, dtype=float)
    safe_scale = np.where(np.abs(scale) > 1e-12, scale, 1.0)
    raw_dirs = np.asarray(pca_model.components_, dtype=float) / safe_scale[None, :]
    return _row_normalize(raw_dirs)


def _align_pca_components_to_anchor(pca_model, scaler_model, anchor_state):
    """
    Apparente les PC courantes aux PC du premier fold par similarité cosinus absolue,
    puis corrige leur signe. Les étiquettes PC1...PC5 conservent ainsi une identité
    approximativement stable malgré la réestimation walk-forward.
    """
    cur_dirs = _pca_raw_directions(pca_model, scaler_model)

    if anchor_state.get('anchor_pc_dirs_raw') is None:
        anchor_state['anchor_pc_dirs_raw'] = cur_dirs.copy()
        return pca_model, anchor_state, {
            'component_permutation': list(range(len(cur_dirs))),
            'component_signs': [1] * len(cur_dirs),
            'component_match_abs_cosine': [1.0] * len(cur_dirs),
        }

    anchor_dirs = np.asarray(anchor_state['anchor_pc_dirs_raw'])
    similarity = anchor_dirs @ cur_dirs.T
    row_ind, col_ind = linear_sum_assignment(-np.abs(similarity))

    permutation = np.empty(len(anchor_dirs), dtype=int)
    signs = np.ones(len(anchor_dirs), dtype=int)
    match_quality = np.empty(len(anchor_dirs), dtype=float)
    for r, c in zip(row_ind, col_ind):
        permutation[r] = c
        signs[r] = 1 if similarity[r, c] >= 0 else -1
        match_quality[r] = abs(similarity[r, c])

    pca_model.components_ = pca_model.components_[permutation] * signs[:, None]
    for attr in ('explained_variance_', 'explained_variance_ratio_', 'singular_values_'):
        if hasattr(pca_model, attr):
            setattr(pca_model, attr, getattr(pca_model, attr)[permutation])

    return pca_model, anchor_state, {
        'component_permutation': permutation.tolist(),
        'component_signs': signs.tolist(),
        'component_match_abs_cosine': match_quality.tolist(),
    }


def _collect_fold_pc_words(w2v_model, scaler_model, pca_model, k=W2V_TOP_WORDS_K):
    out = {}
    for pc in range(1, len(W2V_PC_COLS) + 1):
        top_pos, top_neg = _top_words_for_pc(
            w2v_model, scaler_model, pca_model, pc, k=k)
        out[f'PC{pc}'] = {'top_pos': top_pos, 'top_neg': top_neg}
    return out


def export_w2v_dynamic_diagnostics(w2v_fold_records, out_dir):
    word_rows, align_rows = [], []
    for rec in w2v_fold_records:
        align_rows.append({
            'fold': rec['fold'],
            'train_end_date': rec['train_end_date'],
            'n_train_docs': rec['n_train_docs'],
            'vocab_size': rec['vocab_size'],
            'n_common_vocab': rec.get('n_common_vocab'),
            'mean_common_cosine': rec.get('mean_common_cosine'),
            'pc_explained_var': json.dumps(rec.get('pc_explained_var', [])),
            'component_permutation': json.dumps(rec.get('component_permutation', [])),
            'component_signs': json.dumps(rec.get('component_signs', [])),
            'component_match_abs_cosine': json.dumps(
                rec.get('component_match_abs_cosine', [])),
        })
        for pc, directions in rec.get('pc_top_words', {}).items():
            for direction, key in [('positive', 'top_pos'), ('negative', 'top_neg')]:
                for rank, (word, score) in enumerate(directions.get(key, []), start=1):
                    word_rows.append({
                        'fold': rec['fold'],
                        'train_end_date': rec['train_end_date'],
                        'pc': pc,
                        'direction': direction,
                        'rank': rank,
                        'word': word,
                        'score': float(score),
                    })

    df_words = pd.DataFrame(word_rows)
    df_align = pd.DataFrame(align_rows)
    df_words.to_csv(
        os.path.join(out_dir, 'diag_w2v_pc_top_words_by_fold.csv'), index=False)
    df_align.to_csv(
        os.path.join(out_dir, 'diag_w2v_pc_alignment_by_fold.csv'), index=False)

    if not df_words.empty:
        stability = (df_words.groupby(['pc', 'direction', 'word'], as_index=False)
                     .agg(n_folds=('fold', 'nunique'),
                          mean_rank=('rank', 'mean'),
                          mean_score=('score', 'mean'),
                          mean_abs_score=('score', lambda x: float(np.mean(np.abs(x)))),
                          first_fold=('fold', 'min'),
                          last_fold=('fold', 'max')))
        stability = stability.sort_values(
            ['pc', 'direction', 'n_folds', 'mean_rank'],
            ascending=[True, True, False, True])
        stability.to_csv(
            os.path.join(out_dir, 'diag_w2v_pc_keyword_stability.csv'), index=False)
    else:
        stability = pd.DataFrame()

    return df_words, df_align, stability


W2V_CKPT_NAME    = 'w2v_pca_walkforward_v15_aligned'
W2V_PARTIAL_CKPT = ckpt_path('w2v_pca_partial_v15_aligned', 'pkl')
SKIP_W2V = ckpt_exists(W2V_CKPT_NAME, 'pkl', FORCE_RERUN_W2V)
_w2v_ckpt = None
if SKIP_W2V:
    _w2v_ckpt = load_ckpt(W2V_CKPT_NAME, 'pkl')
    _cached_fp = _w2v_ckpt.get('input_fingerprint') if isinstance(_w2v_ckpt, dict) else None
    _cached_series = _w2v_ckpt.get('w2v_pcs_series', {}) if isinstance(_w2v_ckpt, dict) else {}
    _cached_len = len(next(iter(_cached_series.values()))) if _cached_series else -1
    if _cached_len != len(df_raw):
        print('  ⚠️  Longueur du cache Word2Vec incompatible ; recalcul obligatoire.')
        SKIP_W2V = False
        _w2v_ckpt = None
    elif _cached_fp is not None and _cached_fp != TEXT_INPUT_FINGERPRINT:
        print('  ⚠️  Fingerprint Word2Vec incompatible ; recalcul obligatoire.')
        SKIP_W2V = False
        _w2v_ckpt = None
    elif _cached_fp is None:
        print('  ⚠️  Cache Word2Vec historique sans fingerprint : réutilisation autorisée car la logique daily_text est inchangée.')

if SKIP_W2V:
    print('  ← ✅ Restauration depuis le checkpoint v15 aligné ; recalcul walk-forward Word2Vec+PCA ignoré')
    w2v_pcs_series      = _w2v_ckpt['w2v_pcs_series']
    w2v_fold_records    = _w2v_ckpt['w2v_fold_records']
    w2v_alignment_state = _w2v_ckpt['w2v_alignment_state']
else:
    _resume_w2v = False
    if (os.path.exists(W2V_PARTIAL_CKPT)
            and not FORCE_RERUN_TEXT_MODELS
            and not FORCE_RERUN_W2V):
        with open(W2V_PARTIAL_CKPT, 'rb') as f:
            _w2v_partial = pickle.load(f)
        _partial_series = _w2v_partial.get('w2v_pcs_series', {})
        _partial_len = len(next(iter(_partial_series.values()))) if _partial_series else -1
        _resume_w2v = (
            _w2v_partial.get('input_fingerprint') == TEXT_INPUT_FINGERPRINT
            and _partial_len == len(df_raw))
        if not _resume_w2v:
            print('  ⚠️  Checkpoint Word2Vec partiel sans fingerprint compatible ; reprise ignorée.')
    if _resume_w2v:
        start_fold_w2v      = _w2v_partial['next_fold']
        w2v_pcs_series      = _w2v_partial['w2v_pcs_series']
        w2v_fold_records    = _w2v_partial['w2v_fold_records']
        w2v_alignment_state = _w2v_partial['w2v_alignment_state']
        print(f'  ↺ Reprise de Word2Vec+PCA au fold {start_fold_w2v}/{len(boundaries)-1}')
    else:
        start_fold_w2v      = 0
        w2v_pcs_series      = {col: pd.Series(np.nan, index=df_raw.index)
                               for col in W2V_PC_COLS}
        w2v_fold_records    = []
        w2v_alignment_state = None

    for i, b in enumerate(boundaries[:-1]):
        if i < start_fold_w2v:
            continue

        train_mask = valid_text_mask & (df_raw.index < b)
        texts_train_i = df_raw.loc[train_mask, TEXT_COL_FOR_TFIDF].values
        token_corpus_train_i = [
            simple_preprocess(t, min_len=3) for t in texts_train_i]

        w2v_i = Word2Vec(
            sentences=token_corpus_train_i, vector_size=100, window=5,
            min_count=5, sg=1, workers=1, epochs=10, seed=42,
        )
        w2v_i, w2v_alignment_state, embedding_diag = \
            _align_w2v_model_to_anchor(w2v_i, w2v_alignment_state)

        doc_vecs_train_i = np.vstack([
            doc2vec_mean(t, w2v_i) for t in token_corpus_train_i])
        scaler_i = StandardScaler().fit(doc_vecs_train_i)
        pca_i = PCA(n_components=5, random_state=42)
        pca_i.fit(scaler_i.transform(doc_vecs_train_i))
        pca_i, w2v_alignment_state, pca_diag = \
            _align_pca_components_to_anchor(
                pca_i, scaler_i, w2v_alignment_state)

        fold_top_words = _collect_fold_pc_words(
            w2v_i, scaler_i, pca_i, k=W2V_TOP_WORDS_K)

        fill_ranges = []
        if i == 0:
            fill_ranges.append((0, b))
        next_b = boundaries[i + 1]
        fill_ranges.append((b, next_b))

        for r_start, r_end in fill_ranges:
            predict_mask = (valid_text_mask
                            & (df_raw.index >= r_start)
                            & (df_raw.index < r_end))
            texts_pred = df_raw.loc[predict_mask, TEXT_COL_FOR_TFIDF].values
            if len(texts_pred) == 0:
                continue
            token_corpus_pred = [
                simple_preprocess(t, min_len=3) for t in texts_pred]
            doc_vecs_pred = np.vstack([
                doc2vec_mean(t, w2v_i) for t in token_corpus_pred])
            pcs_pred = pca_i.transform(scaler_i.transform(doc_vecs_pred))
            for j, col in enumerate(W2V_PC_COLS):
                w2v_pcs_series[col].loc[predict_mask] = pcs_pred[:, j]

        train_end_date = df_raw['date'].iloc[b-1].date()
        w2v_fold_records.append({
            'fold': i,
            'train_end_date': train_end_date,
            'n_train_docs': len(texts_train_i),
            'vocab_size': len(w2v_i.wv),
            'pc_explained_var': pca_i.explained_variance_ratio_.round(6).tolist(),
            'pca_total_var': float(pca_i.explained_variance_ratio_.sum()),
            'pc_top_words': fold_top_words,
            **embedding_diag,
            **pca_diag,
        })

        print(f'  Fold {i+1}/{len(boundaries)-1}  entraînement jusqu’au {train_end_date}  '
              f'documents={len(texts_train_i)}  vocabulaire={len(w2v_i.wv)}  '
              f'cosinus moyen d’alignement={embedding_diag["mean_common_cosine"]:.3f}  '
              f'qualité PC min={min(pca_diag["component_match_abs_cosine"]):.3f}  '
              f'variance expliquée totale={pca_i.explained_variance_ratio_.sum():.3f}')

        with open(W2V_PARTIAL_CKPT, 'wb') as f:
            pickle.dump({
                'next_fold': i + 1,
                'w2v_pcs_series': w2v_pcs_series,
                'w2v_fold_records': w2v_fold_records,
                'w2v_alignment_state': w2v_alignment_state,
                'input_fingerprint': TEXT_INPUT_FINGERPRINT,
            }, f, protocol=4)

    save_ckpt(W2V_CKPT_NAME, {
        'w2v_pcs_series': w2v_pcs_series,
        'w2v_fold_records': w2v_fold_records,
        'w2v_alignment_state': w2v_alignment_state,
        'input_fingerprint': TEXT_INPUT_FINGERPRINT,
    }, 'pkl')
    if os.path.exists(W2V_PARTIAL_CKPT):
        os.remove(W2V_PARTIAL_CKPT)

for col in W2V_PC_COLS:
    df_raw[col] = w2v_pcs_series[col]
    df_raw[col] = df_raw[col].ffill()

_w2v_fold_summary = pd.DataFrame([{
    'fold': r['fold'],
    'train_end_date': r['train_end_date'],
    'n_train_docs': r['n_train_docs'],
    'vocab_size': r['vocab_size'],
    'mean_common_cosine': r.get('mean_common_cosine'),
    'min_pc_match': min(r.get('component_match_abs_cosine', [np.nan])),
    'pca_total_var': r.get('pca_total_var'),
} for r in w2v_fold_records])

print('\n  Évolution du vocabulaire et qualité de l’alignement Word2Vec/PCA :')
print(_w2v_fold_summary.to_string(index=False))

df_w2v_words_by_fold, df_w2v_alignment, df_w2v_keyword_stability = \
    export_w2v_dynamic_diagnostics(w2v_fold_records, OUT_DIR)

if w2v_fold_records:
    _latest = w2v_fold_records[-1]
    print(f'\n  Mots dominants du dernier fold (fin d’entraînement : {_latest["train_end_date"]}) :')
    for _pc in range(1, 6):
        _rec = _latest['pc_top_words'][f'PC{_pc}']
        print(f'    PC{_pc} + : {", ".join(w for w, _ in _rec["top_pos"][:8])}')
        print(f'    PC{_pc} - : {", ".join(w for w, _ in _rec["top_neg"][:8])}')

print(f'  → Word2Vec+PCA alignée terminée en {time.time()-t0:.1f}s')


# %% ── 5.5 Modèle descriptif complet aligné sur les PC du premier fold ──
print('\n[5.5] Modèle descriptif Word2Vec+PCA sur l’échantillon complet, aligné sur la base walk-forward ...')
t0_w2v_full = time.time()

_token_corpus_full = [
    simple_preprocess(t, min_len=3)
    for t in df_raw.loc[valid_text_mask, TEXT_COL_FOR_TFIDF].values
]
w2v_full = Word2Vec(
    sentences=_token_corpus_full, vector_size=100, window=5,
    min_count=5, sg=1, workers=1, epochs=10, seed=42,
)
w2v_full, _, _full_embedding_diag = \
    _align_w2v_model_to_anchor(w2v_full, w2v_alignment_state)

doc_vecs_full = np.vstack([
    doc2vec_mean(t, w2v_full) for t in _token_corpus_full])
scaler_full = StandardScaler().fit(doc_vecs_full)
pca_full = PCA(n_components=5, random_state=42)
pca_full.fit(scaler_full.transform(doc_vecs_full))
pca_full, _, _full_pca_diag = \
    _align_pca_components_to_anchor(
        pca_full, scaler_full, w2v_alignment_state)

print(f'  Modèle descriptif complet : documents={len(_token_corpus_full)}, '
      f'vocabulaire={len(w2v_full.wv)}, '
      f'cosinus moyen d’alignement={_full_embedding_diag["mean_common_cosine"]:.3f}, '
      f'qualité PC min={min(_full_pca_diag["component_match_abs_cosine"]):.3f}, '
      f'variance expliquée totale={pca_full.explained_variance_ratio_.sum():.3f}')

print('\n  Mots à forte charge sur PC1~PC5 — modèle complet aligné, descriptif uniquement :')
w2v_pc_top_words = {}
_pc_words_records = []
for _pc in range(1, 6):
    _top_pos, _top_neg = _top_words_for_pc(
        w2v_full, scaler_full, pca_full, _pc, k=W2V_TOP_WORDS_K)
    w2v_pc_top_words[f'PC{_pc}'] = {
        'top_pos': _top_pos, 'top_neg': _top_neg}
    print(f'    PC{_pc} — charge positive : {", ".join(w for w, _ in _top_pos)}')
    print(f'    PC{_pc} — charge négative : {", ".join(w for w, _ in _top_neg)}')
    for rank, (w, s) in enumerate(_top_pos, 1):
        _pc_words_records.append({
            'pc': f'PC{_pc}', 'direction': 'positive',
            'rank': rank, 'word': w, 'score': s})
    for rank, (w, s) in enumerate(_top_neg, 1):
        _pc_words_records.append({
            'pc': f'PC{_pc}', 'direction': 'negative',
            'rank': rank, 'word': w, 'score': s})

pd.DataFrame(_pc_words_records).to_csv(
    os.path.join(OUT_DIR, 'diag_w2v_pc_top_words.csv'), index=False)
print(f'  → Terminé en {time.time()-t0_w2v_full:.1f}s ; '
      'sorties : diag_w2v_pc_top_words_by_fold.csv, '
      'diag_w2v_pc_keyword_stability.csv, '
      'diag_w2v_pc_alignment_by_fold.csv et diag_w2v_pc_top_words.csv')


# %% ── 6. Route FinBERT plein article par segmentation et agrégation article/jour ──────────────
if ENABLE_FINBERT:
    SKIP_FB = ckpt_exists(
        FINBERT_DAILY_CKPT_NAME, 'csv', FORCE_RERUN_FINBERT)
else:
    SKIP_FB = False
    print('\n[6] ENABLE_FINBERT=False ; FinBERT ignoré.')

if not ENABLE_FINBERT:
    for c in ['finbert_sent','finbert_pos','finbert_neg',
              'finbert_frac_neg','finbert_sent_lag1','finbert_neg_lag1',
              'finbert_frac_neg_lag1','finbert_sent_std','finbert_sent_std_lag1',
              'finbert_sent_fwd','finbert_sent_fwd_lag1','finbert_n_fwd_arts']:
        df_raw[c] = np.nan

elif SKIP_FB:
    print(f'\n[6] FinBERT plein article ← ✅ restauration depuis {FINBERT_DAILY_CKPT_NAME}.csv ; recalcul ignoré')
    fb_daily = load_ckpt(FINBERT_DAILY_CKPT_NAME, 'csv')
    fb_daily['date'] = pd.to_datetime(fb_daily['date'])
    df_raw = df_raw.merge(fb_daily, on='date', how='left')
    for c in [cc for cc in fb_daily.columns
              if cc not in ('date', 'finbert_sent_fwd',
                            'finbert_sent_fwd_lag1', 'finbert_n_fwd_arts')]:
        df_raw[c] = df_raw[c].ffill()
    if 'finbert_sent_fwd' in df_raw.columns:
        df_raw['finbert_sent_fwd'] = df_raw['finbert_sent_fwd'].fillna(0)
        df_raw['finbert_sent_fwd_lag1'] = df_raw['finbert_sent_fwd_lag1'].fillna(0)
        df_raw['finbert_n_fwd_arts'] = df_raw['finbert_n_fwd_arts'].fillna(0)
    print(f'  Variables journalières FinBERT restaurées : {fb_daily.shape}')

else:
    print('\n[6] Inférence FinBERT sur l’intégralité des articles, par segments (calcul en cours...)')
    t0 = time.time()
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    device = DEVICE
    print(f'  Périphérique utilisé : {device}')
    if device.type == 'cpu':
        print('  ⚠️  Inférence exécutée sur CPU ; elle sera nettement plus lente. '
              'Il est recommandé d’utiliser un environnement T4 GPU.')

    # Un tokenizer rapide est requis afin d’obtenir overflow_to_sample_mapping,
    # qui rattache chaque segment de 512 tokens à son article d’origine.
    tokenizer_fb = AutoTokenizer.from_pretrained(
        FINBERT_MODEL, use_fast=True)
    if not tokenizer_fb.is_fast:
        raise RuntimeError(
            'Le tokenizer FinBERT rapide est requis pour segmenter les articles complets.')

    model_fb = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
    model_fb.eval().to(device)

    # Détection explicite de l’ordre des classes au lieu de supposer silencieusement
    # que les colonnes 0 et 1 correspondent toujours à positive et negative.
    _id2label = {
        int(i): str(label).lower()
        for i, label in model_fb.config.id2label.items()
    }
    _label2id = {label: i for i, label in _id2label.items()}
    try:
        FINBERT_POS_INDEX = _label2id['positive']
        FINBERT_NEG_INDEX = _label2id['negative']
    except KeyError as exc:
        raise RuntimeError(
            f'Classes FinBERT inattendues : {_id2label}. '
            'Les classes positive et negative sont requises.'
        ) from exc
    print(f'  Ordre des classes FinBERT détecté : {_id2label}')

    # Attribution simultanée de l’indicateur prospectif à chaque article afin
    # d’agréger séparément le sentiment du sous-échantillon prospectif.
    art_records = []
    for row_idx, row in df_raw.iterrows():
        for article in row['_canonical_articles']:
            txt = article['text']
            fwd_flag = _canonical_article_is_fwd(article)
            art_records.append((row_idx, row['date'], txt, fwd_flag))

    print(f'  Nombre total d’articles : {len(art_records)} '
          f'(dont {sum(r[3] for r in art_records)} classés comme prospectifs)')
    art_dates  = [r[1] for r in art_records]
    art_texts  = [r[2] for r in art_records]
    art_is_fwd = [r[3] for r in art_records]

    if len(art_texts) == 0:
        raise RuntimeError('Aucun article valide n’est disponible pour FinBERT.')

    BATCH_CKPT = ckpt_path(FINBERT_PARTIAL_CKPT_NAME, 'pkl')
    _partial_signature = {
        'cache_version': FINBERT_CACHE_VERSION,
        'model': FINBERT_MODEL,
        'max_length': FINBERT_MAX_LENGTH,
        'stride': FINBERT_STRIDE,
        'aggregation': FINBERT_AGGREGATION,
        'article_input_fingerprint': ARTICLE_INPUT_FINGERPRINT,
        'n_articles': len(art_texts),
    }

    start_article = 0
    completed_article_prob_blocks = []
    completed_article_chunk_blocks = []

    if (os.path.exists(BATCH_CKPT)
            and not FORCE_RERUN_TEXT_MODELS
            and not FORCE_RERUN_FINBERT):
        with open(BATCH_CKPT, 'rb') as f:
            partial = pickle.load(f)

        if partial.get('signature') == _partial_signature:
            start_article = int(partial['next_article'])
            completed_article_prob_blocks = partial['article_prob_blocks']
            completed_article_chunk_blocks = partial.get(
                'article_chunk_blocks', [])
            print(f'  ↺ Reprise après l’article {start_article}/{len(art_texts)}')
        else:
            print('  ⚠️  Checkpoint FinBERT partiel incompatible ; reprise ignorée et recalcul depuis le début.')

    # Nombre de tokens spéciaux ajouté à chaque segment ([CLS] et [SEP] pour BERT).
    n_special_tokens = tokenizer_fb.num_special_tokens_to_add(pair=False)
    total_chunks_processed = int(sum(
        np.asarray(block, dtype=int).sum()
        for block in completed_article_chunk_blocks
    )) if completed_article_chunk_blocks else 0

    for group_start in range(
            start_article, len(art_texts), FINBERT_ARTICLE_GROUP_SIZE):
        group_end = min(
            group_start + FINBERT_ARTICLE_GROUP_SIZE, len(art_texts))
        group_texts = art_texts[group_start:group_end]
        n_group_articles = len(group_texts)

        # return_overflowing_tokens=True crée autant de segments que nécessaire
        # pour couvrir l’intégralité de chaque article. Aucun [:2000] n’est appliqué.
        encoded = tokenizer_fb(
            group_texts,
            add_special_tokens=True,
            truncation=True,
            max_length=FINBERT_MAX_LENGTH,
            stride=FINBERT_STRIDE,
            return_overflowing_tokens=True,
            return_attention_mask=True,
            padding='max_length',
            return_tensors='pt',
        )

        overflow_mapping = encoded.pop(
            'overflow_to_sample_mapping').cpu().numpy().astype(int)
        attention_masks = encoded['attention_mask']
        n_group_chunks = int(len(overflow_mapping))

        if n_group_chunks == 0:
            raise RuntimeError(
                f'Aucun segment produit pour les articles {group_start}:{group_end}.')

        group_chunk_probs = []
        for chunk_start in range(
                0, n_group_chunks, FINBERT_INFERENCE_BATCH_SIZE):
            chunk_end = min(
                chunk_start + FINBERT_INFERENCE_BATCH_SIZE, n_group_chunks)
            model_batch = {
                key: value[chunk_start:chunk_end].to(device)
                for key, value in encoded.items()
            }
            with torch.no_grad():
                logits = model_fb(**model_batch).logits
                probs = torch.softmax(logits, dim=-1).cpu().numpy()
            group_chunk_probs.append(probs.astype(np.float32))

        group_chunk_probs = np.vstack(group_chunk_probs)

        # Le dernier segment peut être beaucoup plus court que les autres. Une
        # moyenne pondérée par le nombre de tokens de contenu empêche ce petit
        # segment d’avoir le même poids qu’un segment presque complet.
        content_token_counts = (
            attention_masks.sum(dim=1).cpu().numpy().astype(float)
            - float(n_special_tokens)
        )
        content_token_counts = np.maximum(content_token_counts, 1.0)

        group_probability_sums = np.zeros(
            (n_group_articles, group_chunk_probs.shape[1]), dtype=np.float64)
        group_token_sums = np.zeros(n_group_articles, dtype=np.float64)
        group_chunk_counts = np.zeros(n_group_articles, dtype=np.int32)

        np.add.at(
            group_probability_sums,
            overflow_mapping,
            group_chunk_probs * content_token_counts[:, None],
        )
        np.add.at(group_token_sums, overflow_mapping, content_token_counts)
        np.add.at(group_chunk_counts, overflow_mapping, 1)

        if np.any(group_token_sums <= 0):
            raise RuntimeError(
                f'Un article du groupe {group_start}:{group_end} ne possède aucun token valide.')

        group_article_probs = (
            group_probability_sums / group_token_sums[:, None]
        ).astype(np.float32)

        completed_article_prob_blocks.append(group_article_probs)
        completed_article_chunk_blocks.append(group_chunk_counts)
        total_chunks_processed += n_group_chunks

        completed_articles = group_end
        elapsed = time.time() - t0
        pct = completed_articles / len(art_texts) * 100
        eta = (elapsed / max(completed_articles - start_article, 1)
               * (len(art_texts) - completed_articles))
        print(
            f'  articles {completed_articles}/{len(art_texts)} ({pct:.1f} %)  '
            f'segments cumulés={total_chunks_processed}  '
            f'temps écoulé={elapsed:.0f}s  temps restant estimé={eta:.0f}s'
        )

        # Le checkpoint est enregistré uniquement après un groupe d’articles
        # entièrement agrégé. Une reprise ne peut donc pas dupliquer un segment.
        with open(BATCH_CKPT, 'wb') as f:
            pickle.dump({
                'signature': _partial_signature,
                'next_article': completed_articles,
                'article_prob_blocks': completed_article_prob_blocks,
                'article_chunk_blocks': completed_article_chunk_blocks,
            }, f, protocol=4)

    art_probs_all = np.vstack(completed_article_prob_blocks)
    article_chunk_counts_all = np.concatenate(
        completed_article_chunk_blocks).astype(int)

    if len(art_probs_all) != len(art_texts):
        raise RuntimeError(
            f'Nombre de probabilités article incorrect : {len(art_probs_all)} '
            f'au lieu de {len(art_texts)}.')

    art_df = pd.DataFrame({
        'date': art_dates,
        'is_fwd': art_is_fwd,
        'n_finbert_chunks': article_chunk_counts_all,
        'p_pos': art_probs_all[:, FINBERT_POS_INDEX],
        'p_neg': art_probs_all[:, FINBERT_NEG_INDEX],
    })
    art_df['net_sent'] = art_df['p_pos'] - art_df['p_neg']
    art_df['is_neg'] = (art_df['p_neg'] > 0.5).astype(float)
    art_df['date'] = pd.to_datetime(art_df['date'])

    fb_daily_all = art_df.groupby('date').agg(
        finbert_sent       = ('net_sent',          'mean'),
        finbert_sent_med   = ('net_sent',          'median'),
        finbert_sent_std   = ('net_sent',          'std'),
        finbert_pos        = ('p_pos',              'mean'),
        finbert_neg        = ('p_neg',              'mean'),
        finbert_frac_neg   = ('is_neg',             'mean'),
        finbert_n_arts     = ('p_pos',              'count'),
        finbert_n_chunks   = ('n_finbert_chunks',   'sum'),
        finbert_mean_chunks_per_article = ('n_finbert_chunks', 'mean'),
    ).reset_index()

    # Réagrégation du sentiment uniquement sur les articles classés comme prospectifs.
    fb_daily_fwd = (art_df[art_df['is_fwd']].groupby('date')
                    .agg(finbert_sent_fwd=('net_sent', 'mean'),
                         finbert_n_fwd_arts=('net_sent', 'count'))
                    .reset_index())

    fb_daily = fb_daily_all.merge(fb_daily_fwd, on='date', how='left')
    fb_daily = fb_daily.sort_values('date').reset_index(drop=True)

    # Zéro signifie ici « aucun article prospectif », et non une valeur manquante.
    fb_daily['finbert_sent_fwd'] = fb_daily['finbert_sent_fwd'].fillna(0)
    fb_daily['finbert_n_fwd_arts'] = fb_daily['finbert_n_fwd_arts'].fillna(0)

    for col in ['finbert_sent','finbert_sent_med','finbert_neg',
                'finbert_frac_neg','finbert_sent_std','finbert_sent_fwd']:
        fb_daily[f'{col}_lag1'] = fb_daily[col].shift(1)
    fb_daily['finbert_sent_fwd_lag1'] = (
        fb_daily['finbert_sent_fwd_lag1'].fillna(0))

    save_ckpt(FINBERT_DAILY_CKPT_NAME, fb_daily, 'csv')
    if os.path.exists(BATCH_CKPT):
        os.remove(BATCH_CKPT)

    # Diagnostic article par article : utile pour vérifier la distribution du
    # nombre de segments et repérer les textes exceptionnellement longs.
    art_df[['date', 'is_fwd', 'n_finbert_chunks', 'p_pos', 'p_neg',
            'net_sent', 'is_neg']].to_csv(
        os.path.join(OUT_DIR, 'diag_finbert_article_fulltext.csv'), index=False)

    df_raw = df_raw.merge(fb_daily, on='date', how='left')
    for c in ['finbert_sent_lag1','finbert_neg_lag1',
              'finbert_frac_neg_lag1','finbert_sent_std_lag1',
              'finbert_sent','finbert_neg','finbert_frac_neg',
              'finbert_sent_std']:
        if c in df_raw.columns:
            df_raw[c] = df_raw[c].ffill()

    df_raw['finbert_sent_fwd'] = df_raw['finbert_sent_fwd'].fillna(0)
    df_raw['finbert_sent_fwd_lag1'] = (
        df_raw['finbert_sent_fwd_lag1'].fillna(0))
    df_raw['finbert_n_fwd_arts'] = df_raw['finbert_n_fwd_arts'].fillna(0)

    print(f'  → FinBERT plein article terminé en {time.time()-t0:.1f}s ; '
          f'{len(art_texts)} articles et {int(article_chunk_counts_all.sum())} segments traités.')

# %% ── 6.5 [OPTIMISATION MEMOIRE] Suppression du texte brut par article devenu inutile ──────────
# À ce stade, LM (section 3), TF-IDF (section 4), Word2Vec (section 5) et FinBERT (section 6)
# ont tous consommé les colonnes de texte brut / la collection canonique et ont déjà produit
# leurs variables numériques dans df_raw (lm_*, tfidf_idx, w2v_pc*, finbert_*). Les colonnes
# ci-dessous ne sont plus lues par aucune section suivante et représentent la majeure partie
# de la mémoire occupée par df_raw (jusqu'à 24 colonnes de texte intégral par jour, plus la
# collection canonique qui duplique ce même texte). daily_text_stemmed est conservé car la
# section 7.5 (diagnostic de classification, désactivée par défaut) peut encore s'en servir.
print('\n[6.5] Nettoyage mémoire : suppression du texte brut par article devenu inutile ...')
_mem_before_mb = df_raw.memory_usage(deep=True).sum() / 1e6
_cols_to_drop_after_text_routes = [
    c for c in (TEXT_COLS_RAW + TITLE_COLS_RAW + ['_canonical_articles', 'daily_text'])
    if c in df_raw.columns
]
df_raw.drop(columns=_cols_to_drop_after_text_routes, inplace=True, errors='ignore')
gc.collect()
_mem_after_mb = df_raw.memory_usage(deep=True).sum() / 1e6
print(f'  df_raw : {_mem_before_mb:.1f} MB → {_mem_after_mb:.1f} MB '
      f'({len(_cols_to_drop_after_text_routes)} colonnes supprimées, dont le texte brut de {len(TEXT_COLS_RAW)} '
      f'colonnes d’articles et la collection canonique)')

# %% ── 6.6 [OPTIMISATION MEMOIRE] Libération des tableaux intermédiaires FinBERT ──────────
# Ces objets n'existent que dans la branche de calcul effectif (ni ENABLE_FINBERT=False, ni SKIP_FB) ;
# la vérification "in globals()" les couvre donc de façon uniforme quelle que soit la branche prise.
print('[6.6] Nettoyage mémoire : libération des tableaux intermédiaires FinBERT ...')
for _name in ['art_df', 'art_probs_all', 'art_records', 'art_dates', 'art_texts', 'art_is_fwd',
              'completed_article_prob_blocks', 'completed_article_chunk_blocks',
              'article_chunk_counts_all', 'fb_daily_all', 'fb_daily_fwd', '_token_corpus_full',
              'doc_vecs_full', 'doc_vecs_train_i', 'token_corpus_train_i', 'token_corpus_pred']:
    if _name in globals():
        del globals()[_name]
gc.collect()
print('  → Objets intermédiaires FinBERT/Word2Vec libérés de la mémoire')


# %% ── 7. Construction des variables HAR + spillover transversal + rendements positifs/négatifs ─────────────────
print('\n[7] Construction des cibles HAR multi-horizons, du spillover transversal et des rendements positifs/négatifs ...')

for ticker in TICKERS:
    if f'rv_{ticker}' not in df_raw.columns:
        print(f'  ⚠️ {ticker} ignoré : la colonne rv_{ticker} est absente des données')
        continue
    rv = df_raw[f'rv_{ticker}']
    df_raw[f'rv_d_{ticker}'] = rv
    df_raw[f'rv_w_{ticker}'] = rv.rolling(5).mean()
    df_raw[f'rv_m_{ticker}'] = rv.rolling(22).mean()
    for h in H_LIST:
        if h == 1:
            df_raw[f'rv_fwd_h{h}_{ticker}'] = rv.shift(-1)
        else:
            df_raw[f'rv_fwd_h{h}_{ticker}'] = (
                rv.shift(-1).rolling(h).mean().shift(-(h-1))
            )
    df_raw[f'log_rv_{ticker}'] = np.log(rv.clip(lower=1e-8))

    if f'ret_{ticker}' in df_raw.columns:
        ret_t = df_raw[f'ret_{ticker}']
        df_raw[f'good_{ticker}'] = ret_t.clip(lower=0)
        df_raw[f'bad_{ticker}']  = ret_t.clip(upper=0)

_valid_tickers_for_spillover = [t for t in TICKERS if f'rv_d_{t}' in df_raw.columns]
for ticker in _valid_tickers_for_spillover:
    other_cols = [f'rv_d_{t}' for t in _valid_tickers_for_spillover if t != ticker]
    spillover_today = df_raw[other_cols].mean(axis=1, skipna=True)
    df_raw[f'rv_spillover_{ticker}'] = spillover_today.shift(1)

print('  → Cibles HAR, spillover et rendements positifs/négatifs terminés')

# %% ── 7.5 Diagnostic de classification hiérarchique (exploratoire, désactivé par défaut) ─────────────────
if ENABLE_CLUSTER_DIAG:
    print('\n[7.5] Diagnostic de classification hiérarchique : part du contenu « lié au marché » dans le corpus (calcul en cours...)')
    t0 = time.time()

    tfidf_cluster = TfidfVectorizer(max_features=2000, stop_words='english',
                                    min_df=5, max_df=0.9)
    X_cluster = tfidf_cluster.fit_transform(df_raw.loc[valid_text_mask, TEXT_COL_FOR_TFIDF].values)
    feat_names_cluster = tfidf_cluster.get_feature_names_out()

    N_CLUSTERS_DIAG = 10
    n_docs_cluster = X_cluster.shape[0]
    sample_size = min(2000, n_docs_cluster)
    np.random.seed(2026)
    sample_idx = np.random.choice(n_docs_cluster, size=sample_size, replace=False)
    X_sample_dense = X_cluster[sample_idx].toarray()

    hc = AgglomerativeClustering(n_clusters=N_CLUSTERS_DIAG, metric='euclidean', linkage='ward')
    clusters_diag = hc.fit_predict(X_sample_dense)

    cluster_series_diag = pd.Series(clusters_diag)
    print(f'  Taille de l’échantillon : {sample_size} (tiré parmi {n_docs_cluster} textes journaliers concaténés)')
    print('  Distribution du nombre de documents par cluster :')
    print(cluster_series_diag.value_counts().sort_index().to_string())

    X_sample_df = pd.DataFrame(X_sample_dense, columns=feat_names_cluster)
    X_sample_df['cluster'] = clusters_diag
    print('\n  Termes représentatifs de chaque cluster (Top 8) :')
    cluster_topwords_summary = []
    for c_i in range(N_CLUSTERS_DIAG):
        cluster_data = X_sample_df[X_sample_df['cluster'] == c_i].drop('cluster', axis=1)
        if len(cluster_data) == 0:
            continue
        top_words = cluster_data.mean().sort_values(ascending=False).head(8).index.tolist()
        n_docs_in_cluster = (cluster_series_diag == c_i).sum()
        print(f'    Cluster {c_i} (n={n_docs_in_cluster}): {", ".join(top_words)}')
        cluster_topwords_summary.append({
            'cluster': c_i, 'n_docs': int(n_docs_in_cluster),
            'top_words': ', '.join(top_words),
        })
    pd.DataFrame(cluster_topwords_summary).to_csv(
        os.path.join(OUT_DIR, 'diag_cluster_topwords.csv'), index=False)
    print(f'  → Diagnostic de classification terminé en {time.time()-t0:.1f}s.')
else:
    print('\n[7.5] ENABLE_CLUSTER_DIAG=False ; diagnostic de classification ignoré.')

# %% ── 8. Fonctions auxiliaires (QLIKE + VIF + orthogonalisation + extraction des coefficients v8 / trajectoires rolling) ─────────────

def _har_hac_lags(h):
    # Régressions IS sur une cible future chevauchante : bande h-1, avec au moins un lag.
    return max(h - 1, 1)


def _har_oos_hac_lags(h):
    # HAR produit une prévision chaque jour. Les cibles futures se chevauchent pour h>1.
    return max(int(h) - 1, 0)


def _garch_oos_hac_lags(h):
    # GARCH conserve une seule origine tous les 22 jours : aucune cible OOS ne se chevauche.
    return int(GARCH_CW_DM_HAC_LAGS)


def _oos_hac_lags(h, model_type='HAR'):
    model_type = str(model_type).upper()
    if model_type.startswith('GARCH'):
        return _garch_oos_hac_lags(h)
    return _har_oos_hac_lags(h)


def run_ols_is(dm, ycol, xcols, h):
    sub = dm[[ycol]+xcols].dropna()
    return sm.OLS(sub[ycol], sm.add_constant(sub[xcols])).fit(
        cov_type='HAC', cov_kwds={'maxlags': _har_hac_lags(h)}
    )


def qlike_loss(actual_vol, pred_vol, eps=1e-12):
    """QLIKE for volatility forecasts, evaluated on the implied variance scale.

    Negative or zero volatility forecasts are invalid and are clipped before squaring,
    rather than being made positive by squaring a negative number.
    """
    min_vol = np.sqrt(eps)
    actual_vol = np.clip(np.asarray(actual_vol, dtype=float), min_vol, None)
    pred_vol   = np.clip(np.asarray(pred_vol, dtype=float), min_vol, None)
    actual_var = np.square(actual_vol)
    pred_var   = np.square(pred_vol)
    return float(np.mean(np.log(pred_var) + actual_var / pred_var))


def qlike_loss_vector(actual_vol, pred_vol, eps=1e-12):
    """QLIKE observation par observation, sur l'échelle de variance."""
    min_vol = np.sqrt(eps)
    actual_vol = np.clip(np.asarray(actual_vol, dtype=float), min_vol, None)
    pred_vol   = np.clip(np.asarray(pred_vol, dtype=float), min_vol, None)
    actual_var = np.square(actual_vol)
    pred_var   = np.square(pred_vol)
    return np.log(pred_var) + actual_var / pred_var


def compute_vif(dm, xcols):
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    sub = dm[xcols].dropna()
    if len(sub) < len(xcols) * 5:
        return pd.Series(np.nan, index=xcols)
    X = sm.add_constant(sub)
    vifs = []
    for i in range(X.shape[1]):
        try:
            v = variance_inflation_factor(X.values, i)
        except Exception:
            v = np.nan
        vifs.append(v)
    return pd.Series(vifs, index=X.columns).drop('const', errors='ignore')


def orthogonalize_text_feature(dm, text_col, control_cols):
    sub = dm[[text_col] + control_cols].dropna()
    if len(sub) < len(control_cols) * 5 + 10:
        return None
    aux_res = sm.OLS(sub[text_col], sm.add_constant(sub[control_cols])).fit()
    resid = aux_res.resid
    return resid.reindex(dm.index)


def oos_r2_expanding(dm, ycol, xcols, h=1,
                     min_train=MIN_TRAIN_OOS, min_train_date=None,
                     step=None, prediction_block=None,
                     return_series=False, window_type='expanding',
                     trailing_window=ROLLING_WINDOW):
    """
    Évaluation OOS walk-forward des modèles linéaires HAR.

    Conception principale v17B : les coefficients sont réestimés tous les
    HAR_REFIT_STEP=22 jours. À chaque réestimation, le modèle produit ensuite
    une prévision pour chacune des observations du bloc journalier suivant
    (HAR_PREDICTION_BLOCK=22). Les variables explicatives restent donc datées
    de chaque jour de prévision, tandis que les coefficients sont maintenus
    fixes à l'intérieur du bloc.

    Pour éviter toute fuite d'information, les étiquettes d'entraînement sont
    purgées de h-1 observations. Pour h=5 et h=22, les cibles OOS journalières
    sont chevauchantes ; les tests CW/DM utilisent donc une bande HAC h-1.
    """
    if window_type not in {'expanding', 'rolling'}:
        raise ValueError("window_type must be 'expanding' or 'rolling'")
    if int(h) < 1:
        raise ValueError('h must be a positive integer')
    if step is None:
        step = HAR_REFIT_STEP
    if prediction_block is None:
        prediction_block = HAR_PREDICTION_BLOCK
    step = int(step)
    prediction_block = int(prediction_block)
    if step < 1 or prediction_block < 1:
        raise ValueError('step and prediction_block must be positive integers')

    valid_idx = dm[[ycol] + xcols].dropna().index
    sub = dm.loc[valid_idx, [ycol] + xcols].reset_index(drop=True)
    n = len(sub)
    has_date = 'date' in dm.columns
    if has_date:
        sub_dates = pd.to_datetime(dm.loc[valid_idx, 'date']).reset_index(drop=True)
        if min_train_date is not None:
            min_train = int(np.searchsorted(sub_dates.values,
                                             np.datetime64(min_train_date)))
            min_train = max(min_train, min(MIN_TRAIN_OOS, max(n // 2, 1)))

    if n < min_train + 1:
        if return_series:
            return np.nan, 0, np.array([]), np.array([]), np.array([])
        return np.nan, 0

    purge = max(int(h) - 1, 0)
    pred_list, act_list, bench_list, date_list = [], [], [], []

    for t in range(min_train, n, step):
        train_end = t - purge
        if train_end <= 0:
            continue

        if window_type == 'rolling':
            train_start = max(0, train_end - trailing_window)
        else:
            train_start = 0

        train = sub.iloc[train_start:train_end]
        test_end = min(t + prediction_block, n)
        test = sub.iloc[t:test_end]
        if len(test) == 0 or len(train) == 0:
            continue

        X_tr = np.column_stack([np.ones(len(train)), train[xcols].values])
        X_te = np.column_stack([np.ones(len(test)), test[xcols].values])
        try:
            res = sm.OLS(train[ycol].values, X_tr).fit()
            pred = X_te @ res.params
        except Exception:
            continue

        pred_list.extend(np.asarray(pred, dtype=float).tolist())
        act_list.extend(test[ycol].astype(float).values.tolist())
        bench_list.extend([float(train[ycol].mean())] * len(test))
        if has_date:
            date_list.extend(sub_dates.iloc[t:test_end].tolist())

    if len(pred_list) == 0:
        if return_series:
            return np.nan, 0, np.array([]), np.array([]), np.array([])
        return np.nan, 0

    pred_arr = np.asarray(pred_list, dtype=float)
    act_arr = np.asarray(act_list, dtype=float)
    bench_arr = np.asarray(bench_list, dtype=float)
    sse_m = np.sum((act_arr - pred_arr) ** 2)
    sse_b = np.sum((act_arr - bench_arr) ** 2)
    oos_r2 = 1 - sse_m / max(sse_b, 1e-16)

    if return_series:
        dates_out = np.array(date_list) if date_list else np.arange(len(act_arr))
        return oos_r2, len(pred_list), dates_out, act_arr, pred_arr
    return oos_r2, len(pred_list)

def extract_text_coef(res, xcols_text):
    for col in xcols_text:
        if col in res.params.index:
            return res.params[col], res.pvalues[col]
    return np.nan, np.nan

def fmt_stars(pval):
    if pd.isna(pval): return ''
    if pval < 0.01:   return '***'
    if pval < 0.05:   return '**'
    if pval < 0.10:   return '*'
    return ''


# Convertit tous les coefficients d’une régression en format long, avec statistiques de modèle.
def extract_all_coefs(res, ticker, h, spec, model_name, route=None, window_type='full_sample'):
    """Exporte coefficient, erreur-type, statistique t, p-value, IC à 95 %, N, R² et R² ajusté."""
    if res is None:
        return []
    try:
        ci = res.conf_int(alpha=0.05)
    except Exception:
        ci = None
    recs = []
    for term in res.params.index:
        recs.append({
            'ticker': ticker, 'h': h, 'spec': spec, 'model': model_name,
            'route': route, 'window_type': window_type, 'term': term,
            'coef': float(res.params[term]),
            'se': float(res.bse[term]),
            'tstat': float(res.tvalues[term]),
            'pval': float(res.pvalues[term]),
            'ci_low': float(ci.loc[term].iloc[0]) if ci is not None and term in ci.index else np.nan,
            'ci_high': float(ci.loc[term].iloc[1]) if ci is not None and term in ci.index else np.nan,
            'n_obs': int(getattr(res, 'nobs', 0)),
            'R2': float(getattr(res, 'rsquared', np.nan)),
            'Adj_R2': float(getattr(res, 'rsquared_adj', np.nan)),
            'cov_type': str(getattr(res, 'cov_type', 'classical')),
        })
    return recs


# Paramètres natifs GARCH(1,1)-skewt (omega/alpha[1]/beta[1]/eta/lambda)
def extract_garch_coefs(res_g, ticker, h):
    """Exporte les paramètres natifs GARCH avec erreurs-types et statistiques usuelles."""
    if res_g is None:
        return []
    try:
        ci = res_g.conf_int()
    except Exception:
        ci = None
    recs = []
    for term in res_g.params.index:
        recs.append({
            'ticker': ticker, 'h': h, 'spec': 'Native', 'model': 'GARCH-native',
            'route': 'Benchmark', 'window_type': 'full_sample', 'term': term,
            'coef': float(res_g.params[term]), 'se': float(res_g.std_err[term]),
            'tstat': float(res_g.tvalues[term]), 'pval': float(res_g.pvalues[term]),
            'ci_low': float(ci.loc[term].iloc[0]) if ci is not None and term in ci.index else np.nan,
            'ci_high': float(ci.loc[term].iloc[1]) if ci is not None and term in ci.index else np.nan,
            'n_obs': int(getattr(res_g, 'nobs', 0)),
            'R2': np.nan, 'Adj_R2': np.nan,
            'AIC': float(getattr(res_g, 'aic', np.nan)),
            'BIC': float(getattr(res_g, 'bic', np.nan)),
            'cov_type': 'GARCH MLE',
        })
    return recs


# Trajectoires des coefficients disponibles en temps réel, en expanding ou rolling.
def rolling_coef_trend(dm, ycol, xcols, h=1,
                       min_train=None, step=None, window_type='rolling',
                       trailing_window=ROLLING_WINDOW, hac_lags=1):
    """
    Réestime le modèle à chaque origine de prévision, avec purge de (h-1).
    window_type='expanding' conserve tout l'historique disponible ;
    window_type='rolling' conserve les trailing_window dernières étiquettes réalisées.
    """
    valid_idx = dm[[ycol] + xcols].dropna().index
    sub = dm.loc[valid_idx, [ycol] + xcols].reset_index(drop=True)
    has_date = 'date' in dm.columns
    if has_date:
        sub_dates = pd.to_datetime(dm.loc[valid_idx, 'date']).reset_index(drop=True)
    n = len(sub)
    if min_train is None:
        min_train = min(MIN_TRAIN_OOS, max(n // 3, 30))
    if step is None:
        step = HAR_REFIT_STEP

    purge = max(int(h) - 1, 0)
    records = []
    for t in range(min_train, n, step):
        train_end = t - purge
        if train_end <= 0:
            continue
        if window_type == 'expanding':
            train_start = 0
        elif window_type == 'rolling':
            train_start = max(0, train_end - trailing_window)
        else:
            raise ValueError("window_type must be 'expanding' or 'rolling'")
        train = sub.iloc[train_start:train_end]
        if len(train) < len(xcols) * 5:
            continue
        X_tr = sm.add_constant(train[xcols])
        try:
            res = sm.OLS(train[ycol], X_tr).fit(
                cov_type='HAC', cov_kwds={'maxlags': hac_lags})
        except Exception:
            continue
        refit_date = sub_dates.iloc[t] if has_date else t
        for term in res.params.index:
            records.append({
                'refit_date': refit_date, 'term': term,
                'coef': res.params[term], 'se': res.bse[term],
                'tstat': res.tvalues[term], 'pval': res.pvalues[term],
                'window_type': window_type,
                'train_start': train_start, 'train_end': train_end,
                'n_train': len(train),
            })
    return pd.DataFrame(records)


def _safe_filename(text):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(text)).strip('_')


def plot_coef_trend(coef_df, term, ticker, h, out_dir, sig_level=0.05,
                    window_type=None, route=None, component=None):
    if coef_df is None or coef_df.empty or 'term' not in coef_df.columns:
        return None
    sub = coef_df[coef_df['term'] == term].sort_values('refit_date').copy()
    if window_type is not None and 'window_type' in sub.columns:
        sub = sub[sub['window_type'] == window_type]
    if sub.empty:
        return None
    wt = window_type or (sub['window_type'].iloc[0] if 'window_type' in sub.columns else 'unknown')
    sig = sub['pval'] < sig_level

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(sub['refit_date'], sub['coef'], color='lightgray', lw=1, zorder=1)
    ax.scatter(sub.loc[~sig, 'refit_date'], sub.loc[~sig, 'coef'],
               color='steelblue', s=22, alpha=0.7, label=f'p≥{sig_level}', zorder=2)
    ax.scatter(sub.loc[sig, 'refit_date'], sub.loc[sig, 'coef'],
               color='crimson', s=32, label=f'p<{sig_level}', zorder=3)
    ax.axhline(0, color='black', lw=0.8, ls='--')
    extra = ' | '.join(str(x) for x in [route, component] if x)
    ax.set_title(f'{ticker}  h={h}  coefficient {wt} : {term}' + (f' | {extra}' if extra else ''))
    ax.set_ylabel('Coefficient estimé')
    ax.legend(loc='best', fontsize=8)
    fig.tight_layout()

    bits = ['coef_trend', wt, ticker, f'h{h}']
    if route: bits.append(route)
    if component: bits.append(component)
    bits.append(term)
    out_path = os.path.join(out_dir, '_'.join(_safe_filename(x) for x in bits) + '.png')
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path


# %% ── 9. Définition des variables par route — alignement temporel uniforme ──────────
GKG_ROUTE_FEATS = {
    'GKG': ['gkg_sentiment', 'gkg_negative_tone'],
}
GKG_ROUTE_PRIMARY = {
    'GKG': 'gkg_sentiment',
}


def _route_aligned_feature(base_col, zero_semantic=False):
    """
    Renvoie la colonne utilisée par les régressions selon TEXT_TIMING_MODE.
    En mode lag1_all, le décalage est appliqué uniformément à toutes les routes.
    """
    if base_col not in df_raw.columns:
        return base_col
    if TEXT_TIMING_MODE == 'same_day':
        return base_col

    aligned_col = f'{base_col}__lag1_all'
    if aligned_col not in df_raw.columns:
        df_raw[aligned_col] = df_raw[base_col].shift(1)
        if zero_semantic:
            df_raw[aligned_col] = df_raw[aligned_col].fillna(0)
    return aligned_col


_BASE_TEXT_ROUTE_FEATS = {
    'LM':          ['lm_polarity', 'lm_neg_prop', 'lm_unc_prop', 'lm_polarity_std'],
    'TFIDF_LASSO': ['tfidf_idx'],
    'W2V_PCA':     W2V_PC_COLS,
    'FINBERT':     ['finbert_sent', 'finbert_neg', 'finbert_frac_neg', 'finbert_sent_std'],
}

TEXT_ROUTE_FEATS = {
    route: [_route_aligned_feature(f) for f in feats]
    for route, feats in _BASE_TEXT_ROUTE_FEATS.items()
}

TEXT_ROUTE_PRIMARY = {
    'LM':          TEXT_ROUTE_FEATS['LM'][0],
    'TFIDF_LASSO': TEXT_ROUTE_FEATS['TFIDF_LASSO'][0],
    'W2V_PCA':     TEXT_ROUTE_FEATS['W2V_PCA'][0],
    'FINBERT':     TEXT_ROUTE_FEATS['FINBERT'][0],
}

# Interaction FINBERT × présence de contenu prospectif avec le même alignement temporel.
_has_fwd_route = _route_aligned_feature('has_fwd', zero_semantic=True)
_finbert_sent_route = TEXT_ROUTE_PRIMARY['FINBERT']
_fb_interaction_col = f'finbert_sent_x_hasfwd__{TEXT_TIMING_MODE}'
df_raw[_fb_interaction_col] = (
    pd.to_numeric(df_raw[_finbert_sent_route], errors='coerce')
    * pd.to_numeric(df_raw[_has_fwd_route], errors='coerce')
)
TEXT_ROUTE_FEATS['FINBERT'] = TEXT_ROUTE_FEATS['FINBERT'] + [_fb_interaction_col]

# Route prospective : les trois variables sont désormais toutes contemporaines,
# ou toutes retardées d’un jour en mode lag1_all.
_fwd_lm_route = _route_aligned_feature('fwd_lm_polarity', zero_semantic=True)
_fwd_fb_route = _route_aligned_feature('finbert_sent_fwd', zero_semantic=True)
TEXT_ROUTE_FEATS['FWD_SENT'] = [
    _fwd_lm_route, _fwd_fb_route, _has_fwd_route]
TEXT_ROUTE_PRIMARY['FWD_SENT'] = _fwd_lm_route

# GARCH-X utilise la même origine temporelle ET le même ensemble complet de variables que HAR-X.
TEXT_ROUTE_FEATS_FOR_GARCH = {
    route: list(feats) for route, feats in TEXT_ROUTE_FEATS.items()
}

print('✅ Définition des routes terminée')
print(f'   TEXT_TIMING_MODE={TEXT_TIMING_MODE}')
print(f'   Routes GKG exploratoires : {list(GKG_ROUTE_FEATS.keys())}')
print(f'   Routes textuelles principales : {list(TEXT_ROUTE_FEATS.keys())}')
print(f'   FINBERT : {TEXT_ROUTE_FEATS["FINBERT"]}')
print(f'   FWD_SENT : {TEXT_ROUTE_FEATS["FWD_SENT"]}')
print(f'   HAR : réestimation tous les {HAR_REFIT_STEP} jours, prévisions quotidiennes par blocs de {HAR_PREDICTION_BLOCK} jours')
print(f'   GARCH : conception A, pas OOS fixe = {GARCH_OOS_STEP} jours pour tous les horizons')
print(f'   Checkpoints HAR : {HAR_MODEL_CKPT_DIR}')
print(f'   Checkpoints GARCH : {GARCH_MODEL_CKPT_DIR}')

_run_manifest = {
    'run_tag': RUN_TAG,
    'text_timing_mode': TEXT_TIMING_MODE,
    'text_route_features': TEXT_ROUTE_FEATS,
    'text_route_primary': TEXT_ROUTE_PRIMARY,
    'garch_text_route_features': TEXT_ROUTE_FEATS_FOR_GARCH,
    'har_oos_sampling_mode': HAR_OOS_SAMPLING_MODE,
    'har_refit_step_all_h': HAR_REFIT_STEP,
    'har_prediction_block': HAR_PREDICTION_BLOCK,
    'garch_oos_sampling_mode': GARCH_OOS_SAMPLING_MODE,
    'garch_oos_step_all_h': GARCH_OOS_STEP,
    'model_checkpoint_root': MODEL_CKPT_ROOT,
    'har_cw_dm_hac_lags_by_h': {str(h): _har_oos_hac_lags(h) for h in H_LIST},
    'garch_cw_dm_hac_lags': GARCH_CW_DM_HAC_LAGS,
    'text_input_fingerprint': TEXT_INPUT_FINGERPRINT,
    'article_input_fingerprint': ARTICLE_INPUT_FINGERPRINT,
    'force_rerun': {
        'all_text_models': FORCE_RERUN_TEXT_MODELS,
        'tfidf': FORCE_RERUN_TFIDF,
        'w2v': FORCE_RERUN_W2V,
        'finbert': FORCE_RERUN_FINBERT,
    },
    'w2v_checkpoint': W2V_CKPT_NAME,
    'finbert_checkpoint': FINBERT_DAILY_CKPT_NAME,
    'finbert_model': FINBERT_MODEL,
    'finbert_cache_version': FINBERT_CACHE_VERSION,
    'finbert_max_length': FINBERT_MAX_LENGTH,
    'finbert_stride': FINBERT_STRIDE,
    'finbert_aggregation': FINBERT_AGGREGATION,
}
with open(os.path.join(OUT_DIR, 'run_manifest.json'), 'w', encoding='utf-8') as _f:
    json.dump(_run_manifest, _f, ensure_ascii=False, indent=2, default=str)


def har_base(ticker):
    return [f'rv_d_{ticker}', f'rv_w_{ticker}', f'rv_m_{ticker}']

def macro_controls(ticker):
    cols = ['VIX', 'T10Y2Y']
    spill_col = f'rv_spillover_{ticker}'
    good_col, bad_col = f'good_{ticker}', f'bad_{ticker}'
    if spill_col in df_raw.columns:
        cols.append(spill_col)
    if good_col in df_raw.columns and bad_col in df_raw.columns:
        cols += [good_col, bad_col]
    return cols

_cutoff_probe = df_raw[['date'] + har_base('SPY') + ['rv_fwd_h1_SPY']].dropna().reset_index(drop=True)
OOS_START_DATE = _cutoff_probe['date'].iloc[MIN_TRAIN_OOS]
print(f'✅ Date globale de début OOS (commune à toutes les routes, actifs et valeurs de h) : {OOS_START_DATE.date()}')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive monté
Répertoire de sortie : /content/drive/MyDrive/har_pipeline_final_same_day/outputs
✅ Configuration terminée
   TICKERS (12 actifs) : ['SPY', 'XLE', 'XLF', 'XLK', 'XLY', 'XLP', 'XLI', 'XLV', 'XLB', 'XLU', 'XLRE', 'XLC']
   ENABLE_FINBERT=True | ENABLE_GARCH=True | ENABLE_COLLINEARITY_DIAG=True | ENABLE_PREDICTIVE_CAUSALITY=True
   TEXT_TIMING_MODE=same_day | RUN_TAG=final_same_day
⚠️  Aucun GPU détecté ; l’inférence FinBERT utilisera le CPU et sera nettement plus lente. Il est recommandé de sélectionner un environnement T4 dans Colab.
   (TF-IDF, Word2Vec, GARCH et les diagnostics utilisent principalement le CPU.)
✅ Fonctions d’identification du contenu prospectif prêtes

[2.1] Fusion des articles prospectifs additionnels (institutional_research + fed_policy_communication) ...
  Nombre de colonnes d’articles porté de 19 à 24 (ajout de donné

In [ ]:

"""
Preliminary analysis for the HAR-RV-X / GARCH-X project.

Colab use
----------
1. Run the main pipeline through the construction of HAR variables and text routes
   (sections [7] and [9] in the current script).
2. Paste this whole file into one new Colab cell and run it.
3. Results are written to OUT_DIR/preliminary_analysis without touching model outputs.

Information set
---------------
With TEXT_TIMING_MODE='same_day', variables dated t are related to / used to explain
average realized volatility over t+1,...,t+h. Contemporaneous correlations are exported
separately and must not be interpreted as forecasts or causal effects.
"""

from __future__ import annotations

import json
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import acf


# =============================================================================
# 0. Configuration -- change only this short block if desired
# =============================================================================

PRELIM_ACF_MAX_LAG = 60
PRELIM_ROLLING_CORR_WINDOW = 252       # approximately one trading year
PRELIM_ROLLING_CORR_MIN_OBS = 126
PRELIM_FDR_ALPHA = 0.05
PRELIM_MIN_CORR_OBS = 100
PRELIM_RUN_LOG_HAR_ROBUSTNESS = True  # Corsi-style log-RV robustness
PRELIM_INCLUDE_LEARNED_ROUTES = True  # TF-IDF and W2V are descriptive only here
PRELIM_KEY_TICKER = "SPY"


def _safe_filename(value: object) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def _as_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)


def _future_average(series: pd.Series, h: int) -> pd.Series:
    """Mean of observations t+1,...,t+h, identical to the main pipeline."""
    if int(h) == 1:
        return series.shift(-1)
    return series.shift(-1).rolling(int(h)).mean().shift(-(int(h) - 1))


def _bh_adjust(frame: pd.DataFrame, p_col: str, group_cols: list[str], q_col: str) -> pd.DataFrame:
    """Benjamini-Hochberg correction within explicitly defined exploratory families."""
    out = frame.copy()
    out[q_col] = np.nan
    if out.empty or p_col not in out:
        return out
    grouped = out.groupby(group_cols, dropna=False).groups if group_cols else {"all": out.index}
    for _, idx in grouped.items():
        idx = list(idx)
        valid_idx = [i for i in idx if pd.notna(out.at[i, p_col])]
        if valid_idx:
            pvals = out.loc[valid_idx, p_col].astype(float).clip(0, 1).to_numpy()
            out.loc[valid_idx, q_col] = multipletests(pvals, method="fdr_bh")[1]
    return out


def _save_csv_tex(frame: pd.DataFrame, output_dir: Path, stem: str, index: bool = False) -> None:
    frame.to_csv(output_dir / f"{stem}.csv", index=index)
    try:
        frame.to_latex(
            output_dir / f"{stem}.tex",
            index=index,
            escape=True,
            float_format=lambda x: f"{x:.4f}",
        )
    except Exception as exc:
        print(f"  [note] LaTeX export skipped for {stem}: {exc}")


def _save_figure(fig: plt.Figure, output_dir: Path, stem: str) -> None:
    fig.tight_layout()
    fig.savefig(output_dir / f"{stem}.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


def _ensure_core_columns(work: pd.DataFrame, tickers: list[str], horizons: list[int]) -> None:
    """Construct only missing HAR columns; existing main-pipeline columns are never overwritten."""
    for ticker in tickers:
        raw_col = f"rv_{ticker}"
        if raw_col not in work:
            continue
        rv = _as_numeric(work[raw_col])
        constructions = {
            f"rv_d_{ticker}": rv,
            f"rv_w_{ticker}": rv.rolling(5).mean(),
            f"rv_m_{ticker}": rv.rolling(22).mean(),
        }
        for col, values in constructions.items():
            if col not in work:
                work[col] = values
        for h in horizons:
            col = f"rv_fwd_h{h}_{ticker}"
            if col not in work:
                work[col] = _future_average(rv, h)


def _aligned_fallback_column(
    work: pd.DataFrame,
    base_col: str,
    timing_mode: str,
) -> str | None:
    """Apply lag1 only when the main pipeline has not already supplied an aligned column."""
    if base_col not in work:
        return None
    if timing_mode == "same_day":
        return base_col
    aligned = f"{base_col}__prelim_lag1_all"
    if aligned not in work:
        work[aligned] = _as_numeric(work[base_col]).shift(1)
    return aligned


def _collect_feature_specs(
    work: pd.DataFrame,
    timing_mode: str,
    route_primary: dict[str, str] | None,
    include_learned: bool,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Return two registries:
      - broad_specs: all GKG measures + VIX, for descriptive correlations;
      - screen_specs: parsimonious route-level variables for HAR-conditional screens.
    """
    broad_rows: list[dict] = []
    screen_rows: list[dict] = []

    gkg_raw = [
        "gkg_tone_mean", "gkg_tone_median", "gkg_tone_std",
        "gkg_positive_mean", "gkg_negative_mean", "gkg_polarity_mean",
        "gkg_sentiment", "gkg_negative_tone",
    ]
    for base_col in gkg_raw:
        col = _aligned_fallback_column(work, base_col, timing_mode)
        if col:
            broad_rows.append({"route": "GKG", "feature": base_col, "column": col})

    # Log counts are easier to compare with other continuous variables than highly skewed raw counts.
    for base_col in ["gkg_n_articles", "gkg_n_sources"]:
        col0 = _aligned_fallback_column(work, base_col, timing_mode)
        if col0:
            log_col = f"log1p_{col0}"
            work[log_col] = np.log1p(_as_numeric(work[col0]).clip(lower=0))
            broad_rows.append({"route": "GKG", "feature": f"log1p_{base_col}", "column": log_col})

    vix_col = _aligned_fallback_column(work, "VIX", timing_mode)
    if vix_col:
        broad_rows.append({"route": "BENCHMARK", "feature": "VIX", "column": vix_col})

    # A compact GKG screen avoids treating mechanically related GKG aggregates as separate models.
    for base_col in ["gkg_sentiment", "gkg_negative_tone", "gkg_n_articles"]:
        if base_col == "gkg_n_articles":
            candidates = [x for x in broad_rows if x["feature"] == "log1p_gkg_n_articles"]
            if candidates:
                screen_rows.append(candidates[0].copy())
        else:
            candidates = [x for x in broad_rows if x["feature"] == base_col]
            if candidates:
                screen_rows.append(candidates[0].copy())

    fallback_primary = {
        "LM": "lm_polarity",
        "TFIDF_LASSO": "tfidf_idx",
        "W2V_PCA": "w2v_pc1",
        "FINBERT": "finbert_sent",
        "FWD_SENT": "fwd_lm_polarity",
    }
    source_primary = route_primary if route_primary else fallback_primary
    for route, supplied_col in source_primary.items():
        if not include_learned and route in {"TFIDF_LASSO", "W2V_PCA"}:
            continue
        if supplied_col in work:
            col = supplied_col  # already aligned by section [9] of the main pipeline
            feature = re.sub(r"__(lag1_all|same_day)$", "", supplied_col)
        else:
            base_col = fallback_primary.get(route, supplied_col)
            col = _aligned_fallback_column(work, base_col, timing_mode)
            feature = base_col
        if col:
            screen_rows.append({"route": route, "feature": feature, "column": col})

    registry_cols = ["route", "feature", "column"]
    broad = pd.DataFrame(broad_rows, columns=registry_cols).drop_duplicates(subset=registry_cols)
    screen = pd.DataFrame(screen_rows, columns=registry_cols).drop_duplicates(subset=registry_cols)
    return broad.reset_index(drop=True), screen.reset_index(drop=True)


# =============================================================================
# 1. Descriptive statistics and basic time-series figures
# =============================================================================

def descriptive_analysis(
    work: pd.DataFrame,
    tickers: list[str],
    broad_specs: pd.DataFrame,
    screen_specs: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    selected = [f"ret_{t}" for t in tickers] + [f"rv_{t}" for t in tickers]
    selected += [c for c in ["VIX", "FEDFUNDS", "CPIAUCSL", "T10Y2Y"] if c in work]
    selected += broad_specs["column"].tolist() if not broad_specs.empty else []
    selected += screen_specs["column"].tolist() if not screen_specs.empty else []
    selected = list(dict.fromkeys([c for c in selected if c in work]))

    rows = []
    for col in selected:
        x = _as_numeric(work[col])
        valid = x.dropna()
        rows.append({
            "variable": col,
            "n": int(valid.size),
            "missing_pct": float(100 * x.isna().mean()),
            "mean": valid.mean(), "sd": valid.std(), "median": valid.median(),
            "p01": valid.quantile(0.01), "p05": valid.quantile(0.05),
            "p95": valid.quantile(0.95), "p99": valid.quantile(0.99),
            "skewness": valid.skew(), "excess_kurtosis": valid.kurt(),
        })
    stats = pd.DataFrame(rows)
    _save_csv_tex(stats, output_dir, "table_01_descriptive_statistics")

    rv_cols = [f"rv_{t}" for t in tickers if f"rv_{t}" in work]
    if rv_cols:
        ncols = 3
        nrows = int(np.ceil(len(rv_cols) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.2 * nrows), sharex=True)
        axes = np.asarray(axes).reshape(-1)
        for ax, col in zip(axes, rv_cols):
            ticker = col.split("rv_", 1)[1]
            x = _as_numeric(work[col])
            ax.plot(work["date"], x, color="#91b7d8", lw=0.55, alpha=0.8, label="daily RV")
            ax.plot(work["date"], x.rolling(22).mean(), color="#d95f02", lw=1.1, label="22-day mean")
            ax.set_title(ticker)
            ax.set_ylabel("Realized volatility")
        for ax in axes[len(rv_cols):]:
            ax.set_visible(False)
        axes[0].legend(frameon=True, fontsize=8)
        fig.suptitle("Realized volatility across assets", y=1.01, fontsize=15, fontweight="bold")
        _save_figure(fig, output_dir, "figure_01_rv_time_series_all_assets")

    return stats


# =============================================================================
# 2. Corsi/HAR preliminary diagnostics
# =============================================================================

def corsi_persistence_diagnostics(
    work: pd.DataFrame,
    tickers: list[str],
    output_dir: Path,
    key_ticker: str,
) -> pd.DataFrame:
    rows = []
    requested_lags = [1, 5, 22, 60]
    for ticker in tickers:
        col = f"rv_{ticker}"
        if col not in work:
            continue
        x = _as_numeric(work[col]).dropna()
        if len(x) < 100:
            continue
        max_lag = min(PRELIM_ACF_MAX_LAG, len(x) // 2 - 1)
        acf_values = acf(x, nlags=max_lag, fft=True, missing="drop")
        record = {"ticker": ticker, "n": int(len(x))}
        for lag in requested_lags:
            record[f"acf_lag_{lag}"] = acf_values[lag] if lag <= max_lag else np.nan
        lb_lags = [lag for lag in [5, 22, 60] if lag <= max_lag]
        if lb_lags:
            lb = acorr_ljungbox(x, lags=lb_lags, return_df=True)
            for lag in lb_lags:
                record[f"ljung_box_p_lag_{lag}"] = lb.loc[lag, "lb_pvalue"]
        rows.append(record)
    table = pd.DataFrame(rows)
    _save_csv_tex(table, output_dir, "table_02_corsi_persistence_acf_ljung_box")

    key_col = f"rv_{key_ticker}"
    if key_col in work:
        x = _as_numeric(work[key_col]).dropna()
        plot_lags = min(PRELIM_ACF_MAX_LAG, len(x) // 2 - 1)
        fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
        plot_acf(x, lags=plot_lags, alpha=0.05, fft=True, ax=axes[0], zero=False)
        plot_pacf(x, lags=plot_lags, alpha=0.05, method="ywm", ax=axes[1], zero=False)
        axes[0].set_title(f"ACF — RV {key_ticker}")
        axes[1].set_title(f"PACF — RV {key_ticker}")
        fig.suptitle("Persistence and approximate long memory of realized volatility", fontweight="bold")
        _save_figure(fig, output_dir, f"figure_02_acf_pacf_{_safe_filename(key_ticker)}")
    return table


def _har_data(
    work: pd.DataFrame,
    ticker: str,
    h: int,
    form: str,
) -> tuple[pd.DataFrame, str, list[str]]:
    if form == "level":
        y_col = f"rv_fwd_h{h}_{ticker}"
        x_cols = [f"rv_d_{ticker}", f"rv_w_{ticker}", f"rv_m_{ticker}"]
        return work[[y_col] + x_cols].apply(_as_numeric), y_col, x_cols

    rv = _as_numeric(work[f"rv_{ticker}"]).clip(lower=1e-10)
    log_rv = np.log(rv)
    temp = pd.DataFrame(index=work.index)
    y_col = f"log_rv_fwd_h{h}_{ticker}"
    x_cols = [f"log_rv_d_{ticker}", f"log_rv_w_{ticker}", f"log_rv_m_{ticker}"]
    temp[y_col] = _future_average(log_rv, h)
    temp[x_cols[0]] = log_rv
    temp[x_cols[1]] = log_rv.rolling(5).mean()
    temp[x_cols[2]] = log_rv.rolling(22).mean()
    return temp, y_col, x_cols


def _wald_test(res, restriction: np.ndarray, value: np.ndarray | None = None) -> float:
    try:
        if value is None:
            test = res.wald_test(restriction, scalar=True)
        else:
            test = res.wald_test((restriction, value), scalar=True)
        return float(np.asarray(test.pvalue).squeeze())
    except Exception:
        return np.nan


def estimate_nested_har_models(
    work: pd.DataFrame,
    tickers: list[str],
    horizons: list[int],
    output_dir: Path,
) -> dict[str, pd.DataFrame]:
    forms = ["level"] + (["log"] if PRELIM_RUN_LOG_HAR_ROBUSTNESS else [])
    fit_rows, coef_rows, wald_rows = [], [], []

    for form in forms:
        for ticker in tickers:
            if f"rv_{ticker}" not in work:
                continue
            for h in horizons:
                data, y_col, x_cols = _har_data(work, ticker, h, form)
                model_specs = {"D": x_cols[:1], "D_W": x_cols[:2], "D_W_M": x_cols}
                for spec, cols in model_specs.items():
                    sample = data[[y_col] + cols].dropna()
                    if len(sample) <= len(cols) + 30:
                        continue
                    X = sm.add_constant(sample[cols], has_constant="add")
                    res = sm.OLS(sample[y_col], X).fit(
                        cov_type="HAC", cov_kwds={"maxlags": max(int(h) - 1, 1)}
                    )
                    fitted_error = sample[y_col] - res.fittedvalues
                    fit_rows.append({
                        "form": form, "ticker": ticker, "horizon": h, "specification": spec,
                        "n": int(res.nobs), "r2": res.rsquared, "adj_r2": res.rsquared_adj,
                        "in_sample_rmse": float(np.sqrt(np.mean(np.square(fitted_error)))),
                        "aic": res.aic, "bic": res.bic,
                        "note": "in-sample diagnostic; not OOS forecast evidence",
                    })
                    for position, term in enumerate(["const"] + cols):
                        component = "const" if term == "const" else ["daily", "weekly", "monthly"][x_cols.index(term)]
                        coef_rows.append({
                            "form": form, "ticker": ticker, "horizon": h,
                            "specification": spec, "component": component, "term": term,
                            "coef": res.params.iloc[position], "se_hac": res.bse.iloc[position],
                            "t_hac": res.tvalues.iloc[position], "p_hac": res.pvalues.iloc[position],
                        })

                    if spec == "D_W_M":
                        k = len(res.params)
                        joint_wm = np.zeros((2, k))
                        joint_wm[0, 2] = 1.0
                        joint_wm[1, 3] = 1.0
                        sum_dwm = np.zeros((1, k))
                        sum_dwm[0, 1:] = 1.0
                        wald_rows.append({
                            "form": form, "ticker": ticker, "horizon": h,
                            "beta_daily": res.params.iloc[1],
                            "beta_weekly": res.params.iloc[2],
                            "beta_monthly": res.params.iloc[3],
                            "beta_sum": res.params.iloc[1:].sum(),
                            "p_joint_weekly_monthly_eq_0": _wald_test(res, joint_wm),
                            "p_monthly_eq_0": float(res.pvalues.iloc[3]),
                            "p_beta_sum_eq_1": _wald_test(res, sum_dwm, np.array([1.0])),
                        })

    fits = pd.DataFrame(fit_rows)
    coefs = pd.DataFrame(coef_rows)
    wald = pd.DataFrame(wald_rows)
    if not coefs.empty:
        coefs = _bh_adjust(coefs, "p_hac", ["form", "horizon"], "q_hac_bh")
        coefs["sign"] = np.where(coefs["coef"] > 0, "positive", np.where(coefs["coef"] < 0, "negative", "zero"))
    if not wald.empty:
        wald = _bh_adjust(
            wald, "p_joint_weekly_monthly_eq_0", ["form", "horizon"],
            "q_joint_weekly_monthly_bh",
        )

    _save_csv_tex(fits, output_dir, "table_03_nested_har_fit")
    _save_csv_tex(coefs, output_dir, "table_04_har_coefficients_hac")
    _save_csv_tex(wald, output_dir, "table_05_har_multiscale_wald_tests")

    if not coefs.empty:
        full = coefs[(coefs["form"] == "level") & (coefs["specification"] == "D_W_M") & (coefs["component"] != "const")]
        if not full.empty:
            plot_data = full.pivot_table(index=["ticker", "component"], columns="horizon", values="coef")
            fig, ax = plt.subplots(figsize=(8, max(5, 0.30 * len(plot_data))))
            sns.heatmap(plot_data, cmap="vlag", center=0, annot=True, fmt=".2f", ax=ax, cbar_kws={"label": "HAR coefficient"})
            ax.set_title("HAR daily, weekly and monthly coefficients (level specification)")
            ax.set_xlabel("Forecast horizon")
            ax.set_ylabel("Asset / component")
            _save_figure(fig, output_dir, "figure_03_har_multiscale_coefficients")

    return {"fits": fits, "coefficients": coefs, "wald": wald}


# =============================================================================
# 3. GKG correlations: contemporaneous versus genuinely forward targets
# =============================================================================

def _safe_correlations(x: pd.Series, y: pd.Series, min_obs: int) -> dict:
    pair = pd.concat([_as_numeric(x), _as_numeric(y)], axis=1).dropna()
    if len(pair) < min_obs or pair.iloc[:, 0].nunique() < 2 or pair.iloc[:, 1].nunique() < 2:
        return {"n": int(len(pair)), "pearson_r": np.nan, "pearson_p": np.nan,
                "spearman_rho": np.nan, "spearman_p": np.nan}
    pr = pearsonr(pair.iloc[:, 0], pair.iloc[:, 1])
    sr = spearmanr(pair.iloc[:, 0], pair.iloc[:, 1])
    return {
        "n": int(len(pair)), "pearson_r": float(pr.statistic), "pearson_p": float(pr.pvalue),
        "spearman_rho": float(sr.statistic), "spearman_p": float(sr.pvalue),
    }


def gkg_correlation_analysis(
    work: pd.DataFrame,
    tickers: list[str],
    horizons: list[int],
    broad_specs: pd.DataFrame,
    output_dir: Path,
    key_ticker: str,
) -> pd.DataFrame:
    rows = []
    for spec in broad_specs.to_dict("records"):
        feature_col = spec["column"]
        for ticker in tickers:
            current_col = f"rv_d_{ticker}"
            if current_col in work:
                corr = _safe_correlations(work[feature_col], work[current_col], PRELIM_MIN_CORR_OBS)
                rows.append({
                    "relation": "contemporaneous_t", "horizon": 0, "ticker": ticker,
                    "route": spec["route"], "feature": spec["feature"], "feature_column": feature_col,
                    "target": current_col, **corr,
                })
            for h in horizons:
                target_col = f"rv_fwd_h{h}_{ticker}"
                if target_col not in work:
                    continue
                corr = _safe_correlations(work[feature_col], work[target_col], PRELIM_MIN_CORR_OBS)
                rows.append({
                    "relation": "forward_t_plus_1_to_t_plus_h", "horizon": h, "ticker": ticker,
                    "route": spec["route"], "feature": spec["feature"], "feature_column": feature_col,
                    "target": target_col, **corr,
                })
    table = pd.DataFrame(rows)
    table = _bh_adjust(table, "pearson_p", ["relation", "horizon"], "pearson_q_bh")
    table = _bh_adjust(table, "spearman_p", ["relation", "horizon"], "spearman_q_bh")
    if not table.empty:
        table["pearson_sign"] = np.where(table["pearson_r"] > 0, "positive", np.where(table["pearson_r"] < 0, "negative", "zero"))
    _save_csv_tex(table, output_dir, "table_06_gkg_and_vix_correlations")

    if table.empty:
        print("  [note] No GKG/VIX series was available for the correlation block.")
        return table

    # Heatmap uses cross-asset means only for legibility; the complete asset-level data are in the CSV.
    gkg_forward = table[(table["route"] == "GKG") & (table["relation"] == "forward_t_plus_1_to_t_plus_h")]
    if not gkg_forward.empty:
        heat = gkg_forward.pivot_table(index="feature", columns="horizon", values="pearson_r", aggfunc="mean")
        fig, ax = plt.subplots(figsize=(7.5, max(4.5, 0.48 * len(heat))))
        sns.heatmap(heat, cmap="vlag", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f", ax=ax)
        ax.set_title("Mean forward correlation: GKG at t vs future RV across assets")
        ax.set_xlabel("Forecast horizon h")
        ax.set_ylabel("GKG variable")
        _save_figure(fig, output_dir, "figure_04_gkg_forward_correlation_heatmap")

    # Scatterplots deliberately use future h=5 RV, not same-day RV.
    h_show = 5 if 5 in horizons else horizons[0]
    target = f"rv_fwd_h{h_show}_{key_ticker}"
    scatter_features = ["gkg_sentiment", "gkg_negative_tone", "VIX"]
    chosen = broad_specs[broad_specs["feature"].isin(scatter_features)].drop_duplicates("feature")
    if target in work and not chosen.empty:
        fig, axes = plt.subplots(1, len(chosen), figsize=(5.2 * len(chosen), 4.4), squeeze=False)
        for ax, spec in zip(axes[0], chosen.to_dict("records")):
            pair = work[[spec["column"], target]].apply(_as_numeric).dropna()
            sns.regplot(
                data=pair, x=spec["column"], y=target, ax=ax,
                scatter_kws={"alpha": 0.16, "s": 12},
                line_kws={"color": "#d95f02", "lw": 1.5}, ci=None,
            )
            corr = _safe_correlations(pair.iloc[:, 0], pair.iloc[:, 1], PRELIM_MIN_CORR_OBS)
            ax.set_title(f"{spec['feature']}\nr={corr['pearson_r']:.3f}; rho={corr['spearman_rho']:.3f}")
            ax.set_xlabel(f"{spec['feature']} at t")
            ax.set_ylabel(f"Mean RV {key_ticker}, t+1:t+{h_show}")
        fig.suptitle("Forward, not contemporaneous, relationships", fontweight="bold", y=1.03)
        _save_figure(fig, output_dir, f"figure_05_forward_scatter_{_safe_filename(key_ticker)}_h{h_show}")
    return table


# =============================================================================
# 4. Conditional exploratory screen: future RV ~ HAR + one text indicator
# =============================================================================

def conditional_har_text_screen(
    work: pd.DataFrame,
    tickers: list[str],
    horizons: list[int],
    screen_specs: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    rows = []
    for ticker in tickers:
        har_cols = [f"rv_d_{ticker}", f"rv_w_{ticker}", f"rv_m_{ticker}"]
        if any(c not in work for c in har_cols):
            continue
        for h in horizons:
            y_col = f"rv_fwd_h{h}_{ticker}"
            if y_col not in work:
                continue
            base_sample = work[[y_col] + har_cols].apply(_as_numeric).dropna()
            if len(base_sample) < 100:
                continue
            X0 = sm.add_constant(base_sample[har_cols], has_constant="add")
            base_res = sm.OLS(base_sample[y_col], X0).fit()
            base_adj_r2 = base_res.rsquared_adj

            for spec in screen_specs.to_dict("records"):
                feature_col = spec["column"]
                sample = work[[y_col] + har_cols + [feature_col]].apply(_as_numeric).dropna()
                if len(sample) < PRELIM_MIN_CORR_OBS or sample[feature_col].std(ddof=0) <= 0:
                    continue
                z_col = "__feature_z"
                sample[z_col] = (sample[feature_col] - sample[feature_col].mean()) / sample[feature_col].std(ddof=0)
                X = sm.add_constant(sample[har_cols + [z_col]], has_constant="add")
                res = sm.OLS(sample[y_col], X).fit(
                    cov_type="HAC", cov_kwds={"maxlags": max(int(h) - 1, 1)}
                )

                # Refit the HAR baseline on the exact same observations for a clean delta-R2.
                X_same = sm.add_constant(sample[har_cols], has_constant="add")
                base_same = sm.OLS(sample[y_col], X_same).fit()
                beta = float(res.params[z_col])
                rows.append({
                    "ticker": ticker, "horizon": h, "route": spec["route"],
                    "feature": spec["feature"], "feature_column": feature_col,
                    "n": int(res.nobs), "beta_per_1sd": beta,
                    "se_hac": float(res.bse[z_col]), "t_hac": float(res.tvalues[z_col]),
                    "p_hac": float(res.pvalues[z_col]),
                    "sign": "positive" if beta > 0 else ("negative" if beta < 0 else "zero"),
                    "har_adj_r2_same_sample": float(base_same.rsquared_adj),
                    "harx_adj_r2": float(res.rsquared_adj),
                    "delta_adj_r2": float(res.rsquared_adj - base_same.rsquared_adj),
                    "full_available_har_adj_r2_reference": float(base_adj_r2),
                    "interpretation": "conditional in-sample association; not causal and not OOS evidence",
                })
    table = pd.DataFrame(rows)
    table = _bh_adjust(table, "p_hac", ["horizon"], "q_hac_bh")
    if not table.empty:
        table["significant_bh_5pct"] = table["q_hac_bh"] < PRELIM_FDR_ALPHA
    _save_csv_tex(table, output_dir, "table_07_har_conditional_text_screen")

    if not table.empty:
        heat = table.pivot_table(index=["route", "feature"], columns="horizon", values="beta_per_1sd", aggfunc="mean")
        fig, ax = plt.subplots(figsize=(8, max(4.5, 0.55 * len(heat))))
        sns.heatmap(heat, cmap="vlag", center=0, annot=True, fmt=".3f", ax=ax, cbar_kws={"label": "Mean beta per 1 SD"})
        ax.set_title("HAR-conditional association with future RV (mean across assets)")
        ax.set_xlabel("Forecast horizon h")
        ax.set_ylabel("Route / feature")
        _save_figure(fig, output_dir, "figure_06_har_conditional_text_screen")
    return table


# =============================================================================
# 5. Redundancy and stability: GKG/route correlations and rolling correlations
# =============================================================================

def redundancy_and_rolling_analysis(
    work: pd.DataFrame,
    broad_specs: pd.DataFrame,
    screen_specs: pd.DataFrame,
    output_dir: Path,
    key_ticker: str,
    horizons: list[int],
) -> None:
    specs = pd.concat([broad_specs[broad_specs["route"] == "GKG"], screen_specs], ignore_index=True)
    specs = specs.drop_duplicates("column")
    cols = [c for c in specs["column"] if c in work]
    labels = dict(zip(specs["column"], specs["route"] + ": " + specs["feature"]))
    if len(cols) >= 2:
        corr = work[cols].apply(_as_numeric).corr(method="spearman").rename(index=labels, columns=labels)
        corr.to_csv(output_dir / "table_08_text_redundancy_spearman.csv")
        mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
        fig, ax = plt.subplots(figsize=(max(9, 0.62 * len(corr)), max(7, 0.55 * len(corr))))
        sns.heatmap(corr, mask=mask, cmap="vlag", center=0, vmin=-1, vmax=1,
                    annot=len(corr) <= 14, fmt=".2f", square=True, ax=ax)
        ax.set_title("Redundancy among GKG and primary text indicators (Spearman)")
        _save_figure(fig, output_dir, "figure_07_text_redundancy_heatmap")

    h_show = 5 if 5 in horizons else horizons[0]
    target = f"rv_fwd_h{h_show}_{key_ticker}"
    wanted = broad_specs[broad_specs["feature"].isin(["gkg_sentiment", "gkg_negative_tone", "VIX"])]
    if target in work and not wanted.empty:
        rolling = pd.DataFrame({"date": work["date"]})
        for spec in wanted.drop_duplicates("feature").to_dict("records"):
            rolling[spec["feature"]] = _as_numeric(work[spec["column"]]).rolling(
                PRELIM_ROLLING_CORR_WINDOW, min_periods=PRELIM_ROLLING_CORR_MIN_OBS
            ).corr(_as_numeric(work[target]))
        rolling.to_csv(output_dir / f"table_09_rolling_correlations_{key_ticker}_h{h_show}.csv", index=False)
        fig, ax = plt.subplots(figsize=(14, 5))
        colors = {"gkg_sentiment": "#1b9e77", "gkg_negative_tone": "#d95f02", "VIX": "#7570b3"}
        for col in rolling.columns.drop("date"):
            ax.plot(rolling["date"], rolling[col], label=col, lw=1.25, color=colors.get(col))
        ax.axhline(0, color="black", lw=0.8, ls="--")
        ax.set_ylim(-1, 1)
        ax.set_ylabel("Rolling Pearson correlation")
        ax.set_title(f"{PRELIM_ROLLING_CORR_WINDOW}-day rolling correlation with future {key_ticker} RV (h={h_show})")
        ax.legend(ncol=3, frameon=True)
        _save_figure(fig, output_dir, f"figure_08_rolling_correlations_{_safe_filename(key_ticker)}_h{h_show}")


# =============================================================================
# 6. Main entry point
# =============================================================================

def run_preliminary_analysis(
    input_frame: pd.DataFrame,
    tickers: list[str] | None = None,
    horizons: list[int] | None = None,
    output_root: str | os.PathLike | None = None,
    timing_mode: str | None = None,
    route_primary: dict[str, str] | None = None,
) -> dict[str, pd.DataFrame]:
    """Run the complete preliminary-analysis block and return its main tables."""
    sns.set_theme(style="whitegrid", context="notebook")
    warnings.filterwarnings("ignore", category=FutureWarning)

    work = input_frame.copy()
    if "date" not in work:
        raise ValueError("df_raw must contain a 'date' column.")
    work["date"] = pd.to_datetime(work["date"], errors="coerce")
    work = work.sort_values("date").reset_index(drop=True)

    tickers = list(tickers or globals().get("TICKERS", [
        "SPY", "XLE", "XLF", "XLK", "XLY", "XLP", "XLI", "XLV", "XLB", "XLU", "XLRE", "XLC"
    ]))
    horizons = sorted({int(h) for h in (horizons or globals().get("H_LIST", [1, 5, 22]))})
    timing_mode = str(timing_mode or globals().get("TEXT_TIMING_MODE", "same_day"))
    if timing_mode not in {"same_day", "lag1_all"}:
        raise ValueError("timing_mode must be 'same_day' or 'lag1_all'.")

    root = Path(output_root or globals().get("OUT_DIR", "/content/preliminary_outputs"))
    output_dir = root / "preliminary_analysis"
    output_dir.mkdir(parents=True, exist_ok=True)

    valid_tickers = [t for t in tickers if f"rv_{t}" in work]
    if not valid_tickers:
        raise ValueError("No rv_{ticker} column was found in df_raw.")
    key_ticker = PRELIM_KEY_TICKER if PRELIM_KEY_TICKER in valid_tickers else valid_tickers[0]

    _ensure_core_columns(work, valid_tickers, horizons)
    route_primary = route_primary or globals().get("TEXT_ROUTE_PRIMARY", None)
    broad_specs, screen_specs = _collect_feature_specs(
        work, timing_mode, route_primary, PRELIM_INCLUDE_LEARNED_ROUTES
    )
    broad_specs.to_csv(output_dir / "feature_registry_broad_correlations.csv", index=False)
    screen_specs.to_csv(output_dir / "feature_registry_conditional_screen.csv", index=False)

    print("\n[Preliminary analysis]")
    print(f"  Dates: {work['date'].min().date()} to {work['date'].max().date()}")
    print(f"  Assets: {valid_tickers}")
    print(f"  Horizons: {horizons}")
    print(f"  Information set: {timing_mode}; predictors at t, target over t+1,...,t+h")
    print(f"  Output: {output_dir}")

    print("\n1/5 Descriptive statistics and time series ...")
    descriptive = descriptive_analysis(work, valid_tickers, broad_specs, screen_specs, output_dir)

    print("2/5 Corsi persistence and heterogeneous-horizon diagnostics ...")
    persistence = corsi_persistence_diagnostics(work, valid_tickers, output_dir, key_ticker)
    har_results = estimate_nested_har_models(work, valid_tickers, horizons, output_dir)

    print("3/5 GKG contemporaneous and forward correlations ...")
    correlations = gkg_correlation_analysis(
        work, valid_tickers, horizons, broad_specs, output_dir, key_ticker
    )

    print("4/5 HAR-conditional text screen (positive and negative signs retained) ...")
    conditional = conditional_har_text_screen(
        work, valid_tickers, horizons, screen_specs, output_dir
    )

    print("5/5 Redundancy and rolling-stability diagnostics ...")
    redundancy_and_rolling_analysis(
        work, broad_specs, screen_specs, output_dir, key_ticker, horizons
    )

    manifest = {
        "timing_mode": timing_mode,
        "target_definition": "mean realized volatility over t+1,...,t+h",
        "tickers": valid_tickers,
        "horizons": horizons,
        "hac_lags": {str(h): max(h - 1, 1) for h in horizons},
        "fdr_method": "Benjamini-Hochberg",
        "fdr_alpha": PRELIM_FDR_ALPHA,
        "rolling_correlation_window": PRELIM_ROLLING_CORR_WINDOW,
        "interpretation_limits": [
            "All HAR comparisons in this module are in-sample preliminary diagnostics.",
            "Contemporaneous correlation is separated from forward correlation.",
            "Conditional HAR-X coefficients are associations, not causal effects.",
            "Positive and negative significant coefficients are both retained.",
            "OOS RMSE/QLIKE/Clark-West/DM results from the main pipeline remain the prediction evidence.",
        ],
    }
    with open(output_dir / "preliminary_manifest.json", "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)

    if not conditional.empty:
        significant = conditional[conditional["q_hac_bh"] < PRELIM_FDR_ALPHA].copy()
        significant = significant.sort_values(["q_hac_bh", "horizon", "route", "ticker"])
        significant.to_csv(output_dir / "table_10_significant_conditional_associations_both_signs.csv", index=False)
        print(f"\n  BH-significant conditional associations: {len(significant)}")
        if len(significant):
            display_cols = ["ticker", "horizon", "route", "feature", "beta_per_1sd", "sign", "q_hac_bh", "delta_adj_r2"]
            print(significant[display_cols].head(20).to_string(index=False))

    print("\nDone. These outputs are preliminary diagnostics; use the main OOS tables for predictive claims.")
    return {
        "descriptive": descriptive,
        "persistence": persistence,
        "har_fits": har_results["fits"],
        "har_coefficients": har_results["coefficients"],
        "har_wald": har_results["wald"],
        "correlations": correlations,
        "conditional_screen": conditional,
    }


# When pasted into a Colab cell after the main pipeline, this runs automatically.
if __name__ == "__main__":
    if "df_raw" not in globals():
        raise RuntimeError(
            "df_raw is not in memory. Run the main pipeline through sections [7] and [9] first, "
            "then execute this preliminary-analysis cell."
        )
    PRELIM_RESULTS = run_preliminary_analysis(
        df_raw,
        tickers=globals().get("TICKERS"),
        horizons=globals().get("H_LIST"),
        output_root=globals().get("OUT_DIR"),
        timing_mode=globals().get("TEXT_TIMING_MODE", "same_day"),
        route_primary=globals().get("TEXT_ROUTE_PRIMARY"),
    )



[Preliminary analysis]
  Dates: 2015-02-04 to 2026-05-21
  Assets: ['SPY', 'XLE', 'XLF', 'XLK', 'XLY', 'XLP', 'XLI', 'XLV', 'XLB', 'XLU', 'XLRE', 'XLC']
  Horizons: [1, 5, 22]
  Information set: same_day; predictors at t, target over t+1,...,t+h
  Output: /content/drive/MyDrive/har_pipeline_final_same_day/outputs/preliminary_analysis

1/5 Descriptive statistics and time series ...
2/5 Corsi persistence and heterogeneous-horizon diagnostics ...
3/5 GKG contemporaneous and forward correlations ...
4/5 HAR-conditional text screen (positive and negative signs retained) ...
5/5 Redundancy and rolling-stability diagnostics ...

  BH-significant conditional associations: 25
ticker  horizon    route              feature  beta_per_1sd     sign  q_hac_bh  delta_adj_r2
   XLE       22      GKG log1p_gkg_n_articles     -0.013523 negative  0.000008      0.021222
  XLRE       22      GKG log1p_gkg_n_articles     -0.011078 negative  0.000158      0.027693
   XLK       22      GKG log1p_gkg_n_article

In [ ]:

# %% ── 9.5 Fonction OOS GARCH en deux étapes ──────────────────
def _coerce_text_frame(text_data, reference_index):
    """Normalise une Series/DataFrame de variables textuelles sans modifier leurs noms."""
    if text_data is None:
        return pd.DataFrame(index=reference_index)
    if isinstance(text_data, pd.Series):
        name = text_data.name if text_data.name is not None else 'text_feature'
        frame = text_data.rename(name).to_frame()
    elif isinstance(text_data, pd.DataFrame):
        frame = text_data.copy()
    else:
        raise TypeError('text_data doit être une pandas Series, un DataFrame ou None')
    frame = frame.reindex(reference_index)
    for col in frame.columns:
        frame[col] = pd.to_numeric(frame[col], errors='coerce')
    return frame


def garch_two_step_oos(ret_series, rv_fwd_series, text_data=None, h=22,
                       min_train=MIN_TRAIN_OOS, min_train_date=OOS_START_DATE,
                       step=None, coef_sink=None,
                       window_type='expanding', trailing_window=ROLLING_WINDOW,
                       return_series=False):
    """
    Procédure GARCH en deux étapes avec la conception A : une seule prévision
    tous les GARCH_OOS_STEP=22 jours, quel que soit h. text_data contient toutes
    les variables de la route, exactement comme dans HAR-X.
    """
    if window_type not in {'expanding', 'rolling'}:
        raise ValueError("window_type must be 'expanding' or 'rolling'")
    if step is None:
        step = GARCH_OOS_STEP

    ret_pct = pd.to_numeric(ret_series, errors='coerce') * 100
    base = pd.concat({
        'ret': ret_pct,
        'rv_fwd': pd.to_numeric(rv_fwd_series, errors='coerce'),
        'date': pd.to_datetime(df_raw['date']),
    }, axis=1)
    text_frame = _coerce_text_frame(text_data, base.index)
    text_cols = list(text_frame.columns)
    comb = pd.concat([base, text_frame], axis=1)
    required = ['ret', 'rv_fwd', 'date'] + text_cols
    comb = comb.dropna(subset=required).reset_index(drop=True)
    n = len(comb)
    if min_train_date is not None:
        min_train = int(np.searchsorted(comb['date'].values, np.datetime64(min_train_date)))
        min_train = max(min_train, min(MIN_TRAIN_OOS, max(n // 2, 1)))
    if n < min_train + step:
        empty = (np.array([]), np.array([]), np.array([]))
        return (np.nan, np.nan, np.nan, 0, *empty) if return_series else (np.nan, np.nan, np.nan, 0)

    purge = max(int(h) - 1, 0)
    preds, acts, dates, bench = [], [], [], []

    for t in range(min_train, n, step):
        stage2_end = t - purge
        if stage2_end <= 0:
            continue
        stage2_start = max(0, stage2_end - trailing_window) if window_type == 'rolling' else 0
        map_train = comb.iloc[stage2_start:stage2_end].copy()
        if len(map_train) < 30:
            continue
        garch_train = comb.iloc[stage2_start:t + 1].copy()
        if len(garch_train) < 50:
            continue

        try:
            g_rv_hist, res_g = _fit_garch_get_sigma(garch_train['ret'])
            g_rv_hist = pd.Series(np.asarray(g_rv_hist, dtype=float),
                                  index=garch_train.index, name='garch_rv')
            refit_date = comb['date'].iloc[t]

            if coef_sink is not None:
                for term in res_g.params.index:
                    coef_sink.append({
                        'refit_date': refit_date, 'component': 'native_garch', 'term': term,
                        'coef': res_g.params[term], 'se': res_g.std_err[term],
                        'tstat': res_g.tvalues[term], 'pval': res_g.pvalues[term],
                        'window_type': window_type, 'n_train': len(garch_train),
                    })

            garch_pred = _garch_forecast_sigma(res_g, h)
            train2 = pd.concat(
                [map_train['rv_fwd'], g_rv_hist] + [map_train[c] for c in text_cols],
                axis=1).dropna()
            if len(train2) < 30:
                continue

            xcols = ['garch_rv'] + [
                c for c in text_cols if train2[c].notna().sum() > 30]
            ols_model = sm.OLS(train2['rv_fwd'], sm.add_constant(train2[xcols], has_constant='add'))
            res_ols = ols_model.fit()

            if coef_sink is not None:
                try:
                    res_ols_hac = ols_model.fit(
                        cov_type='HAC', cov_kwds={'maxlags': _har_hac_lags(h)})
                    for term in res_ols_hac.params.index:
                        coef_sink.append({
                            'refit_date': refit_date, 'component': 'second_stage', 'term': term,
                            'coef': res_ols_hac.params[term], 'se': res_ols_hac.bse[term],
                            'tstat': res_ols_hac.tvalues[term], 'pval': res_ols_hac.pvalues[term],
                            'window_type': window_type, 'n_train': len(train2),
                        })
                except Exception:
                    pass

            feat_dict = {'const': 1.0, 'garch_rv': garch_pred}
            for c in text_cols:
                feat_dict[c] = comb[c].iloc[t]
            x_te = np.array([feat_dict.get(c, 0.0) for c in res_ols.params.index], dtype=float)
            rv_pred = float(x_te @ res_ols.params.values)

            preds.append(rv_pred)
            acts.append(float(comb['rv_fwd'].iloc[t]))
            dates.append(refit_date)
            bench.append(float(train2['rv_fwd'].mean()))
        except Exception:
            continue

    if len(preds) < 5:
        empty = (np.asarray(dates), np.asarray(acts), np.asarray(preds))
        return (np.nan, np.nan, np.nan, len(preds), *empty) if return_series else (np.nan, np.nan, np.nan, len(preds))

    p_arr = np.asarray(preds, dtype=float)
    a_arr = np.asarray(acts, dtype=float)
    b_arr = np.asarray(bench, dtype=float)
    oos_r2 = 1 - np.sum((a_arr - p_arr) ** 2) / max(np.sum((a_arr - b_arr) ** 2), 1e-16)
    oos_rmse = np.sqrt(np.mean((a_arr - p_arr) ** 2))
    oos_qlike = qlike_loss(a_arr, p_arr)
    if return_series:
        return oos_r2, oos_rmse, oos_qlike, len(preds), np.asarray(dates), a_arr, p_arr
    return oos_r2, oos_rmse, oos_qlike, len(preds)


print('✅ Protection de l’ensemble d’information activée : purge (h-1) appliquée aux évaluations OOS HAR et GARCH')


# %% ── 10. Matrice principale HAR-RV-X — réestimation tous les 22 jours, prévisions quotidiennes (Spec1/2/3 + orthogonalisation Spec4 + export des coefficients v8) ──────
print('\n[10] Matrice principale HAR-RV-X (calcul en cours...)')
t0 = time.time()

all_gkg_feats  = [f for feats in GKG_ROUTE_FEATS.values()  for f in feats]
all_text_feats = [f for feats in TEXT_ROUTE_FEATS.values() for f in feats]

records_gkg  = []
records_text = []
records_orthog = []
pred_records = []
vif_records = []
records_text_rolling = []   # [ajout v11] Synthèse OOS_R2/RMSE/QLIKE de Spec1-Rolling/Spec2-Rolling
pred_records_rolling = []   # [ajout v11] Prévisions journalières Spec1-Rolling/Spec2-Rolling pour le test CW en fenêtre rolling
records_har_coefs = []   # [ajout v8]
records_text_allcoefs = []   # [ajout v9, correction n°8] Collecte de tous les coefficients/p-values textuels de chaque route dans Spec2,
                              # et non uniquement du premier coefficient sélectionné par extract_text_coef,
                              # afin de les inclure dans la correction FDR et d’éviter d’omettre les autres variables
                              # des routes FINBERT/LM/FWD_SENT.


_HAR_ACCUMULATORS = {
    'records_gkg': records_gkg,
    'records_text': records_text,
    'records_orthog': records_orthog,
    'pred_records': pred_records,
    'vif_records': vif_records,
    'records_text_rolling': records_text_rolling,
    'pred_records_rolling': pred_records_rolling,
    'records_har_coefs': records_har_coefs,
    'records_text_allcoefs': records_text_allcoefs,
}

for ticker in TICKERS:
    if f'rv_d_{ticker}' not in df_raw.columns:
        continue
    base  = har_base(ticker)
    ret_c = f'ret_{ticker}'
    macro_cols_t = macro_controls(ticker)

    for h in H_LIST:
        y_col = f'rv_fwd_h{h}_{ticker}'
        if y_col not in df_raw.columns:
            continue

        need_cols = (['date', y_col] + base + macro_cols_t
                     + all_gkg_feats + all_text_feats)
        dm = df_raw[[c for c in need_cols if c in df_raw.columns]].dropna(subset=['date'])

        if dm[[y_col] + base].dropna().shape[0] < MIN_TRAIN_OOS + 1:
            print(f'  ⚠️ {ticker} h={h} ignoré : nombre insuffisant d’observations valides, possiblement en raison d’une date de lancement plus récente')
            continue

        _har_key = f'har_full_matrix_{ticker}_h{h}'
        _har_feature_spec = {
            'base': base,
            'macro': macro_cols_t,
            'gkg_routes': GKG_ROUTE_FEATS,
            'text_routes': TEXT_ROUTE_FEATS,
            'expanding': True,
            'rolling': True,
            'input_cols': ['date', y_col] + base + macro_cols_t + all_gkg_feats + all_text_feats,
        }
        _har_sig = _model_cache_signature('HAR', ticker, h, 'full_matrix', _har_feature_spec)
        _har_cached = _load_model_cache(
            'HAR', _har_key, _har_sig, force=FORCE_RERUN_HAR_MODELS)
        if _har_cached is not None:
            _restore_lists(_HAR_ACCUMULATORS, _har_cached)
            print(f'  ↺ HAR {ticker} h={h} restauré depuis le nouveau cache')
            continue
        _har_starts = _snapshot_lengths(_HAR_ACCUMULATORS)

        # Spec1 : benchmark purement financier
        res1 = run_ols_is(dm, y_col, base, h)
        oos1, _, dates1, act1, pred1 = oos_r2_expanding(
            dm, y_col, base, h=h, min_train_date=OOS_START_DATE, return_series=True)
        is_rmse1  = np.sqrt(res1.mse_resid)
        oos_rmse1 = np.sqrt(np.mean((act1 - pred1) ** 2)) if len(act1) else np.nan
        oos_qlike1 = qlike_loss(act1, pred1) if len(act1) else np.nan

        records_har_coefs += extract_all_coefs(res1, ticker, h, 'Spec1', 'HAR-RV', route='Benchmark')  # [ajout v8]

        bench_rec = {
            'model': 'HAR-RV', 'route': 'Benchmark', 'spec': 'Spec1',
            'ticker': ticker, 'h': h,
            'IS_R2': res1.rsquared, 'IS_AdjR2': res1.rsquared_adj,
            'IS_RMSE': is_rmse1, 'OOS_R2': oos1, 'OOS_RMSE': oos_rmse1,
            'OOS_QLIKE': oos_qlike1,
            'text_coef': np.nan, 'text_pval': np.nan,
            'n_obs': int(res1.nobs), 'n_oos': len(act1),
        }
        records_gkg.append(bench_rec.copy())
        records_text.append(bench_rec.copy())

        for d, a, p in zip(dates1, act1, pred1):
            pred_records.append({
                'date': d, 'ticker': ticker, 'h': h,
                'model': 'HAR-RV', 'route': 'Benchmark', 'spec': 'Spec1',
                'actual': a, 'pred': p,
            })

# [ajout v11] Spec1-Rolling : nouvelle évaluation OOS avec exactement la même fenêtre trailing_window=ROLLING_WINDOW
# que les trajectoires de coefficients de la section 15, plutôt qu’une fenêtre expanding utilisant tout l’historique.
# Les trajectoires rolling et la précision OOS rolling reposent ainsi sur la même conception et peuvent être interprétées conjointement.
        oos1_roll, _, dates1_roll, act1_roll, pred1_roll = oos_r2_expanding(
            dm, y_col, base, h=h, min_train_date=OOS_START_DATE, return_series=True,
            window_type='rolling', trailing_window=ROLLING_WINDOW)
        oos_rmse1_roll = np.sqrt(np.mean((act1_roll - pred1_roll) ** 2)) if len(act1_roll) else np.nan
        oos_qlike1_roll = qlike_loss(act1_roll, pred1_roll) if len(act1_roll) else np.nan

        records_text_rolling.append({
            'model': 'HAR-RV', 'route': 'Benchmark', 'spec': 'Spec1-Rolling',
            'ticker': ticker, 'h': h,
            'OOS_R2': oos1_roll, 'OOS_RMSE': oos_rmse1_roll, 'OOS_QLIKE': oos_qlike1_roll,
            'n_oos': len(act1_roll),
        })
        for d, a, p in zip(dates1_roll, act1_roll, pred1_roll):
            pred_records_rolling.append({
                'date': d, 'ticker': ticker, 'h': h,
                'model': 'HAR-RV', 'route': 'Benchmark', 'spec': 'Spec1-Rolling',
                'actual': a, 'pred': p,
            })

        # Première partie : SpecG — HAR + GKG
        for route, feats in GKG_ROUTE_FEATS.items():
            avail = [f for f in feats
                     if f in dm.columns and dm[f].notna().sum() > 100]
            if not avail: continue

            xcols_g = base + avail
            res_g   = run_ols_is(dm, y_col, xcols_g, h)
            oos_g, _, dg, ag, pg = oos_r2_expanding(
                dm, y_col, xcols_g, h=h, min_train_date=OOS_START_DATE, return_series=True)
            tc_g, tp_g   = extract_text_coef(res_g, avail)
            is_rmse_g    = np.sqrt(res_g.mse_resid)
            oos_rmse_g   = np.sqrt(np.mean((ag - pg) ** 2)) if len(ag) else np.nan
            oos_qlike_g  = qlike_loss(ag, pg) if len(ag) else np.nan

            records_gkg.append({
                'model': 'HAR-RV-GKG', 'route': route, 'spec': 'SpecG',
                'ticker': ticker, 'h': h,
                'IS_R2': res_g.rsquared, 'IS_AdjR2': res_g.rsquared_adj,
                'IS_RMSE': is_rmse_g, 'OOS_R2': oos_g, 'OOS_RMSE': oos_rmse_g,
                'OOS_QLIKE': oos_qlike_g,
                'text_coef': tc_g, 'text_pval': tp_g,
                'n_obs': int(res_g.nobs), 'n_oos': len(ag),
            })

            xcols_gm = base + avail + macro_cols_t
            res_gm   = run_ols_is(dm, y_col, xcols_gm, h)
            oos_gm, _, dgm, agm, pgm = oos_r2_expanding(
                dm, y_col, xcols_gm, h=h, min_train_date=OOS_START_DATE, return_series=True)
            tc_gm, tp_gm = extract_text_coef(res_gm, avail)
            is_rmse_gm   = np.sqrt(res_gm.mse_resid)
            oos_rmse_gm  = np.sqrt(np.mean((agm - pgm) ** 2)) if len(agm) else np.nan
            oos_qlike_gm = qlike_loss(agm, pgm) if len(agm) else np.nan

            records_gkg.append({
                'model': 'HAR-RV-GKG+Macro', 'route': route, 'spec': 'SpecG+Macro',
                'ticker': ticker, 'h': h,
                'IS_R2': res_gm.rsquared, 'IS_AdjR2': res_gm.rsquared_adj,
                'IS_RMSE': is_rmse_gm, 'OOS_R2': oos_gm, 'OOS_RMSE': oos_rmse_gm,
                'OOS_QLIKE': oos_qlike_gm,
                'text_coef': tc_gm, 'text_pval': tp_gm,
                'n_obs': int(res_gm.nobs), 'n_oos': len(agm),
            })

        # Deuxième partie : Spec2/3 — HAR + routes purement textuelles + orthogonalisation Spec4
        for route, feats in TEXT_ROUTE_FEATS.items():
            if route == 'FINBERT' and not ENABLE_FINBERT: continue
            avail = [f for f in feats
                     if f in dm.columns and dm[f].notna().sum() > 100]
            if not avail: continue

            xcols2 = base + avail
            res2   = run_ols_is(dm, y_col, xcols2, h)
            oos2, _, dates2, act2, pred2 = oos_r2_expanding(
                dm, y_col, xcols2, h=h, min_train_date=OOS_START_DATE, return_series=True)
            tc2, tp2  = extract_text_coef(res2, avail)
            records_har_coefs += extract_all_coefs(res2, ticker, h, 'Spec2', 'HAR-RV-X', route=route)
            is_rmse2  = np.sqrt(res2.mse_resid)
            oos_rmse2 = np.sqrt(np.mean((act2 - pred2) ** 2)) if len(act2) else np.nan
            oos_qlike2 = qlike_loss(act2, pred2) if len(act2) else np.nan

            # [ajout v9, correction n°8] Enregistrement de tous les coefficients de la route, et non uniquement du premier sélectionné par extract_text_coef.
            # Les routes FINBERT/LM/FWD_SENT comportent 2 à 5 variables ; auparavant, seule la première entrait dans la correction FDR.
            for _feat in avail:
                if _feat in res2.params.index:
                    records_text_allcoefs.append({
                        'route': route, 'ticker': ticker, 'h': h, 'feature': _feat,
                        'coef': res2.params[_feat], 'se': res2.bse[_feat],
                        'tstat': res2.tvalues[_feat], 'pval': res2.pvalues[_feat],
                        'n_obs': int(res2.nobs),
                    })

            records_text.append({
                'model': 'HAR-RV-X', 'route': route, 'spec': 'Spec2',
                'ticker': ticker, 'h': h,
                'IS_R2': res2.rsquared, 'IS_AdjR2': res2.rsquared_adj,
                'IS_RMSE': is_rmse2, 'OOS_R2': oos2, 'OOS_RMSE': oos_rmse2,
                'OOS_QLIKE': oos_qlike2,
                'text_coef': tc2, 'text_pval': tp2,
                'n_obs': int(res2.nobs), 'n_oos': len(act2),
            })
            for d, a, p in zip(dates2, act2, pred2):
                pred_records.append({
                    'date': d, 'ticker': ticker, 'h': h,
                    'model': 'HAR-RV-X', 'route': route, 'spec': 'Spec2',
                    'actual': a, 'pred': p,
                })

            # [ajout v11] Spec2-Rolling : appariée à Spec1-Rolling avec le même trailing_window=ROLLING_WINDOW,
            # afin d’effectuer le test de Clark-West rolling entre « HAR + texte » et le HAR rolling de référence.
            oos2_roll, _, dates2_roll, act2_roll, pred2_roll = oos_r2_expanding(
                dm, y_col, xcols2, h=h, min_train_date=OOS_START_DATE, return_series=True,
                window_type='rolling', trailing_window=ROLLING_WINDOW)
            oos_rmse2_roll = np.sqrt(np.mean((act2_roll - pred2_roll) ** 2)) if len(act2_roll) else np.nan
            oos_qlike2_roll = qlike_loss(act2_roll, pred2_roll) if len(act2_roll) else np.nan

            records_text_rolling.append({
                'model': 'HAR-RV-X', 'route': route, 'spec': 'Spec2-Rolling',
                'ticker': ticker, 'h': h,
                'OOS_R2': oos2_roll, 'OOS_RMSE': oos_rmse2_roll, 'OOS_QLIKE': oos_qlike2_roll,
                'n_oos': len(act2_roll),
            })
            for d, a, p in zip(dates2_roll, act2_roll, pred2_roll):
                pred_records_rolling.append({
                    'date': d, 'ticker': ticker, 'h': h,
                    'model': 'HAR-RV-X', 'route': route, 'spec': 'Spec2-Rolling',
                    'actual': a, 'pred': p,
                })

            xcols3 = base + avail + macro_cols_t
            res3   = run_ols_is(dm, y_col, xcols3, h)
            oos3, _, dates3, act3, pred3 = oos_r2_expanding(
                dm, y_col, xcols3, h=h, min_train_date=OOS_START_DATE, return_series=True)
            tc3, tp3  = extract_text_coef(res3, avail)
            is_rmse3  = np.sqrt(res3.mse_resid)
            oos_rmse3 = np.sqrt(np.mean((act3 - pred3) ** 2)) if len(act3) else np.nan
            oos_qlike3 = qlike_loss(act3, pred3) if len(act3) else np.nan

            records_har_coefs += extract_all_coefs(res3, ticker, h, 'Spec3', 'HAR-RV-X+Macro', route=route)  # [ajout v8]

            records_text.append({
                'model': 'HAR-RV-X+Macro', 'route': route, 'spec': 'Spec3',
                'ticker': ticker, 'h': h,
                'IS_R2': res3.rsquared, 'IS_AdjR2': res3.rsquared_adj,
                'IS_RMSE': is_rmse3, 'OOS_R2': oos3, 'OOS_RMSE': oos_rmse3,
                'OOS_QLIKE': oos_qlike3,
                'text_coef': tc3, 'text_pval': tp3,
                'n_obs': int(res3.nobs), 'n_oos': len(act3),
            })
            for d, a, p in zip(dates3, act3, pred3):
                pred_records.append({
                    'date': d, 'ticker': ticker, 'h': h,
                    'model': 'HAR-RV-X+Macro', 'route': route, 'spec': 'Spec3',
                    'actual': a, 'pred': p,
                })

            if h == 22 and ENABLE_COLLINEARITY_DIAG:
                # Le VIF doit couvrir l’ensemble complet des régresseurs de Spec3 pour la route, et non uniquement HAR et les variables macro.
                vif_s = compute_vif(dm, xcols3)
                n_vif_obs = int(dm[xcols3].dropna().shape[0])
                for feat, v in vif_s.items():
                    vif_records.append({
                        'route': route, 'ticker': ticker, 'h': h,
                        'feature': feat, 'VIF': v, 'n_obs': n_vif_obs,
                    })

                # Résidualise la variable textuelle principale par rapport aux variables HAR, macro et aux autres variables textuelles de la même route,
                # puis prévoit avec HAR + autres variables textuelles + résidu. Ce test mesure si la composante du texte
                # non expliquée linéairement par les contrôles conserve une relation prédictive.
                primary_feat = TEXT_ROUTE_PRIMARY.get(route)
                if primary_feat is not None and primary_feat in avail:
                    primary_coef2 = float(res2.params.get(primary_feat, np.nan))
                    primary_pval2 = float(res2.pvalues.get(primary_feat, np.nan))
                    primary_coef3 = float(res3.params.get(primary_feat, np.nan))
                    primary_pval3 = float(res3.pvalues.get(primary_feat, np.nan))
                    other_text = [f for f in avail if f != primary_feat]
                    orthog_controls = base + macro_cols_t + other_text
                    resid_series = orthogonalize_text_feature(
                        dm, primary_feat, orthog_controls)
                    if resid_series is not None:
                        dm_orthog = dm.copy()
                        dm_orthog['_text_resid'] = resid_series
                        xcols4 = base + other_text + ['_text_resid']
                        try:
                            res4 = run_ols_is(dm_orthog, y_col, xcols4, h)
                            tc4, tp4 = extract_text_coef(res4, ['_text_resid'])
                            records_orthog.append({
                                'route': route, 'ticker': ticker, 'h': h,
                                'primary_feature': primary_feat,
                                'spec2_coef': primary_coef2, 'spec2_pval': primary_pval2,
                                'spec3_coef': primary_coef3, 'spec3_pval': primary_pval3,
                                'orthog_coef': tc4, 'orthog_pval': tp4,
                                'orthog_IS_R2': res4.rsquared,
                                'n_obs': int(res4.nobs),
                            })
                        except Exception as e:
                            print(f'  Échec du test de résidualisation du texte {route} {ticker} : {e}')

        _har_payload = _slice_since(_HAR_ACCUMULATORS, _har_starts)
        _save_model_cache('HAR', _har_key, _har_sig, _har_payload)
        print(f'  {ticker} h={h} terminé')

df_har_gkg   = pd.DataFrame(records_gkg)
df_har       = pd.DataFrame(records_text)
df_oos_preds = pd.DataFrame(pred_records)
df_orthog    = pd.DataFrame(records_orthog)
df_vif       = pd.DataFrame(vif_records)
df_har_coefs = pd.DataFrame(records_har_coefs)   # [ajout v8]
df_text_allcoefs_spec2 = pd.DataFrame(records_text_allcoefs)   # [ajout v9, correction n°8]
df_har_rolling       = pd.DataFrame(records_text_rolling)   # [ajout v11]
df_oos_preds_rolling = pd.DataFrame(pred_records_rolling)   # [ajout v11]

print(f'→ HAR terminé en {time.time()-t0:.1f}s')
print(f'  Analyse exploratoire GKG : {len(df_har_gkg)} lignes | expérience principale : {len(df_har)} lignes | '
      f'tests de résidualisation : {len(df_orthog)} lignes | coefficients HAR exportés : {len(df_har_coefs)} lignes | '
      f'tous coefficients Spec2 pour FDR : {len(df_text_allcoefs_spec2)} lignes | '
      f'résultats OOS rolling (fenêtre max. 750 jours) : {len(df_har_rolling)} lignes')

# %% ── 10.5 Synthèse des diagnostics de colinéarité ─────────────────────────────────
print('\n' + '='*65)
print('[Diagnostic] VIF (ensemble complet des régresseurs Spec3 pour chaque route, h=22)')
print('='*65)
if not df_vif.empty:
    vif_pivot = df_vif.pivot_table(
        index=['route', 'feature'], columns='ticker', values='VIF', aggfunc='first')
    print(vif_pivot.round(2).to_string())
    vif_pivot.to_csv(os.path.join(OUT_DIR, 'diag_vif_pivot.csv'))
    df_vif.to_csv(os.path.join(OUT_DIR, 'diag_vif_long.csv'), index=False)

    severe_vif = df_vif[df_vif['VIF'] > 10]
    moderate_vif = df_vif[(df_vif['VIF'] > 5) & (df_vif['VIF'] <= 10)]
    print(f'\n  VIF>10 (sévère) : {len(severe_vif)}  |  5<VIF≤10 (élevé) : {len(moderate_vif)}')
    if not severe_vif.empty:
        print(severe_vif[['route','ticker','feature','VIF','n_obs']]
              .sort_values('VIF', ascending=False).to_string(index=False))
else:
    print('  Aucun résultat VIF disponible.')

# %% ── 10.6 Synthèse du test de robustesse par résidualisation du texte ─────────────────────────
print('\n' + '='*65)
print('[Diagnostic] Test de robustesse par résidualisation du texte (h=22, échantillon interne)')
print('='*65)
if not df_orthog.empty:
    for col in ['spec2_pval', 'spec3_pval', 'orthog_pval']:
        df_orthog[col.replace('pval', 'sig')] = df_orthog[col].apply(fmt_stars)
    df_orthog['spec2_to_orthog_flip'] = (
        np.sign(df_orthog['spec2_coef']) != np.sign(df_orthog['orthog_coef']))
    df_orthog['spec3_to_orthog_flip'] = (
        np.sign(df_orthog['spec3_coef']) != np.sign(df_orthog['orthog_coef']))
    print(df_orthog[[
        'route','ticker','primary_feature',
        'spec2_coef','spec2_sig','spec3_coef','spec3_sig',
        'orthog_coef','orthog_sig','spec2_to_orthog_flip','spec3_to_orthog_flip'
    ]].sort_values(['route','ticker']).to_string(index=False))
    print(f"\n  Changements de signe Spec2→résidualisation : {int(df_orthog['spec2_to_orthog_flip'].sum())} / {len(df_orthog)}")
    print(f"  Changements de signe Spec3→résidualisation : {int(df_orthog['spec3_to_orthog_flip'].sum())} / {len(df_orthog)}")
    df_orthog.to_csv(os.path.join(OUT_DIR, 'diag_orthogonalized_text.csv'), index=False)
else:
    print('  Aucun résultat de résidualisation du texte disponible.')

# %% ── 10.7 Test de robustesse de causalité prédictive de type Granger conditionnel au HAR ────────────────
# Un test de Granger bivarié standard ignorerait la persistance HAR et le chevauchement des cibles pour h=5/22.
# Le test conjoint de Wald conditionnel est donc limité à la cible non chevauchante h=1 : au-delà des contrôles HAR,
# il teste conjointement si les coefficients des retards 1 à PREDICTIVE_CAUSALITY_MAX_LAG des variables textuelles sont nuls relativement à la cible.
print('\n' + '='*65)
print('[Robustesse] Test conjoint de Wald de type Granger conditionnel au HAR (h=1 uniquement)')
print('='*65)

PREDICTIVE_CAUSALITY_SOURCE_FEATS = {
    'LM': ['lm_polarity', 'lm_neg_prop', 'lm_unc_prop', 'lm_polarity_std'],
    'TFIDF_LASSO': ['tfidf_idx'],
    'W2V_PCA': W2V_PC_COLS,
    'FINBERT': ['finbert_sent', 'finbert_neg', 'finbert_frac_neg', 'finbert_sent_std'],
    'FWD_SENT': ['fwd_lm_polarity', 'finbert_sent_fwd', 'has_fwd'],
}


def run_har_conditional_predictive_causality(
        data, ticker, route, source_feats, max_lag=PREDICTIVE_CAUSALITY_MAX_LAG):
    h = 1
    y_col = f'rv_fwd_h{h}_{ticker}'
    base = har_base(ticker)
    available = [f for f in source_feats
                 if f in data.columns and data[f].notna().sum() > 100]
    if y_col not in data.columns or not available:
        return None

    work = data[['date', y_col] + base + available].copy()
    lag_cols = []
# y_col est indexée à l’origine de prévision t et représente la cible du prochain jour de négociation ; feature_t correspond donc au retard 1 relativement à la cible.
    for feat in available:
        for target_lag in range(1, max_lag + 1):
            col = f'{feat}__target_lag{target_lag}'
            work[col] = work[feat].shift(target_lag - 1)
            lag_cols.append(col)

    sub = work[[y_col] + base + lag_cols].dropna()
    if len(sub) < max(200, 8 * (len(base) + len(lag_cols))):
        return None

    X = sm.add_constant(sub[base + lag_cols])
    try:
        res = sm.OLS(sub[y_col], X).fit(
            cov_type='HAC',
            cov_kwds={'maxlags': max(PREDICTIVE_CAUSALITY_MAX_LAG, 1)})
        R = np.zeros((len(lag_cols), len(res.params)))
        for i, col in enumerate(lag_cols):
            R[i, res.params.index.get_loc(col)] = 1.0
        wald = res.wald_test(R, scalar=True)
        return {
            'route': route, 'ticker': ticker, 'h': h,
            'max_target_lag': max_lag,
            'source_features': '|'.join(available),
            'n_restrictions': len(lag_cols),
            'wald_stat': float(np.asarray(wald.statistic).squeeze()),
            'pval': float(np.asarray(wald.pvalue).squeeze()),
            'n_obs': int(res.nobs),
            'adj_R2_unrestricted': float(res.rsquared_adj),
        }
    except Exception as e:
        print(f'  Échec du test de causalité prédictive {route} {ticker} : {e}')
        return None


records_predictive_causality = []
if ENABLE_PREDICTIVE_CAUSALITY:
    for ticker in TICKERS:
        for route, feats in PREDICTIVE_CAUSALITY_SOURCE_FEATS.items():
            if route == 'FINBERT' and not ENABLE_FINBERT:
                continue
            rec = run_har_conditional_predictive_causality(
                df_raw, ticker, route, feats)
            if rec is not None:
                records_predictive_causality.append(rec)

    df_predictive_causality = pd.DataFrame(records_predictive_causality)
    if not df_predictive_causality.empty:
        from statsmodels.stats.multitest import multipletests as _mt_pc
        valid = df_predictive_causality['pval'].notna()
        rej, pfdr, _, _ = _mt_pc(
            df_predictive_causality.loc[valid, 'pval'].values,
            alpha=0.05, method='fdr_bh')
        df_predictive_causality.loc[valid, 'pval_fdr'] = pfdr
        df_predictive_causality.loc[valid, 'sig_fdr'] = rej
        df_predictive_causality['sig_fdr'] = df_predictive_causality['sig_fdr'].fillna(False)
        print(f"  Nombre de tests : {len(df_predictive_causality)}  |  "
              f"p brut<0,05 : {int((df_predictive_causality['pval'] < 0.05).sum())}  |  "
              f"significatifs après FDR : {int(df_predictive_causality['sig_fdr'].sum())}")
        if df_predictive_causality['sig_fdr'].any():
            print(df_predictive_causality[df_predictive_causality['sig_fdr']][
                ['route','ticker','wald_stat','pval','pval_fdr','n_restrictions','n_obs']
            ].sort_values('pval_fdr').to_string(index=False))
        df_predictive_causality.to_csv(
            os.path.join(OUT_DIR, 'diag_har_conditional_predictive_causality_h1.csv'),
            index=False)
    else:
        print('  Aucun résultat de causalité prédictive disponible.')
else:
    df_predictive_causality = pd.DataFrame()
    print('  ENABLE_PREDICTIVE_CAUSALITY=False ; test ignoré.')


# %% ── 11. Matrice GARCH-X — conception A, routes complètes et checkpoints par unité ────────
if ENABLE_GARCH:
    print('\n[11] Matrice GARCH-X (conception A : origine tous les 22 jours ; expanding + rolling)')
    t0 = time.time()
    from arch import arch_model

    def _fit_garch_get_sigma(ret_series_pct):
        mod = arch_model(ret_series_pct, vol='Garch', p=1, q=1,
                         dist='skewt', mean='Constant')
        res = mod.fit(disp='off', show_warning=False)
        sigma_pct2 = res.conditional_volatility ** 2
        garch_rv   = np.sqrt(sigma_pct2 * 252) / 100
        return garch_rv, res

    def _garch_forecast_sigma(res, h):
        fc      = res.forecast(horizon=h, reindex=False)
        var_row = fc.variance.values[-1, :h]
        return np.sqrt(np.mean(var_row) * 252) / 100

    def run_garch_two_step_is(ret_series, rv_fwd_series, text_data=None, h=22):
        ret_pct = pd.to_numeric(ret_series, errors='coerce') * 100
        base = pd.concat({
            'ret': ret_pct,
            'rv_fwd': pd.to_numeric(rv_fwd_series, errors='coerce'),
        }, axis=1)
        text_frame = _coerce_text_frame(text_data, base.index)
        text_cols = list(text_frame.columns)
        comb = pd.concat([base, text_frame], axis=1).dropna(
            subset=['ret', 'rv_fwd'] + text_cols)
        if len(comb) < 100:
            return np.nan, np.nan, np.nan, np.nan, np.nan, 0, None, None
        try:
            garch_rv_s, res_g = _fit_garch_get_sigma(comb['ret'])
            comb2 = pd.concat(
                [comb['rv_fwd'], garch_rv_s.rename('garch_rv')]
                + [comb[c] for c in text_cols], axis=1).dropna()
            if len(comb2) < 50:
                return np.nan, np.nan, np.nan, np.nan, np.nan, 0, None, None
            xcols = ['garch_rv'] + [
                c for c in text_cols if comb2[c].notna().sum() > 50]
            res_ols = sm.OLS(
                comb2['rv_fwd'],
                sm.add_constant(comb2[xcols], has_constant='add')
            ).fit(cov_type='HAC', cov_kwds={'maxlags': _har_hac_lags(h)})
            is_r2   = res_ols.rsquared
            is_rmse = np.sqrt(res_ols.mse_resid)
            primary = text_cols[0] if text_cols else None
            tc = float(res_ols.params[primary]) if primary in res_ols.params.index else np.nan
            tp = float(res_ols.pvalues[primary]) if primary in res_ols.pvalues.index else np.nan
            return is_r2, is_rmse, tc, tp, res_g.bic, len(comb2), res_g, res_ols
        except Exception as e:
            print(f'    Échec GARCH IS : {e}')
            return np.nan, np.nan, np.nan, np.nan, np.nan, 0, None, None

    records_garch_gkg  = []
    records_garch_text = []
    records_garch_gkg_rolling  = []
    records_garch_text_rolling = []
    records_garch_native = []
    records_garch_2ndstep = []
    garch_coef_path_records = []
    pred_records_garch = []
    pred_records_garch_rolling = []

    _GARCH_ACCUMULATOR_NAMES = [
        'records_garch_gkg', 'records_garch_text',
        'records_garch_gkg_rolling', 'records_garch_text_rolling',
        'records_garch_native', 'records_garch_2ndstep',
        'garch_coef_path_records', 'pred_records_garch',
        'pred_records_garch_rolling',
    ]
    _GARCH_GLOBAL_ACCUMULATORS = {
        'records_garch_gkg': records_garch_gkg,
        'records_garch_text': records_garch_text,
        'records_garch_gkg_rolling': records_garch_gkg_rolling,
        'records_garch_text_rolling': records_garch_text_rolling,
        'records_garch_native': records_garch_native,
        'records_garch_2ndstep': records_garch_2ndstep,
        'garch_coef_path_records': garch_coef_path_records,
        'pred_records_garch': pred_records_garch,
        'pred_records_garch_rolling': pred_records_garch_rolling,
    }

    def _new_garch_block():
        return {name: [] for name in _GARCH_ACCUMULATOR_NAMES}

    def _save_garch_state(key, signature, completed_units, block):
        _save_model_cache('GARCH', key, signature, {
            'completed_units': sorted(completed_units),
            'block': block,
        })

    for ticker in TICKERS:
        if f'ret_{ticker}' not in df_raw.columns:
            continue
        ret_s = df_raw[f'ret_{ticker}']

        for h in H_LIST:
            y_col = f'rv_fwd_h{h}_{ticker}'
            if y_col not in df_raw.columns:
                continue
            rv_fwd_s = df_raw[y_col]

            _garch_key = f'garch_full_matrix_{ticker}_h{h}'
            _garch_feature_spec = {
                'gkg_routes': GKG_ROUTE_FEATS,
                'text_routes': TEXT_ROUTE_FEATS_FOR_GARCH,
                'expanding': True,
                'rolling': True,
                'oos_step': GARCH_OOS_STEP,
                'input_cols': (['date', f'ret_{ticker}', y_col]
                               + [f for fs in GKG_ROUTE_FEATS.values() for f in fs]
                               + [f for fs in TEXT_ROUTE_FEATS_FOR_GARCH.values() for f in fs]),
            }
            _garch_sig = _model_cache_signature(
                'GARCH', ticker, h, 'full_matrix', _garch_feature_spec)
            _cached_state = _load_model_cache(
                'GARCH', _garch_key, _garch_sig,
                force=FORCE_RERUN_GARCH_MODELS)

            if _cached_state is None:
                _completed = set()
                _block = _new_garch_block()
            else:
                _completed = set(_cached_state.get('completed_units', []))
                _block = _cached_state.get('block', _new_garch_block())
                for _name in _GARCH_ACCUMULATOR_NAMES:
                    _block.setdefault(_name, [])
                print(f'  ↺ GARCH {ticker} h={h} : {len(_completed)} unité(s) restaurée(s)')

            # ---------- Benchmark ----------
            _unit = 'benchmark'
            if _unit not in _completed:
                is_r2_b, is_rmse_b, _, _, bic_b, nobs_b, res_g_b, res_ols_b = run_garch_two_step_is(
                    ret_s, rv_fwd_s, text_data=None, h=h)
                _block['records_garch_native'] += extract_garch_coefs(res_g_b, ticker, h)
                if res_ols_b is not None:
                    _block['records_garch_2ndstep'] += extract_all_coefs(
                        res_ols_b, ticker, h, 'Spec1', 'GARCH', route='Benchmark')

                _sink_b_exp = [] if h in COEF_TREND_HORIZONS else None
                _sink_b_roll = [] if h in COEF_TREND_HORIZONS else None
                oos_r2_b, oos_rmse_b, oos_qlike_b, n_oos_b, db, ab, pb = garch_two_step_oos(
                    ret_s, rv_fwd_s, text_data=None, h=h,
                    step=GARCH_OOS_STEP, window_type='expanding',
                    coef_sink=_sink_b_exp, return_series=True)
                oos_r2_b_roll, oos_rmse_b_roll, oos_qlike_b_roll, n_oos_b_roll, dbr, abr, pbr = garch_two_step_oos(
                    ret_s, rv_fwd_s, text_data=None, h=h,
                    step=GARCH_OOS_STEP, window_type='rolling',
                    trailing_window=ROLLING_WINDOW,
                    coef_sink=_sink_b_roll, return_series=True)

                for sink, wt in [(_sink_b_exp, 'expanding'), (_sink_b_roll, 'rolling')]:
                    if sink is not None:
                        for rec in sink:
                            rec.update({
                                'ticker': ticker, 'h': h, 'route': 'Benchmark',
                                'spec': 'Spec1' if wt == 'expanding' else 'Spec1-Rolling',
                            })
                        _block['garch_coef_path_records'].extend(sink)

                for d, a, p_ in zip(db, ab, pb):
                    _block['pred_records_garch'].append({
                        'date': d, 'ticker': ticker, 'h': h,
                        'model': 'GARCH', 'route': 'Benchmark', 'spec': 'Spec1',
                        'actual': a, 'pred': p_,
                    })
                for d, a, p_ in zip(dbr, abr, pbr):
                    _block['pred_records_garch_rolling'].append({
                        'date': d, 'ticker': ticker, 'h': h,
                        'model': 'GARCH', 'route': 'Benchmark', 'spec': 'Spec1-Rolling',
                        'actual': a, 'pred': p_,
                    })

                bench_g = {
                    'model': 'GARCH', 'route': 'Benchmark', 'spec': 'Spec1',
                    'ticker': ticker, 'h': h,
                    'IS_R2': is_r2_b, 'IS_RMSE': is_rmse_b,
                    'OOS_R2': oos_r2_b, 'OOS_RMSE': oos_rmse_b,
                    'OOS_QLIKE': oos_qlike_b, 'BIC': bic_b,
                    'n_obs': nobs_b, 'n_oos': n_oos_b,
                    'text_coef': np.nan, 'text_pval': np.nan,
                }
                _block['records_garch_gkg'].append(bench_g.copy())
                _block['records_garch_text'].append(bench_g.copy())

                bench_g_roll = {
                    'model': 'GARCH', 'route': 'Benchmark', 'spec': 'Spec1-Rolling',
                    'ticker': ticker, 'h': h,
                    'OOS_R2': oos_r2_b_roll, 'OOS_RMSE': oos_rmse_b_roll,
                    'OOS_QLIKE': oos_qlike_b_roll, 'n_oos': n_oos_b_roll,
                    'window_type': 'rolling', 'trailing_window': ROLLING_WINDOW,
                }
                _block['records_garch_gkg_rolling'].append(bench_g_roll.copy())
                _block['records_garch_text_rolling'].append(bench_g_roll.copy())

                _completed.add(_unit)
                _save_garch_state(_garch_key, _garch_sig, _completed, _block)
            else:
                print(f'    ↺ {ticker} h={h} Benchmark déjà calculé')

            # ---------- GKG exploratoire ----------
            for route, feats in GKG_ROUTE_FEATS.items():
                _unit = f'gkg::{route}'
                if _unit in _completed:
                    print(f'    ↺ {ticker} h={h} GKG/{route} déjà calculé')
                    continue
                prim = GKG_ROUTE_PRIMARY[route]
                if prim not in df_raw.columns or df_raw[prim].notna().sum() < 100:
                    _completed.add(_unit)
                    _save_garch_state(_garch_key, _garch_sig, _completed, _block)
                    continue
                text_s = df_raw[prim].ffill().fillna(0)

                is_r2, is_rmse, tc, tp, bic, nobs, _, _ = run_garch_two_step_is(
                    ret_s, rv_fwd_s, text_data=text_s, h=h)
                oos_r2, oos_rmse, oos_qlike, n_oos = garch_two_step_oos(
                    ret_s, rv_fwd_s, text_data=text_s, h=h,
                    step=GARCH_OOS_STEP, window_type='expanding')
                oos_r2_roll, oos_rmse_roll, oos_qlike_roll, n_oos_roll = garch_two_step_oos(
                    ret_s, rv_fwd_s, text_data=text_s, h=h,
                    step=GARCH_OOS_STEP, window_type='rolling',
                    trailing_window=ROLLING_WINDOW)

                _block['records_garch_gkg'].append({
                    'model': 'GARCH-GKG', 'route': route, 'spec': 'SpecG',
                    'ticker': ticker, 'h': h,
                    'IS_R2': is_r2, 'IS_RMSE': is_rmse,
                    'OOS_R2': oos_r2, 'OOS_RMSE': oos_rmse,
                    'OOS_QLIKE': oos_qlike, 'BIC': bic,
                    'n_obs': nobs, 'n_oos': n_oos,
                    'text_coef': tc, 'text_pval': tp,
                })
                _block['records_garch_gkg_rolling'].append({
                    'model': 'GARCH-GKG', 'route': route, 'spec': 'SpecG-Rolling',
                    'ticker': ticker, 'h': h,
                    'OOS_R2': oos_r2_roll, 'OOS_RMSE': oos_rmse_roll,
                    'OOS_QLIKE': oos_qlike_roll, 'n_oos': n_oos_roll,
                    'window_type': 'rolling', 'trailing_window': ROLLING_WINDOW,
                })

                _completed.add(_unit)
                _save_garch_state(_garch_key, _garch_sig, _completed, _block)

            # ---------- Routes textuelles principales : mêmes features que HAR ----------
            for route, feats in TEXT_ROUTE_FEATS_FOR_GARCH.items():
                _unit = f'text::{route}'
                if _unit in _completed:
                    print(f'    ↺ {ticker} h={h} route {route} déjà calculée')
                    continue
                if route == 'FINBERT' and not ENABLE_FINBERT:
                    _completed.add(_unit)
                    _save_garch_state(_garch_key, _garch_sig, _completed, _block)
                    continue

                avail = [
                    f for f in feats
                    if f in df_raw.columns and df_raw[f].notna().sum() > 100
                ]
                if not avail:
                    _completed.add(_unit)
                    _save_garch_state(_garch_key, _garch_sig, _completed, _block)
                    continue
                text_route_df = df_raw[avail].copy()

                is_r2, is_rmse, tc, tp, bic, nobs, _, res_ols_r = run_garch_two_step_is(
                    ret_s, rv_fwd_s, text_data=text_route_df, h=h)
                if res_ols_r is not None:
                    _block['records_garch_2ndstep'] += extract_all_coefs(
                        res_ols_r, ticker, h, 'Spec2', 'GARCH-X', route=route)

                _sink_r_exp = [] if h in COEF_TREND_HORIZONS else None
                _sink_r_roll = [] if h in COEF_TREND_HORIZONS else None
                oos_r2, oos_rmse, oos_qlike, n_oos, dg, ag, pg = garch_two_step_oos(
                    ret_s, rv_fwd_s, text_data=text_route_df, h=h,
                    step=GARCH_OOS_STEP, window_type='expanding',
                    coef_sink=_sink_r_exp, return_series=True)
                oos_r2_roll, oos_rmse_roll, oos_qlike_roll, n_oos_roll, dgr, agr, pgr = garch_two_step_oos(
                    ret_s, rv_fwd_s, text_data=text_route_df, h=h,
                    step=GARCH_OOS_STEP, window_type='rolling',
                    trailing_window=ROLLING_WINDOW,
                    coef_sink=_sink_r_roll, return_series=True)

                for sink, wt in [(_sink_r_exp, 'expanding'), (_sink_r_roll, 'rolling')]:
                    if sink is not None:
                        for rec in sink:
                            rec.update({
                                'ticker': ticker, 'h': h, 'route': route,
                                'spec': 'Spec2' if wt == 'expanding' else 'Spec2-Rolling',
                            })
                        _block['garch_coef_path_records'].extend(sink)

                for d, a, p_ in zip(dg, ag, pg):
                    _block['pred_records_garch'].append({
                        'date': d, 'ticker': ticker, 'h': h,
                        'model': 'GARCH-X', 'route': route, 'spec': 'Spec2',
                        'actual': a, 'pred': p_,
                    })
                for d, a, p_ in zip(dgr, agr, pgr):
                    _block['pred_records_garch_rolling'].append({
                        'date': d, 'ticker': ticker, 'h': h,
                        'model': 'GARCH-X', 'route': route, 'spec': 'Spec2-Rolling',
                        'actual': a, 'pred': p_,
                    })

                _block['records_garch_text'].append({
                    'model': 'GARCH-X', 'route': route, 'spec': 'Spec2',
                    'ticker': ticker, 'h': h,
                    'IS_R2': is_r2, 'IS_RMSE': is_rmse,
                    'OOS_R2': oos_r2, 'OOS_RMSE': oos_rmse,
                    'OOS_QLIKE': oos_qlike, 'BIC': bic,
                    'n_obs': nobs, 'n_oos': n_oos,
                    'text_coef': tc, 'text_pval': tp,
                })
                _block['records_garch_text_rolling'].append({
                    'model': 'GARCH-X', 'route': route, 'spec': 'Spec2-Rolling',
                    'ticker': ticker, 'h': h,
                    'OOS_R2': oos_r2_roll, 'OOS_RMSE': oos_rmse_roll,
                    'OOS_QLIKE': oos_qlike_roll, 'n_oos': n_oos_roll,
                    'window_type': 'rolling', 'trailing_window': ROLLING_WINDOW,
                })

                _completed.add(_unit)
                _save_garch_state(_garch_key, _garch_sig, _completed, _block)

            # Le bloc local n'est ajouté aux résultats globaux qu'une seule fois.
            for _name, _global_values in _GARCH_GLOBAL_ACCUMULATORS.items():
                _global_values.extend(_block.get(_name, []))

            print(f'  GARCH {ticker} h={h} terminé | unités={len(_completed)} | pas OOS={GARCH_OOS_STEP}')

    df_garch_gkg = pd.DataFrame(records_garch_gkg)
    df_garch     = pd.DataFrame(records_garch_text)
    df_garch_gkg_rolling = pd.DataFrame(records_garch_gkg_rolling)
    df_garch_rolling     = pd.DataFrame(records_garch_text_rolling)
    df_garch_native   = pd.DataFrame(records_garch_native)
    df_garch_2ndstep  = pd.DataFrame(records_garch_2ndstep)
    df_garch_coef_paths = pd.DataFrame(garch_coef_path_records)
    df_oos_preds_garch = pd.DataFrame(pred_records_garch)
    df_oos_preds_garch_rolling = pd.DataFrame(pred_records_garch_rolling)
    print(f'→ GARCH terminé en {time.time()-t0:.1f}s | expanding={len(df_garch)} lignes | rolling={len(df_garch_rolling)} lignes')
else:
    print('\n[11] ENABLE_GARCH=False ; GARCH ignoré.')
    df_garch_gkg = pd.DataFrame()
    df_garch     = pd.DataFrame()
    df_garch_gkg_rolling = pd.DataFrame()
    df_garch_rolling     = pd.DataFrame()
    df_garch_native  = pd.DataFrame()
    df_garch_2ndstep = pd.DataFrame()
    df_garch_coef_paths = pd.DataFrame()
    df_oos_preds_garch = pd.DataFrame()
    df_oos_preds_garch_rolling = pd.DataFrame()

# %% ── 11.5 Synthèse des résultats GARCH-X (expanding + rolling) ─────────────────
def _route_pivot(df, h, spec, value, include_benchmark=False):
    sub = df[(df['h'] == h) & (df['spec'] == spec)].copy()
    if not include_benchmark:
        sub = sub[sub['route'] != 'Benchmark']
    if sub.empty:
        return pd.DataFrame()
    return sub.pivot_table(index='route', columns='ticker', values=value, aggfunc='first').round(6)


def _rmse_improvement(df, h, route_spec, benchmark_spec):
    route_rmse = _route_pivot(df, h, route_spec, 'OOS_RMSE', include_benchmark=False)
    if route_rmse.empty:
        return route_rmse
    bench = (df[(df['h'] == h) & (df['spec'] == benchmark_spec) & (df['route'] == 'Benchmark')]
             .groupby('ticker')['OOS_RMSE'].first())
    out = route_rmse.copy()
    for ticker in out.columns:
        b = bench.get(ticker, np.nan)
        out[ticker] = (b - route_rmse[ticker]) / b * 100 if pd.notna(b) and b != 0 else np.nan
    return out


def _performance_with_benchmark(df, h, route_spec, benchmark_spec, value):
    route = _route_pivot(df, h, route_spec, value, include_benchmark=False)
    bench = _route_pivot(df, h, benchmark_spec, value, include_benchmark=True)
    if bench.empty:
        return route
    bench = bench.loc[[idx for idx in bench.index if idx == 'Benchmark']]
    return pd.concat([bench, route], axis=0)


garch_tables_by_h = {}
if not df_garch.empty:
    print('\n' + '█'*65)
    print('█  Synthèse GARCH-X : expanding et rolling')
    print('█'*65)
    for h_show in H_LIST:
        print('\n' + '='*65)
        print(f'  GARCH-X h={h_show}')
        print('='*65)
        tables_h = {}
        for label, frame, route_spec, bench_spec in [
            ('Expanding', df_garch, 'Spec2', 'Spec1'),
            ('Rolling', df_garch_rolling, 'Spec2-Rolling', 'Spec1-Rolling'),
        ]:
            if frame.empty:
                continue
            for metric, pretty in [('OOS_R2', 'R² OOS'), ('OOS_RMSE', 'RMSE OOS'), ('OOS_QLIKE', 'QLIKE OOS')]:
                tab = _performance_with_benchmark(frame, h_show, route_spec, bench_spec, metric)
                print(f'\n{label} {pretty}' + (' (plus faible = meilleur)' if metric != 'OOS_R2' else ''))
                print(tab.round(6).to_string())
                tables_h[f'{label}_{metric}'] = tab
            improvement = _rmse_improvement(frame, h_show, route_spec, bench_spec)
            print(f'\n{label} amélioration du RMSE vs benchmark de la même fenêtre, % (positif = meilleur)')
            print(improvement.round(2).to_string())
            tables_h[f'{label}_RMSE_improv'] = improvement

        if all(k in tables_h for k in ['Expanding_OOS_RMSE', 'Rolling_OOS_RMSE']):
            common = tables_h['Rolling_OOS_RMSE'].index.intersection(tables_h['Expanding_OOS_RMSE'].index)
            delta = tables_h['Rolling_OOS_RMSE'].loc[common] - tables_h['Expanding_OOS_RMSE'].loc[common]
            print('\nΔRMSE = rolling - expanding (négatif = rolling meilleur)')
            print(delta.round(6).to_string())
            tables_h['Delta_RMSE_Rolling_minus_Expanding'] = delta
        garch_tables_by_h[h_show] = tables_h
else:
    garch_tables_by_h = {}


# %% ── 12. Régressions de panel# %% ── 12. Régressions de panel (expérience principale : routes purement textuelles, FWD_SENT incluse automatiquement) ────────────────
print('\n[12] Régressions de panel (effets fixes d’actif + erreurs-types de Driscoll-Kraay) (calcul en cours...)')
t0 = time.time()
records_p = []

for h in H_LIST:
    panels = []
    for ticker in TICKERS:
        y_col = f'rv_fwd_h{h}_{ticker}'
        if y_col not in df_raw.columns:
            continue
        base  = har_base(ticker)
        macro_cols_t = macro_controls(ticker)
        all_text = [f for feats in TEXT_ROUTE_FEATS.values() for f in feats]
        need = ['date', y_col] + base + macro_cols_t + all_text
        d = df_raw[[c for c in need if c in df_raw.columns]].copy()
        rename_map = {
            y_col:                    'rv_fwd',
            f'rv_d_{ticker}': 'rv_d',
            f'rv_w_{ticker}': 'rv_w',
            f'rv_m_{ticker}': 'rv_m',
        }
        if f'rv_spillover_{ticker}' in d.columns:
            rename_map[f'rv_spillover_{ticker}'] = 'rv_spillover'
        if f'good_{ticker}' in d.columns:
            rename_map[f'good_{ticker}'] = 'good_ret'
        if f'bad_{ticker}' in d.columns:
            rename_map[f'bad_{ticker}'] = 'bad_ret'
        d = d.rename(columns=rename_map)
        d['ticker'] = ticker
        panels.append(d)

    panel_long = (pd.concat(panels, ignore_index=True)
                  .dropna(subset=['rv_fwd','rv_d','rv_w','rv_m'])
                  .set_index(['ticker','date']))

    for route, feats in TEXT_ROUTE_FEATS.items():
        if route == 'FINBERT' and not ENABLE_FINBERT: continue
        avail = [f for f in feats
                 if f in panel_long.columns and panel_long[f].notna().sum() > 100]
        if not avail: continue

        feat_str = ' + '.join(avail)
        formula  = f'rv_fwd ~ rv_d + rv_w + rv_m + {feat_str} + EntityEffects'
        try:
            sub  = panel_long[['rv_fwd','rv_d','rv_w','rv_m']+avail].dropna()
            pres = PanelOLS.from_formula(formula, data=sub).fit(
                cov_type='kernel', kernel='bartlett', bandwidth=_har_hac_lags(h))
            for feat in avail:
                if feat in pres.params.index:
                    records_p.append({
                        'route': route, 'h': h, 'feature': feat,
                        'coef': pres.params[feat],
                        'tstat': pres.tstats[feat],
                        'pval': pres.pvalues[feat],
                        'within_R2': pres.rsquared_within,
                        'n_obs': int(pres.nobs),
                    })
        except Exception as e:
            print(f'  Échec du panel {route} h={h} : {e}')

    if 'rv_spillover' in panel_long.columns:
        try:
            extra_cols = [c for c in ['rv_spillover', 'good_ret', 'bad_ret']
                          if c in panel_long.columns and panel_long[c].notna().sum() > 100]
            if extra_cols:
                sub_extra = panel_long[['rv_fwd','rv_d','rv_w','rv_m']+extra_cols].dropna()
                formula_extra = f'rv_fwd ~ rv_d + rv_w + rv_m + {" + ".join(extra_cols)} + EntityEffects'
                pres_extra = PanelOLS.from_formula(formula_extra, data=sub_extra).fit(
                    cov_type='kernel', kernel='bartlett', bandwidth=_har_hac_lags(h))
                for feat in extra_cols:
                    if feat in pres_extra.params.index:
                        records_p.append({
                            'route': 'SPILLOVER_GOODBAD', 'h': h, 'feature': feat,
                            'coef': pres_extra.params[feat],
                            'tstat': pres_extra.tstats[feat],
                            'pval': pres_extra.pvalues[feat],
                            'within_R2': pres_extra.rsquared_within,
                            'n_obs': int(pres_extra.nobs),
                        })
        except Exception as e:
            print(f'  Échec du panel (spillover/rendements positifs-négatifs) h={h} : {e}')

df_panel = pd.DataFrame(records_p)
print(f'→ Panel terminé : {len(df_panel)} lignes de résultats, temps écoulé {time.time()-t0:.1f}s')

# %% ── 13. Synthèse HAR-RV-X : expanding + rolling ─────────────────────────
def _har_performance_table(h, metric, window_type='expanding'):
    if window_type == 'expanding':
        frame, route_spec, bench_spec = df_har, 'Spec2', 'Spec1'
    else:
        frame, route_spec, bench_spec = df_har_rolling, 'Spec2-Rolling', 'Spec1-Rolling'
    return _performance_with_benchmark(frame, h, route_spec, bench_spec, metric)


print('\n' + '█'*65)
print('█  Synthèse HAR-RV-X : expanding et rolling')
print('█'*65)
main_tables_by_h = {}
for h_show in H_LIST:
    print('\n' + '='*65)
    print(f'  HAR-RV-X h={h_show}')
    print('='*65)
    tables_h = {}
    for label, wt in [('Expanding', 'expanding'), ('Rolling', 'rolling')]:
        frame = df_har if wt == 'expanding' else df_har_rolling
        route_spec = 'Spec2' if wt == 'expanding' else 'Spec2-Rolling'
        bench_spec = 'Spec1' if wt == 'expanding' else 'Spec1-Rolling'
        if frame.empty:
            continue
        for metric, pretty in [('OOS_R2', 'R² OOS'), ('OOS_RMSE', 'RMSE OOS'), ('OOS_QLIKE', 'QLIKE OOS')]:
            tab = _performance_with_benchmark(frame, h_show, route_spec, bench_spec, metric)
            print(f'\n{label} {pretty}' + (' (plus faible = meilleur)' if metric != 'OOS_R2' else ''))
            print(tab.round(6).to_string())
            tables_h[f'{label}_{metric}'] = tab
        improvement = _rmse_improvement(frame, h_show, route_spec, bench_spec)
        print(f'\n{label} amélioration du RMSE vs benchmark de la même fenêtre, % (positif = meilleur)')
        print(improvement.round(2).to_string())
        tables_h[f'{label}_RMSE_improv'] = improvement

    if all(k in tables_h for k in ['Expanding_OOS_RMSE', 'Rolling_OOS_RMSE']):
        common = tables_h['Rolling_OOS_RMSE'].index.intersection(tables_h['Expanding_OOS_RMSE'].index)
        delta_rmse = tables_h['Rolling_OOS_RMSE'].loc[common] - tables_h['Expanding_OOS_RMSE'].loc[common]
        delta_r2 = tables_h['Rolling_OOS_R2'].loc[common] - tables_h['Expanding_OOS_R2'].loc[common]
        delta_qlike = tables_h['Rolling_OOS_QLIKE'].loc[common] - tables_h['Expanding_OOS_QLIKE'].loc[common]
        print('\nΔRMSE = rolling - expanding (négatif = rolling meilleur)')
        print(delta_rmse.round(6).to_string())
        tables_h['Delta_RMSE_Rolling_minus_Expanding'] = delta_rmse
        tables_h['Delta_R2_Rolling_minus_Expanding'] = delta_r2
        tables_h['Delta_QLIKE_Rolling_minus_Expanding'] = delta_qlike

    # Compatibilité avec les noms historiques utilisés en aval.
    if 'Expanding_OOS_R2' in tables_h:
        tables_h['OOS_R2'] = tables_h['Expanding_OOS_R2'].drop(index='Benchmark', errors='ignore')
        tables_h['OOS_RMSE'] = tables_h['Expanding_OOS_RMSE'].drop(index='Benchmark', errors='ignore')
        tables_h['OOS_QLIKE'] = tables_h['Expanding_OOS_QLIKE'].drop(index='Benchmark', errors='ignore')
        tables_h['RMSE_improv'] = tables_h['Expanding_RMSE_improv']
        is_r2 = _route_pivot(df_har, h_show, 'Spec2', 'IS_R2')
        is_rmse = _route_pivot(df_har, h_show, 'Spec2', 'IS_RMSE')
        tables_h['IS_R2'] = is_r2
        tables_h['IS_RMSE'] = is_rmse
        tables_h['gap'] = is_r2 - tables_h['OOS_R2']
    main_tables_by_h[h_show] = tables_h

    _tx = df_har[(df_har['h'] == h_show) & (df_har['spec'] == 'Spec2') & (df_har['route'] != 'Benchmark')].copy()
    if not _tx.empty:
        _tx['sig'] = _tx['text_pval'].apply(fmt_stars)
        print(f'\nCoefficients textuels in-sample (h={h_show}, Spec2, HAC) :')
        print(_tx[['route','ticker','text_coef','text_pval','sig','IS_R2','OOS_R2']]
              .sort_values(['route','ticker']).to_string(index=False))

main_is = main_tables_by_h[22].get('IS_R2', pd.DataFrame())
main_oos = main_tables_by_h[22].get('OOS_R2', pd.DataFrame())
main_gap = main_tables_by_h[22].get('gap', pd.DataFrame())
main_is_rmse = main_tables_by_h[22].get('IS_RMSE', pd.DataFrame())
main_oos_rmse = main_tables_by_h[22].get('OOS_RMSE', pd.DataFrame())
main_rmse_improv = main_tables_by_h[22].get('RMSE_improv', pd.DataFrame())


# %% ── 13.5# %% ── 13.5 [ajout v10, extension v11, correction v12] Test de significativité de la précision prédictive OOS (Clark-West, 2007) ──
# Jusqu’ici, la correction FDR ne concernait que les coefficients textuels **en échantillon interne** de Spec2, alors que les différences de R² OOS
# n’étaient accompagnées d’aucun test de significativité. OOS_R2, OOS_RMSE et OOS_QLIKE n’étaient que des estimations ponctuelles,
# sans indication sur la fiabilité statistique de l’écart ; cette section complète cette lacune.
#
# Le test usuel de Diebold-Mariano n’est pas appliqué directement, car il suppose des modèles comparés non imbriqués.
# Ici, Spec2 (HAR + texte) contient Spec1 (HAR de référence) comme cas particulier
# lorsque tous les coefficients textuels sont nuls. Pour des modèles imbriqués, la distribution asymptotique du test DM standard
# n’est plus normale standard ; même lorsque le texte ne contient aucune information, la MSE OOS attendue de Spec2
# peut être légèrement plus élevée en raison des paramètres supplémentaires estimés, ce qui biaise systématiquement le test DM
# contre Spec2. Clark et West (2007) proposent un terme d’ajustement spécifiquement conçu pour les modèles imbriqués,
# et ce test constitue l’outil standard pour comparer « benchmark » à « benchmark + régresseurs supplémentaires » dans ce contexte.
#
# [ajout v11] En plus du test CW historique en fenêtre expanding sur l’ensemble de l’échantillon OOS, trois dimensions sont ajoutées :
#
#   (a) une version rolling des prévisions OOS avec trailing_window=ROLLING_WINDOW (Spec1-Rolling/Spec2-Rolling),
#       suivie du même test CW ; les trajectoires de coefficients rolling et la significativité CW rolling reposent ainsi sur
#       exactement la même conception de fenêtre et peuvent se corroborer ;
#   (b) un test CW par segments de stabilité, conservant le découpage mécanique première/seconde moitié uniquement comme diagnostic temporel général ;
#   (c) un test CW à dates fixes sur les périodes Pre-COVID/COVID/Post-COVID ; toute interprétation liée à la pandémie
#       repose exclusivement sur ces sous-périodes prédéfinies. Les prévisions expanding et rolling sont toutes deux évaluées.
print('\n' + '='*65)
print('Test de précision prédictive OOS (Clark-West, adapté aux modèles imbriqués : Spec2 vs benchmark Spec1)')
print('='*65)

from scipy.stats import norm

def _cw_stat_pval(f_t, h, model_type='HAR'):
    """Calcule Clark-West avec une bande HAC adaptée à la conception OOS du modèle."""
    n = len(f_t)
    if n < 10:
        return np.nan, np.nan, np.nan
    L = _oos_hac_lags(h, model_type=model_type)
    fbar = f_t.mean()
    fc = f_t - fbar
    gamma0 = np.mean(fc ** 2)
    var = gamma0
    for lag in range(1, min(L, n - 1) + 1):
        w = 1 - lag / (L + 1)
        cov = np.mean(fc[lag:] * fc[:-lag])
        var += 2 * w * cov
    se = np.sqrt(max(var, 1e-16) / n)
    if se <= 0 or np.isnan(se):
        return np.nan, np.nan, np.nan
    cw_stat = fbar / se
    pval_two_sided = 2 * (1 - norm.cdf(abs(cw_stat)))
    pval_one_sided = 1 - norm.cdf(cw_stat)   # H1 : le modèle textuel est significativement meilleur ; cette colonne entre dans la FDR
    return cw_stat, pval_one_sided, pval_two_sided


def _merge_bench_route(dates_bench, act_bench, pred_bench,
                       dates_route, act_route, pred_route):
    """Aligne par date les prévisions du benchmark et de la route par jointure interne, afin d’éviter qu’une différence de dates OOS
    due aux valeurs manquantes des variables n’apparie incorrectement des jours non comparables."""
    df_b = pd.DataFrame({'date': dates_bench, 'act_b': act_bench, 'pred_b': pred_bench})
    df_r = pd.DataFrame({'date': dates_route, 'act_r': act_route, 'pred_r': pred_route})
    return df_b.merge(df_r, on='date', how='inner').sort_values('date').reset_index(drop=True)


def clark_west_test(dates_bench, act_bench, pred_bench,
                     dates_route, act_route, pred_route, h, model_type='HAR'):
    """
    Test de Clark-West (2007) pour modèles imbriqués, synthétisé une fois sur l’ensemble de l’échantillon OOS. Le terme ajusté est :
        f_t = e_bench_t^2 - [ e_route_t^2 - (pred_bench_t - pred_route_t)^2 ]
    Sous H0 d’égalité de précision prédictive OOS, la moyenne de f divisée par son erreur-type HAC converge vers N(0,1).
    """
    m = _merge_bench_route(dates_bench, act_bench, pred_bench,
                           dates_route, act_route, pred_route)
    n = len(m)
    if n < 30:
        return np.nan, np.nan, np.nan, n
    e_b = m['act_b'].values - m['pred_b'].values
    e_r = m['act_r'].values - m['pred_r'].values
    f_t = e_b**2 - (e_r**2 - (m['pred_b'].values - m['pred_r'].values)**2)
    cw_stat, pval_1s, pval_2s = _cw_stat_pval(f_t, h, model_type=model_type)
    return cw_stat, pval_1s, pval_2s, n


def clark_west_test_segmented(dates_bench, act_bench, pred_bench,
                               dates_route, act_route, pred_route, h, n_segments=2,
                               model_type='HAR'):
    """
    [ajout v11] Divise chronologiquement la période OOS en n_segments parts égales (2 par défaut : première/seconde moitié),
    puis applique un test CW dans chaque segment uniquement comme diagnostic général de stabilité temporelle. Si une moitié est significative
    et l’autre non, la relation n’est pas stable sur toute la période ; toute interprétation COVID doit utiliser les sous-périodes à dates fixes.
    """
    m = _merge_bench_route(dates_bench, act_bench, pred_bench,
                           dates_route, act_route, pred_route)
    n = len(m)
    if n < n_segments * 30:
        return []
    bounds = np.linspace(0, n, n_segments + 1).astype(int)
    out = []
    for seg in range(n_segments):
        lo, hi = bounds[seg], bounds[seg + 1]
        seg_m = m.iloc[lo:hi]
        if len(seg_m) < 30:
            out.append({'segment': seg + 1, 'start_date': None, 'end_date': None,
                        'cw_stat': np.nan, 'pval_onesided': np.nan, 'n': len(seg_m)})
            continue
        e_b = seg_m['act_b'].values - seg_m['pred_b'].values
        e_r = seg_m['act_r'].values - seg_m['pred_r'].values
        f_t = e_b**2 - (e_r**2 - (seg_m['pred_b'].values - seg_m['pred_r'].values)**2)
        cw_stat, pval_1s, _ = _cw_stat_pval(f_t, h, model_type=model_type)
        out.append({
            'segment': seg + 1,
            'start_date': seg_m['date'].iloc[0], 'end_date': seg_m['date'].iloc[-1],
            'cw_stat': cw_stat, 'pval_onesided': pval_1s, 'n': len(seg_m),
        })
    return out


def clark_west_test_date_segments(dates_bench, act_bench, pred_bench,
                                   dates_route, act_route, pred_route, h,
                                   date_segments=CW_DATE_SEGMENTS, model_type='HAR'):
    """Applique le test CW sur des intervalles de dates prédéfinis, destinés à l’interprétation Pre-COVID/COVID/Post-COVID."""
    m = _merge_bench_route(dates_bench, act_bench, pred_bench,
                           dates_route, act_route, pred_route)
    if m.empty:
        return []
    m['date'] = pd.to_datetime(m['date'])
    out = []
    for period, start_date, end_date in date_segments:
        mask = pd.Series(True, index=m.index)
        if start_date is not None:
            mask &= m['date'] >= pd.Timestamp(start_date)
        if end_date is not None:
            mask &= m['date'] <= pd.Timestamp(end_date)
        seg_m = m.loc[mask].copy()
        if len(seg_m) < 30:
            out.append({
                'period': period,
                'defined_start': start_date,
                'defined_end': end_date,
                'start_date': seg_m['date'].min() if len(seg_m) else None,
                'end_date': seg_m['date'].max() if len(seg_m) else None,
                'cw_stat': np.nan, 'pval_onesided': np.nan, 'n': len(seg_m),
            })
            continue
        e_b = seg_m['act_b'].values - seg_m['pred_b'].values
        e_r = seg_m['act_r'].values - seg_m['pred_r'].values
        f_t = e_b**2 - (e_r**2 - (
            seg_m['pred_b'].values - seg_m['pred_r'].values) ** 2)
        cw_stat, pval_1s, _ = _cw_stat_pval(f_t, h, model_type=model_type)
        out.append({
            'period': period,
            'defined_start': start_date,
            'defined_end': end_date,
            'start_date': seg_m['date'].iloc[0],
            'end_date': seg_m['date'].iloc[-1],
            'cw_stat': cw_stat, 'pval_onesided': pval_1s, 'n': len(seg_m),
        })
    return out


def run_cw_battery(df_preds, spec_bench, spec_route, label, out_suffix,
                   model_bench='HAR-RV', model_type='HAR'):
    """
    Pour un ensemble de prévisions donné, parcourt toutes les combinaisons route×ticker×h : test CW sur l’ensemble OOS,
    tests de stabilité première/seconde moitié et tests aux dates fixes Pre-COVID/COVID/Post-COVID ; chaque famille est corrigée séparément par BH-FDR.
    spec_bench/spec_route servent à sélectionner dans df_preds les sous-ensembles benchmark/route correspondants
    (par exemple Spec1 vs Spec2, ou Spec1-Rolling vs Spec2-Rolling).
    """
    print(f'\n--- {label} ---')
    records_cw_local = []
    records_cw_seg_local = []
    records_cw_date_local = []

    if df_preds.empty:
        print('  ⚠️ Aucune prévision OOS disponible')
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    _bench = df_preds[(df_preds['model'] == model_bench)
                       & (df_preds['route'] == 'Benchmark')
                       & (df_preds['spec'] == spec_bench)]
    _route_all = df_preds[(df_preds['spec'] == spec_route)
                          & (df_preds['route'] != 'Benchmark')]

    for (route, ticker, h), grp_r in _route_all.groupby(['route', 'ticker', 'h']):
        grp_b = _bench[(_bench['ticker'] == ticker) & (_bench['h'] == h)]
        if grp_b.empty:
            continue

        cw_stat, pval_1s, pval_2s, n_common = clark_west_test(
            grp_b['date'].values, grp_b['actual'].values, grp_b['pred'].values,
            grp_r['date'].values, grp_r['actual'].values, grp_r['pred'].values, h,
            model_type=model_type)
        records_cw_local.append({
            'route': route, 'ticker': ticker, 'h': h,
            'CW_stat': cw_stat, 'CW_pval_onesided': pval_1s, 'CW_pval_twosided': pval_2s,
            'n_oos_common': n_common,
        })

        seg_results = clark_west_test_segmented(
            grp_b['date'].values, grp_b['actual'].values, grp_b['pred'].values,
            grp_r['date'].values, grp_r['actual'].values, grp_r['pred'].values, h,
            n_segments=2, model_type=model_type)
        for seg_res in seg_results:
            records_cw_seg_local.append({'route': route, 'ticker': ticker, 'h': h, **seg_res})

        date_results = clark_west_test_date_segments(
            grp_b['date'].values, grp_b['actual'].values, grp_b['pred'].values,
            grp_r['date'].values, grp_r['actual'].values, grp_r['pred'].values, h,
            model_type=model_type)
        for date_res in date_results:
            records_cw_date_local.append({'route': route, 'ticker': ticker, 'h': h, **date_res})

    df_cw_local = pd.DataFrame(records_cw_local)
    df_cw_seg_local = pd.DataFrame(records_cw_seg_local)
    df_cw_date_local = pd.DataFrame(records_cw_date_local)

    if not df_cw_local.empty:
        _valid = df_cw_local.dropna(subset=['CW_pval_onesided']).copy()
        if len(_valid) > 0:
            _rej, _pfdr, _, _ = multipletests(_valid['CW_pval_onesided'].values,
                                               alpha=0.05, method='fdr_bh')
            _valid['pval_fdr'] = _pfdr
            _valid['sig_raw']  = _valid['CW_pval_onesided'] < 0.05
            _valid['sig_fdr']  = _rej
            n_total, n_raw, n_fdr = len(_valid), int(_valid['sig_raw'].sum()), int(_valid['sig_fdr'].sum())
            print(f'  [Synthèse sur l’échantillon complet] nombre de tests : {n_total}  p brut<0,05 : {n_raw} '
                  f'(environ {0.05*n_total:.1f} attendus sous le bruit)  significatifs après FDR : {n_fdr}')
            if n_fdr > 0:
                print(_valid[_valid['sig_fdr']]
                      [['route','ticker','h','CW_stat','CW_pval_onesided','pval_fdr','n_oos_common']]
                      .sort_values('pval_fdr').to_string(index=False))
            _valid.to_csv(os.path.join(OUT_DIR, f'part2_clarkwest_{out_suffix}.csv'), index=False)
            df_cw_local = _valid
        else:
            print('  [Synthèse sur l’échantillon complet] ⚠️ Aucun résultat CW disponible')
    else:
        print('  [Synthèse sur l’échantillon complet] ⚠️ Aucun résultat CW disponible')

    if not df_cw_seg_local.empty:
        for seg_id, seg_grp in df_cw_seg_local.groupby('segment'):
            seg_valid = seg_grp.dropna(subset=['pval_onesided']).copy()
            if len(seg_valid) == 0:
                continue
            _rej_s, _pfdr_s, _, _ = multipletests(seg_valid['pval_onesided'].values,
                                                   alpha=0.05, method='fdr_bh')
            seg_valid['pval_fdr'] = _pfdr_s
            seg_valid['sig_fdr']  = _rej_s
            df_cw_seg_local.loc[seg_valid.index, 'pval_fdr'] = seg_valid['pval_fdr']
            df_cw_seg_local.loc[seg_valid.index, 'sig_fdr']  = seg_valid['sig_fdr']
            n_seg_fdr = int(seg_valid['sig_fdr'].sum())
            _dr = f'{seg_valid["start_date"].min()}~{seg_valid["end_date"].max()}' if seg_valid['start_date'].notna().any() else 'N/A'
            print(f'  [Segment {seg_id}, {_dr}] nombre de tests : {len(seg_valid)}  '
                  f'p brut<0,05 : {int((seg_valid["pval_onesided"]<0.05).sum())}  significatifs après FDR : {n_seg_fdr}')
        df_cw_seg_local.to_csv(os.path.join(OUT_DIR, f'part2_clarkwest_segmented_{out_suffix}.csv'), index=False)
    else:
        print('  [Test de stabilité première/seconde moitié] ⚠️ Aucun résultat CW segmenté disponible')

    if not df_cw_date_local.empty:
        for period, period_grp in df_cw_date_local.groupby('period', sort=False):
            period_valid = period_grp.dropna(subset=['pval_onesided']).copy()
            if len(period_valid) == 0:
                continue
            _rej_d, _pfdr_d, _, _ = multipletests(
                period_valid['pval_onesided'].values, alpha=0.05, method='fdr_bh')
            period_valid['pval_fdr'] = _pfdr_d
            period_valid['sig_fdr'] = _rej_d
            df_cw_date_local.loc[period_valid.index, 'pval_fdr'] = period_valid['pval_fdr']
            df_cw_date_local.loc[period_valid.index, 'sig_fdr'] = period_valid['sig_fdr']
            print(f'  [{period}] nombre de tests : {len(period_valid)}  '
                  f'p brut<0,05 : {int((period_valid["pval_onesided"] < 0.05).sum())}  '
                  f'significatifs après FDR : {int(period_valid["sig_fdr"].sum())}')
        df_cw_date_local.to_csv(
            os.path.join(OUT_DIR, f'part2_clarkwest_covid_segments_{out_suffix}.csv'),
            index=False)
    else:
        print('  [Sous-périodes COVID à dates fixes] ⚠️ Aucun résultat CW disponible')

    return df_cw_local, df_cw_seg_local, df_cw_date_local


df_cw, df_cw_seg, df_cw_covid = run_cw_battery(
    df_oos_preds, 'Spec1', 'Spec2',
    'HAR, réestimation tous les 22 jours et prévisions quotidiennes', 'oos_significance',
    model_type='HAR')

df_cw_rolling, df_cw_seg_rolling, df_cw_covid_rolling = run_cw_battery(
    df_oos_preds_rolling, 'Spec1-Rolling', 'Spec2-Rolling',
    'HAR, fenêtre rolling (maximum 750 étiquettes réalisées)', 'har_oos_significance_rolling',
    model_bench='HAR-RV', model_type='HAR')

# Même test pour GARCH-X : la deuxième étape Spec2 ajoute text_feature à la Spec1 et reste imbriquée.
df_cw_garch, df_cw_garch_seg, df_cw_garch_covid = run_cw_battery(
    df_oos_preds_garch, 'Spec1', 'Spec2',
    'GARCH-X, fenêtre expanding', 'garch_oos_significance',
    model_bench='GARCH', model_type='GARCH')
df_cw_garch_rolling, df_cw_garch_seg_rolling, df_cw_garch_covid_rolling = run_cw_battery(
    df_oos_preds_garch_rolling, 'Spec1-Rolling', 'Spec2-Rolling',
    'GARCH-X, fenêtre rolling', 'garch_oos_significance_rolling',
    model_bench='GARCH', model_type='GARCH')


def _dm_window_test(df_exp, df_roll, model_label, model_type='HAR'):
    # DM/HAC bilatéral : expanding vs rolling pour la même route. Stat>0 signifie rolling meilleur.
    if df_exp.empty or df_roll.empty:
        return pd.DataFrame()
    keys = ['date', 'ticker', 'h', 'route']
    e = df_exp[keys + ['actual', 'pred']].rename(columns={'actual':'actual_e','pred':'pred_e'})
    r = df_roll[keys + ['actual', 'pred']].rename(columns={'actual':'actual_r','pred':'pred_r'})
    m = e.merge(r, on=keys, how='inner')
    records = []
    for (ticker, h, route), g in m.groupby(['ticker','h','route']):
        if len(g) < 20:
            continue
        for loss_name in ['SE', 'QLIKE']:
            if loss_name == 'SE':
                loss_e = (g['actual_e'].values - g['pred_e'].values) ** 2
                loss_r = (g['actual_r'].values - g['pred_r'].values) ** 2
            else:
                loss_e = qlike_loss_vector(g['actual_e'].values, g['pred_e'].values)
                loss_r = qlike_loss_vector(g['actual_r'].values, g['pred_r'].values)
            d = loss_e - loss_r
            fit = sm.OLS(d, np.ones((len(d), 1))).fit(
                cov_type='HAC', cov_kwds={'maxlags': _oos_hac_lags(int(h), model_type=model_type)})
            stat = float(fit.tvalues[0])
            p2 = float(2 * norm.sf(abs(stat)))
            records.append({'model': model_label, 'ticker': ticker, 'h': h, 'route': route,
                            'loss': loss_name, 'DM_stat': stat, 'pval_twosided': p2,
                            'mean_loss_expanding_minus_rolling': float(np.mean(d)), 'n_common': len(d)})
    out = pd.DataFrame(records)
    if not out.empty:
        for loss_name, idx in out.groupby('loss').groups.items():
            vals = out.loc[idx, 'pval_twosided'].values
            rej, pfdr, _, _ = multipletests(vals, alpha=0.05, method='fdr_bh')
            out.loc[idx, 'pval_fdr'] = pfdr
            out.loc[idx, 'sig_fdr'] = rej
    return out


if ENABLE_WINDOW_DM_TEST:
    df_dm_har_windows = _dm_window_test(df_oos_preds, df_oos_preds_rolling, 'HAR-RV-X', model_type='HAR')
    df_dm_garch_windows = _dm_window_test(df_oos_preds_garch, df_oos_preds_garch_rolling, 'GARCH-X', model_type='GARCH')
else:
    df_dm_har_windows = pd.DataFrame()
    df_dm_garch_windows = pd.DataFrame()


def build_performance_master(df_exp, df_roll, cw_exp, cw_roll, model_name):
    # Table longue unique : métriques OOS, type de fenêtre, amélioration RMSE et p-values CW.
    pieces = []
    for wt, frame, route_spec, bench_spec, cw in [
        ('expanding', df_exp, 'Spec2', 'Spec1', cw_exp),
        ('rolling', df_roll, 'Spec2-Rolling', 'Spec1-Rolling', cw_roll),
    ]:
        if frame.empty:
            continue
        sub = frame[frame['spec'].isin([route_spec, bench_spec])].copy()
        sub['window_type'] = wt
        sub['model_family'] = model_name
        bench = (sub[sub['route'] == 'Benchmark'][['ticker','h','OOS_RMSE']]
                 .rename(columns={'OOS_RMSE':'benchmark_RMSE'}))
        sub = sub.merge(bench, on=['ticker','h'], how='left')
        sub['RMSE_improvement_pct'] = (sub['benchmark_RMSE'] - sub['OOS_RMSE']) / sub['benchmark_RMSE'] * 100
        if not cw.empty:
            cw_cols = [c for c in ['route','ticker','h','CW_stat','CW_pval_onesided','CW_pval_twosided','pval_fdr','sig_fdr','n_oos_common'] if c in cw.columns]
            sub = sub.merge(cw[cw_cols], on=['route','ticker','h'], how='left')
        pieces.append(sub)
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()


df_har_performance_master = build_performance_master(
    df_har, df_har_rolling, df_cw, df_cw_rolling, 'HAR-RV-X')
df_garch_performance_master = build_performance_master(
    df_garch, df_garch_rolling, df_cw_garch, df_cw_garch_rolling, 'GARCH-X')

if ENABLE_WINDOW_DM_TEST:
    print(f'\n[DM fenêtres] HAR : {len(df_dm_har_windows)} tests | GARCH : {len(df_dm_garch_windows)} tests')


# %% ── 14. Enregistrement des fichiers de sortie ─────────────────────────────────
def export_regression_tables(coef_df, prefix, specs=None):
    """
    Exporte des tables de régression de présentation : une ligne de coefficient avec étoiles,
    suivie d’une ligne d’erreur-type entre parenthèses, puis N/R²/R² ajusté.
    """
    if coef_df.empty:
        return
    work = coef_df.copy()
    if specs is not None:
        work = work[work['spec'].isin(specs)]
    group_cols = ['model', 'spec', 'route', 'h']
    for (model, spec, route, h), grp in work.groupby(group_cols, dropna=False):
        route_name = 'NA' if pd.isna(route) else str(route)
        tickers = [t for t in TICKERS if t in grp['ticker'].unique()]
        terms = list(dict.fromkeys(grp['term'].tolist()))
        rows = []
        index = []
        for term in terms:
            row_coef, row_se = {}, {}
            term_grp = grp[grp['term'] == term].set_index('ticker')
            for ticker in tickers:
                if ticker in term_grp.index:
                    rec = term_grp.loc[ticker]
                    if isinstance(rec, pd.DataFrame):
                        rec = rec.iloc[0]
                    row_coef[ticker] = f"{rec['coef']:.6f}{fmt_stars(rec['pval'])}"
                    row_se[ticker] = f"({rec['se']:.6f})"
                else:
                    row_coef[ticker] = ''
                    row_se[ticker] = ''
            rows.extend([row_coef, row_se])
            index.extend([term, f'{term} [SE]'])
        tab = pd.DataFrame(rows, index=index, columns=tickers)
        for stat_col, label, digits in [('n_obs','N',0), ('R2','R²',4), ('Adj_R2','R² ajusté',4), ('AIC','AIC',2), ('BIC','BIC',2)]:
            if stat_col not in grp.columns or grp[stat_col].notna().sum() == 0:
                continue
            vals = grp.groupby('ticker')[stat_col].first()
            tab.loc[label] = [f'{vals.get(t, np.nan):.{digits}f}' if pd.notna(vals.get(t, np.nan)) else '' for t in tickers]
        stem = _safe_filename(f'{prefix}_{model}_{spec}_{route_name}_h{h}')
        tab.to_csv(os.path.join(OUT_DIR, stem + '.csv'))
        try:
            tab.to_latex(os.path.join(OUT_DIR, stem + '.tex'), escape=False,
                         caption=f'{model} {spec} {route_name}, h={h}',
                         label=f'tab:{stem.lower()}')
        except Exception as e:
            print(f'  Avertissement export LaTeX {stem}: {e}')


# Version longue, directement exploitable pour des tableaux personnalisés.
if not df_har_coefs.empty:
    df_har_coefs['sig'] = df_har_coefs['pval'].apply(fmt_stars)
if not df_garch_2ndstep.empty:
    df_garch_2ndstep['sig'] = df_garch_2ndstep['pval'].apply(fmt_stars)
if not df_garch_native.empty:
    df_garch_native['sig'] = df_garch_native['pval'].apply(fmt_stars)

df_har_gkg.to_csv(os.path.join(OUT_DIR, 'part1_har_gkg_full.csv'),      index=False)
df_har.to_csv(os.path.join(OUT_DIR,     'part2_har_text_full.csv'),     index=False)
for h_save in H_LIST:
    for key, df_tab in main_tables_by_h[h_save].items():
        df_tab.to_csv(os.path.join(OUT_DIR, f'part2_{key}_h{h_save}.csv'))
if not df_oos_preds.empty:
    df_oos_preds.to_csv(os.path.join(OUT_DIR, 'part2_oos_predictions.csv'), index=False)
df_panel.to_csv(os.path.join(OUT_DIR,   'part2_panel.csv'),              index=False)
if not df_garch_gkg.empty:
    df_garch_gkg.to_csv(os.path.join(OUT_DIR, 'part1_garch_gkg.csv'),   index=False)
if not df_garch.empty:
    df_garch.to_csv(os.path.join(OUT_DIR,     'part2_garch_text.csv'),  index=False)
if not df_garch_gkg_rolling.empty:
    df_garch_gkg_rolling.to_csv(
        os.path.join(OUT_DIR, 'part1_garch_gkg_rolling.csv'), index=False)
if not df_garch_rolling.empty:
    df_garch_rolling.to_csv(
        os.path.join(OUT_DIR, 'part2_garch_text_rolling.csv'), index=False)

# [ajout v8] Export des coefficients en tableaux
if not df_har_coefs.empty:
    df_har_coefs.to_csv(os.path.join(OUT_DIR, 'part_har_coefs_full.csv'), index=False)
if not df_garch_native.empty:
    df_garch_native.to_csv(os.path.join(OUT_DIR, 'part_garch_native_params.csv'), index=False)
if not df_garch_2ndstep.empty:
    df_garch_2ndstep.to_csv(os.path.join(OUT_DIR, 'part_garch_twostep_coefs.csv'), index=False)
if not df_garch_coef_paths.empty:
    df_garch_coef_paths.to_csv(os.path.join(OUT_DIR, 'part_garch_coef_paths_expanding_rolling.csv'), index=False)
if not df_oos_preds_garch.empty:
    df_oos_preds_garch.to_csv(os.path.join(OUT_DIR, 'part_garch_oos_predictions_expanding.csv'), index=False)
if not df_oos_preds_garch_rolling.empty:
    df_oos_preds_garch_rolling.to_csv(os.path.join(OUT_DIR, 'part_garch_oos_predictions_rolling.csv'), index=False)
if not df_dm_har_windows.empty:
    df_dm_har_windows.to_csv(os.path.join(OUT_DIR, 'part_har_dm_expanding_vs_rolling.csv'), index=False)
if not df_dm_garch_windows.empty:
    df_dm_garch_windows.to_csv(os.path.join(OUT_DIR, 'part_garch_dm_expanding_vs_rolling.csv'), index=False)
if not df_har_performance_master.empty:
    df_har_performance_master.to_csv(os.path.join(OUT_DIR, 'part_har_performance_master_expanding_rolling.csv'), index=False)
if not df_garch_performance_master.empty:
    df_garch_performance_master.to_csv(os.path.join(OUT_DIR, 'part_garch_performance_master_expanding_rolling.csv'), index=False)

export_regression_tables(df_har_coefs, 'regtable_har', specs=['Spec1','Spec2','Spec3'])
export_regression_tables(df_garch_2ndstep, 'regtable_garch_second_stage', specs=['Spec1','Spec2'])
export_regression_tables(df_garch_native, 'regtable_garch_native', specs=['Native'])

# Tableaux de synthèse GARCH-X par horizon (expanding et rolling)
for h_save, tables_h in garch_tables_by_h.items():
    for key, table in tables_h.items():
        table.to_csv(os.path.join(OUT_DIR, f'part2_GARCH_{key}_h{h_save}.csv'))

# Export détaillé HAR rolling et tableaux bruts R²/RMSE/QLIKE.
if not df_har_rolling.empty:
    df_har_rolling.to_csv(os.path.join(OUT_DIR, 'part2_har_text_rolling.csv'), index=False)
    for h_save in H_LIST:
        for metric in ['OOS_R2', 'OOS_RMSE', 'OOS_QLIKE']:
            tab = _performance_with_benchmark(df_har_rolling, h_save, 'Spec2-Rolling', 'Spec1-Rolling', metric)
            tab.to_csv(os.path.join(OUT_DIR, f'part2_HAR_Rolling_{metric}_h{h_save}.csv'))
if not df_oos_preds_rolling.empty:
    df_oos_preds_rolling.to_csv(os.path.join(OUT_DIR, 'part2_oos_predictions_rolling.csv'), index=False)

print(f'\n✅ Sorties numériques enregistrées. Fichiers présents avant génération finale des graphiques :')
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f'  {f:50s}  {size/1024:.0f} KB')
print('='*65)

# %% ── 14.5 [OPTIMISATION MEMOIRE RENFORCEE] Nettoyage avant les trajectoires ──────────
# Les résultats numériques nécessaires ont déjà été exportés. Les sections 15/15.5
# recalculent les trajectoires HAR à partir de df_raw et utilisent uniquement
# df_garch_coef_paths pour les graphiques GARCH. On libère donc les DataFrames OOS,
# leurs listes Python sources et les dictionnaires d'accumulateurs qui maintenaient
# encore ces listes en vie malgré gc.collect().
print('\n[14.5] Nettoyage mémoire approfondi avant les trajectoires de coefficients ...')


def _rss_gb():
    """Mémoire résidente du processus, si psutil est disponible."""
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
    except Exception:
        return np.nan


def print_ram(label):
    value = _rss_gb()
    if np.isfinite(value):
        print(f'  [RAM] {label}: {value:.2f} Go')

def release_ram():
    """Force le ramasse-miettes Python et rend autant que possible le heap glibc à l'OS (Linux/Colab)."""
    plt.close('all')
    gc.collect()
    try:
        import ctypes
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass



print_ram('avant nettoyage')

# 1) Tables de prévisions OOS déjà exportées.
for _name in [
    'df_oos_preds', 'df_oos_preds_rolling',
    'df_oos_preds_garch', 'df_oos_preds_garch_rolling',
]:
    if _name in globals():
        del globals()[_name]

# 2) Listes Python sources. Une liste de dictionnaires peut être nettement plus
#    volumineuse que le DataFrame construit à partir d'elle.
for _name in [
    'pred_records', 'pred_records_rolling',
    'pred_records_garch', 'pred_records_garch_rolling',
]:
    if _name in globals():
        try:
            globals()[_name].clear()
        except Exception:
            pass
        del globals()[_name]

# 3) Accumulateurs : ils conservent des références vers les listes ci-dessus.
for _name in ['_HAR_ACCUMULATORS', '_GARCH_GLOBAL_ACCUMULATORS']:
    if _name in globals():
        try:
            globals()[_name].clear()
        except Exception:
            pass
        del globals()[_name]

# 4) Objets temporaires de checkpoints / dernières régressions.
for _name in [
    '_block', '_cached', '_cached_payload', '_har_cached_payload',
    '_garch_cached_payload', '_tfidf_ckpt', '_tfidf_partial',
    'res', 'res1', 'res2', 'res3', 'res_g', 'res_ols', 'res_ols_r',
    'res_ols_gkg', 'fit', 'train', 'X_tr',
]:
    if _name in globals():
        del globals()[_name]

# 5) Les tables de synthèse ci-dessous ont déjà été exportées et ne sont plus
#    nécessaires aux sections 15/15.5. On conserve explicitement df_raw et
#    df_garch_coef_paths.
for _name in [
    'df_har', 'df_har_rolling', 'df_har_gkg', 'df_har_gkg_rolling',
    'df_garch', 'df_garch_rolling', 'df_garch_gkg', 'df_garch_gkg_rolling',
    'df_har_performance_master', 'df_garch_performance_master',
    'df_dm_har_windows', 'df_dm_garch_windows',
]:
    if _name in globals():
        del globals()[_name]

# Libération du cache GPU FinBERT ; ceci ne réduit pas toujours le RSS CPU,
# mais évite de conserver inutilement de la mémoire CUDA.
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

plt.close('all')
gc.collect()
print_ram('après nettoyage')
print('  → Tables OOS, listes sources et références d’accumulateurs libérées.')


# %% ── 15. Trajectoires de coefficients expanding et rolling ─────────────────
print('\n[15] Génération des trajectoires expanding/rolling des coefficients HAR et GARCH ...')
t0 = time.time()
HAR_COEF_PLOT_SUFFIX = ['rv_d_', 'rv_w_', 'rv_m_']
_COEF_H_TAG = 'h' + '-'.join(str(int(h)) for h in COEF_TREND_HORIZONS)

# Écriture incrémentale : évite de conserver une liste de DataFrames puis de
# créer une seconde copie complète lors de pd.concat().
_har_coef_csv = os.path.join(OUT_DIR, f'part_har_coef_paths_expanding_rolling_{_COEF_H_TAG}.csv')
if os.path.exists(_har_coef_csv):
    os.remove(_har_coef_csv)
_har_header_written = False

for h_plot in COEF_TREND_HORIZONS:
    for ticker in TICKERS:
        if f'rv_d_{ticker}' not in df_raw.columns:
            continue

        base = har_base(ticker)
        y_col = f'rv_fwd_h{h_plot}_{ticker}'
        if y_col not in df_raw.columns:
            continue

        print_ram(f'avant HAR benchmark {ticker} h={h_plot}')
        need = ['date', y_col] + base
        dm = df_raw.loc[:, [c for c in need if c in df_raw.columns]].dropna(subset=['date'])

        if dm[[y_col] + base].dropna().shape[0] < MIN_TRAIN_OOS + 1:
            del dm
            gc.collect()
            continue

        for wt in COEF_TREND_WINDOW_TYPES:
            coef_df_har = rolling_coef_trend(
                dm, y_col, base, h=h_plot, window_type=wt,
                trailing_window=ROLLING_WINDOW, step=HAR_REFIT_STEP,
                hac_lags=_har_hac_lags(h_plot))

            if not coef_df_har.empty:
                coef_df_har = coef_df_har.assign(
                    ticker=ticker, h=h_plot, route='Benchmark', spec='Spec1')
                coef_df_har.to_csv(
                    _har_coef_csv,
                    mode='a', header=not _har_header_written, index=False)
                _har_header_written = True

                for suffix in HAR_COEF_PLOT_SUFFIX:
                    plot_coef_trend(
                        coef_df_har, f'{suffix}{ticker}', ticker, h_plot, OUT_DIR,
                        window_type=wt, route='Benchmark', component='HAR')

            del coef_df_har
            release_ram()

        del dm
        release_ram()

# Pour compatibilité avec les éventuelles cellules exécutées ensuite, on garde
# le nom mais pas une copie complète du CSV en RAM.
df_har_coef_paths = pd.DataFrame()

# GARCH : les trajectoires ont déjà été calculées pendant la section 11.
# On ne fait ici que produire les graphiques à partir de df_garch_coef_paths.
if ENABLE_GARCH_COEF_TREND_PLOTS and ENABLE_GARCH and not df_garch_coef_paths.empty:
    print_ram('avant graphiques GARCH')
    for (ticker, h_plot, route, wt, component), grp in df_garch_coef_paths.groupby(
            ['ticker', 'h', 'route', 'window_type', 'component'], sort=False):
        if int(h_plot) not in {int(x) for x in COEF_TREND_HORIZONS}:
            continue
        terms = (
            ['alpha[1]', 'beta[1]']
            if component == 'native_garch' and route == 'Benchmark'
            else []
        )
        if component == 'second_stage':
            route_feats = (
                TEXT_ROUTE_FEATS_FOR_GARCH.get(route, [])
                if route != 'Benchmark' else []
            )
            unique_terms = set(grp['term'].dropna().unique())
            terms = ['garch_rv'] + [f for f in route_feats if f in unique_terms]

        for term in terms:
            if term in set(grp['term'].dropna().unique()):
                plot_coef_trend(
                    grp, term, ticker, h_plot, OUT_DIR,
                    window_type=wt, route=route,
                    component=f'GARCH_{component}')
        plt.close('all')

    release_ram()
else:
    print('  → Graphiques GARCH ignorés : les trajectoires numériques sont déjà dans part_garch_coef_paths_expanding_rolling.csv')

# La section 15.5 n'utilise plus df_garch_coef_paths : libération immédiate.
if 'df_garch_coef_paths' in globals():
    del df_garch_coef_paths
release_ram()

print_ram('fin section 15')
print(f'→ Trajectoires de base terminées en {time.time()-t0:.1f}s')




✅ Protection de l’ensemble d’information activée : purge (h-1) appliquée aux évaluations OOS HAR et GARCH

[10] Matrice principale HAR-RV-X (calcul en cours...)
  ↺ HAR SPY h=1 restauré depuis le nouveau cache
  ↺ HAR SPY h=5 restauré depuis le nouveau cache
  ↺ HAR SPY h=22 restauré depuis le nouveau cache
  ↺ HAR XLE h=1 restauré depuis le nouveau cache
  ↺ HAR XLE h=5 restauré depuis le nouveau cache
  ↺ HAR XLE h=22 restauré depuis le nouveau cache
  ↺ HAR XLF h=1 restauré depuis le nouveau cache
  ↺ HAR XLF h=5 restauré depuis le nouveau cache
  ↺ HAR XLF h=22 restauré depuis le nouveau cache
  ↺ HAR XLK h=1 restauré depuis le nouveau cache
  ↺ HAR XLK h=5 restauré depuis le nouveau cache
  ↺ HAR XLK h=22 restauré depuis le nouveau cache
  ↺ HAR XLY h=1 restauré depuis le nouveau cache
  ↺ HAR XLY h=5 restauré depuis le nouveau cache
  ↺ HAR XLY h=22 restauré depuis le nouveau cache
  ↺ HAR XLP h=1 restauré depuis le nouveau cache
  ↺ HAR XLP h=5 restauré depuis le nouveau cache
 